# TwinArch on DENTEX: tooth-level diagnosis with contralateral and arch context (Kaggle, 2× T4)

**Pipeline.** A detector only *finds and numbers* the teeth. Every tooth is then cropped and diagnosed by our own classifier, which also learns where the disease box should sit around the tooth:

1. Each tooth crop is encoded **once** by a small CNN (ConvNeXt-nano). Crops from the image-right side (quadrants 2 and 3) are mirrored first, so a tooth and its contralateral twin (16↔26, 36↔46 …) face the same way.
2. Variants: `single` = crop only; `twin` = adds the asymmetry |f_tooth − f_twin| to the twin; `arch` = a 2-layer transformer over all teeth with a learned attention bias per anatomical relation (twin, neighbour, antagonist, same quadrant); `full` = twin + arch (**TwinArch**).
3. **Learned boxes.** Two small heads predict, per disease, how the annotated disease box sits relative to the found tooth box: one from the pooled tooth feature (`token`), one from the encoder's spatial feature map (`map`). Both are detached, so they never change the diagnosis. Which box source is used (token, map, a median adapter or the raw tooth box) is picked on validation X-rays only.

4. **Fusion with the detector.** Our boxes are also merged with the plain YOLO detector's boxes by weighted boxes fusion (the weight of each model and the IoU threshold are picked on validation X-rays). It is reported as its own row. **Control:** the same YOLO trained with a second seed, fused with the first one the same way; and our classifier fused with both YOLOs. So the tables say whether our classifier adds more than a second detector would.

5. **Sharper YOLO (v11).** The same YOLO11m trained at 1280 instead of 1024 (the exported X-rays are 1280 wide, so nothing is downscaled). It is one more model to fuse, part of the teacher, the architecture of the student, and part of the detector-only controls.
6. **Self-training (v11).** DENTEX also has 2,898 X-rays without diagnosis labels (634 enumeration, 693 quadrant-only, 1,571 unlabelled). The *teacher* (our classifier + the three YOLOs of a split, fused, with TTA) labels them: boxes above a per-disease threshold (best F1 on the split's validation X-rays) for a *student YOLO* (at 1280), and a soft score per tooth and disease for a *student classifier*. Both students start again from COCO / ImageNet weights and train on the split's annotated X-rays plus the pseudo-labelled ones (Noisy Student, Xie et al. 2020). Copies of annotated X-rays are removed from the pool first, so no validation or test X-ray is ever pseudo-labelled.
7. **Test-time augmentation (v11).** YOLO: ultralytics' own (3 scales, one mirrored). Classifier: the average over 5 fixed views of each crop (identity, zoom in/out 6%, rotate ±3°). Every model is reported with and without it, the baselines included.
8. **Everything fused (v11).** All classifiers and YOLOs together, with one setting for all diseases or one per disease (picked on validation). Every fusion of ours has a control made of the same detectors without our classifiers, and each comparison gets a paired bootstrap.
9. **Detector boxes for our classifier (v12).** DENTEX disease boxes are boxes around the diseased tooth, so a detector box on a tooth is a good box for any disease of that tooth. In a fusion, each of our tooth-level predictions may take the box of the detectors' best box on the same tooth (the detectors place the box, our classifier scores it, as in DCR, Cheng et al. 2018). Our classifier is a strong recogniser with loose boxes (with the annotated boxes swapped in, its validation AP rises from ~35 to ~56), so this lets it count more without blurring the fused box. Validation decides, per fusion, whether to use it.
10. **Second self-training round (v12).** The best round-1 fusion (our two classifiers + the four YOLOs, with TTA) labels the pool again, and two YOLO students learn from it at 1280 from COCO weights: YOLO11m (the round-1 architecture, so round 2 can be compared with round 1) and YOLO26m (a newer architecture; on split 0 it was the best single model). Noisy Student found later rounds still help, with smaller steps.
11. **Third round at a higher resolution (v14).** The round-2 fusion (with TTA) labels the pool once more and a YOLO26m student learns at 1536 instead of 1280, from a second export of the X-rays (the originals are larger; caries are small). Noisy Student also made its students bigger round by round, and higher input resolution helped on DENTEX before (1024 → 1280: +2 AP for the plain YOLO; DinoDental: 512 → 1280: +6 AP).
12. **Final fusion (v14).** Our two classifiers + all the self-trained YOLOs + the three plain YOLOs, next to the same YOLOs fused without our classifiers. The plain YOLOs are optional (validation decides): the self-trained ones are 4-9 AP stronger and may only dilute the fusion. The headline row takes, on each split, the round-1, round-2 or final fusion, with or without TTA, whichever has the best validation AP (the test X-rays never choose).
13. **Tried in the v12 trial and dropped.** Fine-tuning each student on the annotated X-rays only (ASTOD, Vandeghen et al. 2022) lost 0.6-1.8 AP on split 0 in all four cases, and a round-2 classifier student lost 2.0 AP against the round-1 one. Their split-0 rows are still reported; they are not made on the other splits and never enter a fusion.

**v15: an audit for leaks and overfitting, and what changed because of it.**
* **Copies inside DENTEX.** DENTEX's subsets were cut from one archive of X-rays, so one X-ray can sit in several of them (v11's pool check already removed 579 copies of diagnosis X-rays from the 2,898 X-rays without diagnosis labels). Cell 08b compares every diagnosis X-ray with every other one and with every enumeration X-ray (standardised thumbnails; copies are at ~0.003, other X-rays at ~0.5). A validation or test X-ray with a copy among the training X-rays of its split is left out of the scores.
* **The tooth detector had seen test X-rays.** Up to v14 the tooth detector learned from all 634 enumeration X-rays, and many of them are copies of diagnosis X-rays, validation and test ones included. It never saw a diagnosis, but it had learned where their teeth are, and our classifier's boxes are tooth boxes, so test boxes were more precise than they would be on new X-rays. v15 trains a second tooth detector (same recipe) only on the enumeration X-rays that are not even near copies of a diagnosis X-ray, and **every validation and test X-ray is cropped by it**. The trained models stay as they are: none of them ever saw a validation or test X-ray (the first tooth detector only crops training X-rays, and the pool has no copies of annotated X-rays). Cell 11 shows how much better the first detector's boxes were on the X-rays it had learned from.
* **Same patient, two X-rays.** DENTEX has no patient ids. Cell 08b checks the most similar pairs with ORB keypoints and RANSAC and saves a picture of them (`audit_v15_pairs.jpg`), so this can be looked at by eye. It is reported, not used to drop X-rays.
* **Overfitting the test sets by tuning.** Every setting is picked on validation X-rays, but the method itself grew over many versions while we looked at the test numbers of protocol A and of splits 0-2 (for example, the v12 trial dropped two components because of split 0's test AP). So v15 adds two **fresh splits (seeds 3 and 4)** that no version has looked at, made and scored once with the method frozen. They are reported on their own: that is the honest number for new data.
* What is not checked and stays a limitation: label noise (each X-ray was labelled once and then checked by experts, not by independent readers), and X-rays from the same patient that the ORB check does not catch.

The plain YOLO disease detector that most papers use is trained as the **baseline row** (and is the second model in the fusion).

**Protocols**
* **A (fixed official split):** train on the 705 labelled X-rays, pick settings on a held-out 70 of them, report on the official 50-image validation set (released with proper 4-class labels). The 250 released test labels use a different Turkish, treatment-style label set with no deep-caries class and no published mapping, so they are not used.
* **B (peer protocol):** random 80:20 splits of the 705, as in SSAD 2024, Nassiri & Akhloufi 2025, Sadr et al. 2026, DinoDental 2026 and Hoang et al. 2026: splits 0-2 (development: every version looked at them) and, from v15, splits 3-4 (confirmation: made once, with the method frozen). Within a split, 10% of the training part picks every setting and the test part never picks anything. Plain YOLO is retrained on every split so the comparison is paired.

Every model-vs-baseline difference comes with a 95% paired bootstrap interval over the test X-rays. Efficiency (parameters, GFLOPs, ms per X-ray on a T4) is logged for every component.

**Two metrics.** The main metric is DENTEX / COCO AP (pycocotools). The YOLO-based papers report the numbers ultralytics' validator prints, which match boxes differently and, before ultralytics 8.4.37, integrated AP with an extra linear tail. So every row is also scored that way (`ul_*` columns, `ul_old_*` = the formula the papers used), and the plain detector is checked against `model.val()` itself. The plain detector's boxes are the ones its own validator scores (multi-label NMS, conf 0.001, up to 300 per X-ray), so the baseline is not handicapped.

**How to run on Kaggle** (Settings: Accelerator **GPU T4 x2**, Internet **On**)
* **v15 paper run** (the defaults, about 19 h, so two sessions): attach the DENTEX dataset and the output of the v14 run with **both** of its sessions in it (the second session's output has everything), Internet on, **Save & Run All**. Session 1: the audit, the clean tooth detector, protocol A and splits 0-2 again with clean crops (~2.5 h, no model is retrained), then the fresh split 3 (~8 h). Session 2: run again with this version's output attached, the same settings: split 4 (~8 h). The pool X-rays are downloaded from Hugging Face again (~11 GB, about a minute). Check the first lines of the log: `files saved in WORK so far:` should be about 375 (356 means only the first v14 session's output is attached, and split 2 would be trained again).
* **Development run** (`QUICK = True`, `FINAL = False`): attach the DENTEX dataset and the output of an earlier version (Add Input → Your Work) so detectors, crops and classifiers are reused. `CLS_TAG` says which version's classifiers to reuse (v8 reuses v6's classifiers and v7's detectors, so attach the v7 output). Bump `RUN_TAG` for every change and judge changes on validation AP, never on the test numbers.
* If a session would run past Kaggle's 12 h, protocol B stops before the next split and says so; run again with the same settings and this version's output attached: everything finished is reused and only the missing splits are done.

Both GPUs are used: YOLO with DDP (`device=0,1`), the classifier with `torchrun --nproc_per_node=2`.


## 1. Settings

In [ ]:
import os, sys, json, glob, time, shutil, pickle, random, math
from pathlib import Path
import numpy as np, pandas as pd
T_SESSION = time.time()    # Kaggle stops a session after 12 h, so the long stages check this clock

# v15 = the v14 method, evaluated cleanly, plus two fresh splits. The tooth detector of v5-v14 learned from all 634
# enumeration X-rays, and v11's pool check removed 289 of them as copies (of diagnosis X-rays or of each other): it
# had seen the teeth (not the diagnoses) of many validation and test X-rays, and our classifier's boxes are tooth
# boxes. v15:
#  (1) every validation and test X-ray is cropped by a second tooth detector that never saw any diagnosis X-ray;
#      the trained models are unchanged (they never saw a validation or test X-ray), only their inputs are clean
#  (2) diagnosis X-rays that are copies of each other are found; a validation / test X-ray with a copy among the
#      training X-rays of its split is not scored; possible same-patient pairs are checked and shown (ORB)
#  (3) protocol B gets two more splits (seeds 3 and 4) that no version has ever looked at, run once with the method
#      frozen: the honest check that tuning on splits 0-2 and on the official 50 did not overfit them
# v14 = v11 + detector boxes for our classifiers, self-training rounds 2 and 3 (YOLO26m at 1536) and the final fusion
# (the plain YOLOs optional), each next to the same thing without it (matched controls). The v12 trial (split 0) also
# tried fine-tuning the students and a round-2 classifier student; both were dropped on split 0's TEST numbers (their
# validation numbers were mixed), which is one reason the fresh splits exist.
# PAPER RUN (the defaults below): attach the DENTEX dataset and the output of the v14 run (both sessions merged).
# About 19 h of work, so two Kaggle sessions: protocol B checks the clock before each split and stops in time for
# Kaggle to save everything (12 h limit); run again with this version's output attached and the same settings.
QUICK = False              # False = full schedules (paper). True = short schedules for quick tests
FINAL = True               # True = paper settings: 3 seeds + all variants. False = development: 1 seed, crop-only
SELF_TRAIN = True          # teacher-student round with the 2,898 DENTEX X-rays that have no diagnosis labels
ST_TRIAL = False           # True = only protocol B split 0 (a trial). False = protocol A + all splits of B
RUN_TAG = 'v15'            # names this version's result files; detectors and crops are reused
CONFIRM = True             # v15: also the two fresh splits 3 and 4 (~8 h each). False = only the clean re-scoring
                           # of protocol A and splits 0-2 (~2.5 h, no model is retrained)
CLS_TAG = 'v9' if FINAL else 'v6'      # classifiers are stored under this tag: 'v9' = the paper run's, 'v6' = the dev
                                       # runs' (attach that output to reuse them). Change it when the classifier changes.
ST_TAG = 'v11'             # the self-training results (pseudo labels, students) are stored under this tag
RUN_PROTOCOL_A = not (SELF_TRAIN and ST_TRIAL)   # official split (705 train / 50 val), reported on the 50 official val
RUN_PROTOCOL_B = True      # peer protocol (random 80:20 of the 705)
RUN_YOLO_BASELINE_B = True   # plain YOLO trained on the same 80:20 splits, the fair baseline for protocol B
RUN_YOLO_BASELINE_A = True   # plain YOLO disease detector row (the method peers use). Set False in a detector-swap run
TOOTH_MODEL = 'yolo11s.pt'   # tooth finder only. Detector-swap check: 'rtdetr-l.pt' (a transformer detector, not YOLO)

ON_KAGGLE = os.path.exists('/kaggle/working')
WORK_NAME = f"dentex_work_{Path(TOOTH_MODEL).stem}" + ('_quick' if QUICK else '')   # each run gets its own folder
# Kaggle keeps only ~500 output files per version, so the work is split in two:
#   WORK (saved with the version): weights, pickles, results, keys, one crops.zip
#   SCR  (scratch, not saved):      YOLO image folders, training runs, unpacked crops, raw data
WORK = Path('/kaggle/working') / WORK_NAME if ON_KAGGLE else Path(WORK_NAME).resolve()
SCR = (Path('/kaggle/tmp') if ON_KAGGLE else Path('scratch').resolve()) / WORK_NAME
RAW = SCR.parent / 'dentex_raw'
INPUT_ROOT = '/kaggle/input'   # where attached datasets / previous versions show up
for d in (WORK, WORK / 'weights', SCR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    TOOTH_MODEL=TOOTH_MODEL,
    DIS_MODEL='yolo11m.pt',            # baseline only (what peers use), kept strong on purpose
    DET_IMGSZ=1024, LONG_SIDE=1280,
    TOOTH_EPOCHS=40 if QUICK else 100,
    DIS_EPOCHS=40 if QUICK else 80,
    DET_BATCH=16,                      # total over both GPUs (8 per T4). Use 8 if you hit CUDA OOM
    DET_WORKERS=4,
    CLS_BACKBONE='convnext_nano.in12k_ft_in1k',   # small and fast; 'convnext_tiny.fb_in22k_ft_in1k' if you want more capacity
    CLS_EPOCHS=10 if QUICK else 12,   # v2 peaked around epoch 6 of 15 and then overfit
    CLS_IMG_BS=2,                      # X-rays per GPU per step (each X-ray = ~30 tooth crops)
    CROP_H=384, CROP_W=224,
    CLS_EXTRA='',                      # extra TwinArch args to try, e.g. '--dim 384 --layers 3 --lr 2e-4'
    VARIANTS_A=['single', 'twin', 'arch', 'full'] if FINAL else ['single'],   # dev: crop-only only
    VARIANTS_B=['single', 'full'] if FINAL else ['single'],
    SEEDS_A=[0, 1, 2] if FINAL else [0],      # v3: seed-to-seed spread was at most 0.4 AP, so one seed is enough in dev
    SEEDS_B=[0] if (SELF_TRAIN and ST_TRIAL) else (([0, 1, 2] + ([3, 4] if CONFIRM else [])) if FINAL else [0]),
    CONFIRM_SEEDS=[3, 4],              # v15: fresh splits, never looked at before, run once with the method frozen
    BOX_ADAPTER=True,                  # move tooth boxes to the disease-box style (learned on training X-rays)
    BOX_SRC='both',                    # learned boxes from the pooled tooth token and/or the spatial feature map
    FUSION=True,                       # also fuse our boxes with the plain detector's (WBF), settings picked on validation
    ENSEMBLE_CONTROL=True,             # control: the same YOLO with another seed, fused with the first (every split)
    COCO_TOPK=None,                    # None: pycocotools scores every box (it keeps the best 100 per X-ray and class)
    TIME_BUDGET_H=11.3,                # start a protocol-B split only if it should end before this many hours
                                       # (the estimate is the slower of the measured last split and split_need_h)
    # test-time augmentation: YOLO = ultralytics' own (3 scales, one mirrored); classifier = the average over fixed
    # views of each crop (identity, zoom in/out 6%, rotate +/-3 deg). Rows are reported with and without it.
    TTA=True,
    CLS_TTA=5 if not QUICK else 2,
    # sharper YOLO: the same YOLO11m at 1280 (the jpgs are 1280 wide, so nothing is downscaled), trained on every
    # split as one more model to fuse, as a control, as part of the teacher, and as the student's architecture
    SHARP_YOLO=True, DET_IMGSZ_HI=1280, DET_BATCH_HI=8,   # 4 X-rays per T4 at 1280
    # self-training (SELF_TRAIN): the teacher = our classifier + the YOLOs, fused (WBF settings picked on validation)
    ST_YOLO_EPOCHS=12 if not QUICK else 3,   # student YOLO (COCO start) on annotated x3 + pseudo-labelled X-rays
    ST_LAB_REPEAT=3,                   # each annotated X-ray is listed 3 times next to the pseudo-labelled ones
    ST_POOL_RATIO=1.0,                 # student classifier: pseudo-labelled X-rays per annotated X-ray each epoch
    ST_PSEUDO_W=0.5,                   # loss weight of a pseudo-labelled tooth (annotated = 1)
    ST_SOFT_CLS=0.5,                   # soft label = 0.5 x teacher classifier + 0.5 x mean tooth score of its YOLOs
    ST_MIN_CONF=0.01,                  # teacher boxes below this score are ignored for pseudo labels
    ST_MIN_GT=25,                      # a per-disease pseudo-label threshold needs this many validation boxes
    # v12-v14 (cell 13b, v12_rows)
    V12=SELF_TRAIN,                    # detector boxes for our classifiers, self-training rounds 2 and 3, final fusion
    TRANSPLANT=True,                   # fusions may put our classifiers on the detectors' boxes (validation decides)
    ST2_YOLO11=True,                   # round-2 YOLO11m student (the same architecture as round 1; ~0.7 h a split)
    ST2_YOLO26='yolo26m.pt',           # round-2 student of a newer architecture (None = skip; ~0.8 h a split)
    ST2_YOLO26L=None,                  # optional larger round-2 student, e.g. 'yolo26l.pt' (~1 h a split)
    ST3=SELF_TRAIN,                    # round 3: the round-2 fusion teaches a student at a higher resolution
    ST3_MODEL='yolo26m.pt', DET_IMGSZ_XHI=1536, DET_BATCH_XHI=6,   # 3 X-rays per T4 at 1536 (~1.2 h a split)
    FINETUNE_STUDENTS=False,           # v12 trial: fine-tuning students on the annotated X-rays lost 0.6-1.8 AP
    FT_EPOCHS=15 if not QUICK else 3,  # (only with FINETUNE_STUDENTS) epochs on the annotated X-rays only
    FT_LR=0.0002, FT_CLOSE_MOSAIC=5,   # AdamW, 1/6 of the learning rate ultralytics picks for these runs
    ST2_CLS=False,                     # v12 trial: the round-2 classifier student lost 2.0 AP against round 1
    SMOKE_TEST=True,
    # v15: clean evaluation (cells 08b, 09, 11, 12)
    CLEAN_EVAL=True,                   # validation / test X-rays cropped by a tooth detector that never saw them
    EVAL_CROP_ID0=30_000_000,          # crop ids of the clean crops (training crops start at 0, the pool's at 10 M)
    DUP_HARD=0.17,                     # thumbnail distance below which two files are the same X-ray (copies ~0.003)
    DUP_SOFT=0.30,                     # closer than this to any diagnosis X-ray: not used by the clean tooth detector
    AUDIT_ORB=True,                    # same-patient check on the most similar diagnosis X-ray pairs (CPU, ~2 min)
    AUDIT_MAX_PAIRS=800,
    SPLIT_SAFETY=1.1,                  # a split trained from scratch is assumed to take 10% longer than estimated
)
print('WORK =', WORK, '| QUICK =', QUICK, '| FINAL =', FINAL, '| RUN_TAG =', RUN_TAG, '| classifiers:', CLS_TAG,
      '| self-training:', ('trial on split 0' if ST_TRIAL else 'all splits') if SELF_TRAIN else 'off')

def hours_used():
    return (time.time() - T_SESSION) / 3600

# stage timer -> WORK/timing.json (lets us report GPU hours per stage in the paper)
TIMES_P = WORK / 'timing.json'
TIMES = json.loads(TIMES_P.read_text()) if TIMES_P.exists() else {}
class stage:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); print(f'\n===== {self.name} ====='); return self
    def __exit__(self, exc_type, *a):
        dt = time.time() - self.t
        TIMES[self.name] = TIMES.get(self.name, 0) + dt
        TIMES_P.write_text(json.dumps(TIMES, indent=1))
        print(f'===== {self.name} {"FAILED after" if exc_type else "done in"} {dt/60:.1f} min =====')

## 2. Install and check both GPUs

In [ ]:
# ultralytics pinned to the version of the v11 runs (Oct 2026), so cached and new detector results come from the
# same code (validator, metrics, YOLO26 support)
!pip install -q "ultralytics==8.4.174" pycocotools hf_xet "huggingface_hub>=0.30" timm
import torch
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU >= 1, 'Turn on the GPU T4 x2 accelerator'
DEVICE_STR = ','.join(str(i) for i in range(NGPU))
print('YOLO device string:', DEVICE_STR, '-> DDP on both GPUs' if NGPU > 1 else '-> single GPU')

## 3. Helper code
Written to disk so the `torchrun` workers on both GPUs can import it.

In [ ]:
%%writefile dentex_utils.py
# dentex_utils.py
# Helpers for the DENTEX symmetry project: data discovery, YOLO conversion,
# tooth crops with contralateral twins, prediction building and DENTEX-style COCO evaluation.
import os, json, math, random, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

DIS = ['Caries', 'Deep Caries', 'Periapical Lesion', 'Impacted']   # our fixed order
LABEL_COLS = ['y_caries', 'y_deep', 'y_peri', 'y_impacted']
IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
# Order used in the HierarchicalDet/DENTEX json files, only used if categories_3 is missing.
DEFAULT_DIS_IDS = {0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 'Deep Caries'}
TWIN_Q = {1: 2, 2: 1, 3: 4, 4: 3}


# ----------------------------------------------------------------------------- labels
_TR = str.maketrans({'ç': 'c', 'Ç': 'c', 'ü': 'u', 'Ü': 'u', 'ö': 'o', 'Ö': 'o', 'ğ': 'g', 'Ğ': 'g',
                     'ş': 's', 'Ş': 's', 'ı': 'i', 'İ': 'i'})


def norm_dis(name):
    """English training names and the Turkish names used in the official test labels
    (cürük = caries, derin cürük = deep caries, gömülü = impacted, periapikal lezyon = periapical lesion)."""
    n = str(name).translate(_TR).lower().strip()
    if 'impact' in n or 'gomulu' in n:
        return 'Impacted'
    if 'periap' in n or 'lezyon' in n or 'lesion' in n:
        return 'Periapical Lesion'
    if 'deep' in n or 'derin' in n:
        return 'Deep Caries'
    if 'carie' in n or 'curuk' in n:
        return 'Caries'
    return None


def fdi_to_cls(q, n):
    return (int(q) - 1) * 8 + (int(n) - 1)


def cls_to_fdi(c):
    c = int(c)
    return c // 8 + 1, c % 8 + 1


def twin_of(q, n):
    return TWIN_Q[int(q)], int(n)


# ----------------------------------------------------------------------------- download / unzip
def safe_extract(zpath, dest, skip=('unlabel',), keep=None):
    """Extract a zip but skip members whose path contains any of `skip` (case-insensitive), and, if `keep` is
    given, members for which keep(name) is False."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    n = 0
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            name = m.filename
            if any(s in name.lower() for s in skip) or (keep is not None and not keep(name)):
                continue
            target = (dest / name).resolve()
            if not str(target).startswith(str(root)):
                continue  # path traversal guard
            z.extract(m, dest)
            n += 1
    return n


# ----------------------------------------------------------------------------- discovery
def index_image_dirs(root):
    dirs = {}
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXT:
            dirs.setdefault(p.parent, set()).add(p.name)
    return dirs


def _common_prefix_len(a, b):
    a, b = Path(a).resolve().parts, Path(b).resolve().parts
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def resolve_image_dir(json_path, file_names, img_dirs):
    """Pick the image folder that contains most of the json's file names; ties go to the folder
    closest to the json file. Returns (dir, hit_rate)."""
    names = {Path(f).name for f in file_names}
    best, best_score = None, (-1.0, -1)
    for d, s in img_dirs.items():
        hit = len(names & s) / max(1, len(names))
        score = (round(hit, 4), _common_prefix_len(d, json_path))
        if score > best_score:
            best, best_score = d, score
    return best, best_score[0]


def classify_json(js):
    if not isinstance(js, dict) or 'images' not in js or 'annotations' not in js:
        return None
    anns = js.get('annotations') or []
    keys = set()
    for a in anns[:200]:
        keys |= set(a.keys())
    if 'category_id_3' in keys:
        return 'disease'
    if 'category_id_2' in keys:
        return 'enum'
    if 'category_id_1' in keys or 'category_id' in keys:
        return 'quad'
    return None


def _cat_map(js, key):
    return {int(c['id']): c.get('name', c['id']) for c in (js.get(key) or [])}


def _as_int(v, default):
    try:
        return int(str(v).strip())
    except Exception:
        return default


def parse_json(json_path, img_dirs, tag):
    js = json.loads(Path(json_path).read_text(encoding='utf-8'))
    kind = classify_json(js)
    if kind is None:
        return None
    imgs = js['images']
    d, hit = resolve_image_dir(json_path, [im['file_name'] for im in imgs], img_dirs)
    c1, c2, c3 = _cat_map(js, 'categories_1'), _cat_map(js, 'categories_2'), _cat_map(js, 'categories_3')
    warn = []
    if kind == 'disease' and not c3:
        warn.append('categories_3 missing, using default disease id order ' + str(DEFAULT_DIS_IDS))
        c3 = DEFAULT_DIS_IDS
    by_id = {}
    for im in imgs:
        stem = Path(im['file_name']).stem
        by_id[im['id']] = dict(key=f'{tag}/{stem}', file=im['file_name'],
                               path=str(Path(d) / Path(im['file_name']).name) if d else None,
                               w=int(im.get('width', 0)), h=int(im.get('height', 0)), anns=[])
    for a in js['annotations']:
        rec = by_id.get(a['image_id'])
        if rec is None:
            continue
        x, y, w, h = [float(v) for v in a['bbox']]
        if w <= 1 or h <= 1:
            continue
        ann = dict(box=[x, y, w, h])
        if 'category_id_1' in a:
            cid = int(a['category_id_1'])
            ann['q'] = _as_int(c1.get(cid, cid + 1), cid + 1)
        elif 'category_id' in a:
            cid = int(a['category_id'])
            ann['q'] = _as_int(c1.get(cid, cid + 1), cid + 1)
        if 'category_id_2' in a:
            cid = int(a['category_id_2'])
            ann['n'] = _as_int(c2.get(cid, cid + 1), cid + 1)
        if 'category_id_3' in a:
            cid = int(a['category_id_3'])
            nm = norm_dis(c3.get(cid, ''))
            if nm is None:
                warn.append(f'unknown disease category id {cid}')
                continue
            ann['d'] = DIS.index(nm)
        rec['anns'].append(ann)
    return dict(kind=kind, json=str(json_path), image_dir=str(d), hit=hit, images=list(by_id.values()),
                cats=dict(c1=c1, c2=c2, c3=c3), warn=warn)


def parse_labelme_label(label):
    """'1-çürük-33' -> ('çürük', 33); also accepts 'çürük-33'."""
    parts = [p.strip() for p in str(label).split('-')]
    fdi = int(parts[-1])
    name = '-'.join(parts[1:-1]) if len(parts) >= 3 else parts[0]
    return name, fdi


def parse_labelme_dir(label_dir, files, img_dirs, tag):
    """Official DENTEX test labels: one LabelMe json per X-ray, shape label '<n>-<diagnosis in Turkish>-<FDI>'."""
    import base64
    items, unknown, names_seen = [], {}, {}
    for jp in sorted(files):
        js = json.loads(Path(jp).read_text(encoding='utf-8'))
        fname = Path(js.get('imagePath') or (Path(jp).stem + '.png')).name
        anns = []
        for sh in js.get('shapes', []):
            try:
                name, fdi = parse_labelme_label(sh['label'])
            except Exception:
                unknown[str(sh.get('label'))] = unknown.get(str(sh.get('label')), 0) + 1
                continue
            names_seen[name] = names_seen.get(name, 0) + 1
            nm = norm_dis(name)
            q, n = fdi // 10, fdi % 10
            if nm is None or not (1 <= q <= 4 and 1 <= n <= 8):
                unknown[str(sh['label'])] = unknown.get(str(sh['label']), 0) + 1
                continue
            xs = [float(p[0]) for p in sh['points']]
            ys = [float(p[1]) for p in sh['points']]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            if x2 - x1 <= 1 or y2 - y1 <= 1:
                continue
            anns.append(dict(box=[x1, y1, x2 - x1, y2 - y1], q=q, n=n, d=DIS.index(nm)))
        items.append(dict(fname=fname, w=int(js.get('imageWidth') or 0), h=int(js.get('imageHeight') or 0),
                          anns=anns, data=js.get('imageData'), json=str(jp)))
    d, hit = resolve_image_dir(label_dir, [it['fname'] for it in items], img_dirs)
    if hit < 0.5:   # images only embedded in the json files: decode them once
        d = Path(label_dir).parent / '_decoded_images'
        d.mkdir(exist_ok=True)
        for it in items:
            if it['data'] and not (d / it['fname']).exists():
                (d / it['fname']).write_bytes(base64.b64decode(it['data']))
        hit = 1.0
    images = [dict(key=f'{tag}/{Path(it["fname"]).stem}', file=it['fname'], path=str(Path(d) / it['fname']),
                   w=it['w'], h=it['h'], anns=it['anns']) for it in items]
    warn = [f'label names found: {names_seen}']
    if unknown:
        warn.append(f'UNRECOGNISED labels skipped (tell Claude): {unknown}')
    return dict(kind='disease', json=str(label_dir), image_dir=str(d), hit=hit, images=images,
                cats=dict(c1={}, c2={}, c3={'labelme': 'Turkish names mapped by norm_dis'}), warn=warn)


def _disease_split(json_path, n_images):
    p = str(json_path).lower()
    if 'valid' in p:
        return 'dis_val'
    if 'test' in p:
        return 'dis_test'
    if 'train' in p:
        return 'dis_train'
    sizes = {'dis_train': 705, 'dis_val': 50, 'dis_test': 250}
    return min(sizes, key=lambda k: abs(sizes[k] - n_images))


def discover(raw_dir, verbose=True):
    """Find every DENTEX json under raw_dir and return {'quad','enum','dis_train','dis_val','dis_test'}."""
    raw_dir = Path(raw_dir)
    img_dirs = index_image_dirs(raw_dir)
    out, seen, labelme = {}, set(), {}
    for jp in sorted(raw_dir.rglob('*.json')):
        try:
            js = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            continue
        kind = classify_json(js)
        if kind is None:
            if isinstance(js, dict) and 'shapes' in js:
                labelme.setdefault(jp.parent, []).append(jp)
            continue
        sig = (kind, len(js['images']), len(js['annotations']))
        if sig in seen:          # same file copied twice
            continue
        seen.add(sig)
        if kind == 'disease':
            split = _disease_split(jp, len(js['images']))
        else:
            split = kind
        rec = parse_json(jp, img_dirs, tag=split)
        if rec is None:
            continue
        if split in out:          # keep the bigger one if two candidates map to the same split
            if len(rec['images']) <= len(out[split]['images']):
                continue
        out[split] = rec
        if verbose:
            n_ann = sum(len(i['anns']) for i in rec['images'])
            print(f'[{split}] {jp}  images={len(rec["images"])} anns={n_ann} '
                  f'image_dir={rec["image_dir"]} hit={rec["hit"]:.2f}')
            for w in rec['warn'][:5]:
                print('   WARN', w)
    for d, files in labelme.items():   # official test labels come as one LabelMe file per X-ray
        if len(files) < 10:
            continue
        split = 'dis_test' if 'test' in str(d).lower() else _disease_split(d, len(files))
        if split in out and len(out[split]['images']) >= len(files):
            continue
        rec = parse_labelme_dir(d, files, img_dirs, tag=split)
        out[split] = rec
        if verbose:
            n_ann = sum(len(i['anns']) for i in rec['images'])
            print(f'[{split}] LabelMe {d}  images={len(rec["images"])} anns={n_ann} '
                  f'image_dir={rec["image_dir"]} hit={rec["hit"]:.2f}')
            for w in rec['warn']:
                print('   ', w)
    return out


# ----------------------------------------------------------------------------- geometry
def xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]


def iou_matrix(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)


# ----------------------------------------------------------------------------- images
def read_gray(path):
    import cv2
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise FileNotFoundError(path)
    return im


def resize_long(path, dst, long_side=1280, quality=95):
    """Save a grayscale JPEG copy with the long side capped at long_side. Returns (W, H, scale)."""
    import cv2
    im = read_gray(path)
    h, w = im.shape[:2]
    s = min(1.0, long_side / max(h, w))
    if s < 1.0:
        im = cv2.resize(im, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(dst), im, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return w, h, s


# ----------------------------------------------------------------------------- YOLO export
def export_yolo(records, root, cls_fn, long_side=1280):
    """Write every record once as root/images/all/<key>.jpg (long side <= long_side) and
    root/labels/all/<key>.txt. Splits are plain text lists of image paths (see write_list), so
    the 80/20 seeds of protocol B reuse the same files. Returns {key: (W, H, scale, jpg_path)}."""
    root = Path(root)
    (root / 'images' / 'all').mkdir(parents=True, exist_ok=True)
    (root / 'labels' / 'all').mkdir(parents=True, exist_ok=True)
    meta = {}
    for r in records:
        safe = r['key'].replace('/', '__')
        jpg = root / 'images' / 'all' / f'{safe}.jpg'
        if jpg.exists() and r.get('w') and r.get('h'):
            w, h = r['w'], r['h']
            s = min(1.0, long_side / max(w, h))
        else:
            w, h, s = resize_long(r['path'], jpg, long_side)
        r['w'], r['h'], r['scale'] = w, h, s
        meta[r['key']] = (w, h, s, str(jpg))
        lines = []
        for a in r['anns']:
            c = cls_fn(a)
            if c is None:
                continue
            x, y, bw, bh = a['box']
            cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
            lines.append(f'{c} {min(max(cx, 0), 1):.6f} {min(max(cy, 0), 1):.6f} '
                         f'{min(bw / w, 1):.6f} {min(bh / h, 1):.6f}')
        (root / 'labels' / 'all' / f'{safe}.txt').write_text('\n'.join(lines))
    return meta


def write_list(root, name, keys, meta):
    p = Path(root) / f'{name}.txt'
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text('\n'.join(meta[k][3] for k in keys) + '\n')
    return str(p)


def write_yolo_yaml(root, names, train_txt, val_txt, yaml_name='data.yaml'):
    root = Path(root)
    lines = [f'path: {root}', f'train: {train_txt}', f'val: {val_txt}', 'names:']
    for i, n in enumerate(names):
        lines.append(f'  {i}: "{n}"')
    p = root / yaml_name
    p.write_text('\n'.join(lines) + '\n')
    return str(p)


def tooth_cls(a):
    if 'q' in a and 'n' in a and 1 <= a['q'] <= 4 and 1 <= a['n'] <= 8:
        return fdi_to_cls(a['q'], a['n'])
    return None


def disease_cls(a):
    return a.get('d')


TOOTH_NAMES = [str(10 * (c // 8 + 1) + (c % 8 + 1)) for c in range(32)]


# ----------------------------------------------------------------------------- detector outputs
def teeth_from_yolo(xyxy, cls, conf):
    """Keep the most confident box per FDI number. xyxy in original pixels."""
    best = {}
    for b, c, s in zip(np.asarray(xyxy), np.asarray(cls).astype(int), np.asarray(conf)):
        if c not in best or s > best[c][1]:
            best[c] = (b, float(s))
    out = []
    for c, (b, s) in best.items():
        q, n = cls_to_fdi(c)
        out.append(dict(q=q, n=n, fdi=10 * q + n, box=[float(v) for v in b], conf=s))
    return out


def assign_tooth(box, teeth, min_iou=0.3):
    """Best matching tooth for a box (IoU, else nearest centre)."""
    if not teeth:
        return None, 0.0
    ious = iou_matrix([box], [t['box'] for t in teeth])[0]
    j = int(np.argmax(ious))
    if ious[j] >= min_iou:
        return teeth[j], float(ious[j])
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    d = [((t['box'][0] + t['box'][2]) / 2 - cx) ** 2 + ((t['box'][1] + t['box'][3]) / 2 - cy) ** 2 for t in teeth]
    j = int(np.argmin(d))
    return teeth[j], float(ious[j])


# ----------------------------------------------------------------------------- crops
def crop_tooth(img, box, q, H, W, mx=0.15, my_crown=0.08, my_apex=0.20):
    """Letterboxed crop with extra margin on the root-apex side (up for upper jaw, down for lower)."""
    import cv2
    x1, y1, x2, y2 = box
    w, h = max(x2 - x1, 2), max(y2 - y1, 2)
    upper = int(q) in (1, 2)
    top = my_apex if upper else my_crown
    bot = my_crown if upper else my_apex
    X1 = int(max(0, math.floor(x1 - mx * w)))
    X2 = int(min(img.shape[1], math.ceil(x2 + mx * w)))
    Y1 = int(max(0, math.floor(y1 - top * h)))
    Y2 = int(min(img.shape[0], math.ceil(y2 + bot * h)))
    c = img[Y1:Y2, X1:X2]
    if c.size == 0:
        return np.zeros((H, W), np.uint8)
    ch, cw = c.shape[:2]
    s = min(H / ch, W / cw)
    nh, nw = max(1, int(round(ch * s))), max(1, int(round(cw * s)))
    c = cv2.resize(c, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.zeros((H, W), np.uint8)
    oy, ox = (H - nh) // 2, (W - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = c
    return out


def build_crops(records, teeth_by_key, out_dir, split, H, W, add_gt_only, start_id=0, min_iou=0.3):
    """One crop per detected tooth (plus GT-only teeth if add_gt_only). Returns (rows, next_id).
    Each row carries multi-hot disease labels from overlapping GT boxes and the crop id of its
    contralateral twin (same tooth number, mirrored quadrant) or -1."""
    import cv2
    out_dir = Path(out_dir) / split
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, cid = [], start_id
    for r in records:
        teeth = [dict(t) for t in teeth_by_key.get(r['key'], [])]
        for t in teeth:
            t['labels'] = [0, 0, 0, 0]
            t['gt_only'] = 0
        gts = [a for a in r['anns'] if 'd' in a]
        extra = []
        if gts:
            gboxes = [xywh_to_xyxy(a['box']) for a in gts]
            ious = iou_matrix(gboxes, [t['box'] for t in teeth]) if teeth else np.zeros((len(gts), 0))
            for gi, a in enumerate(gts):
                j = -1
                if teeth:
                    same = [k for k, t in enumerate(teeth)
                            if 'q' in a and 'n' in a and t['q'] == a['q'] and t['n'] == a['n']]
                    if same and ious[gi, same].max() >= 0.1:
                        j = same[int(np.argmax(ious[gi, same]))]
                    elif ious[gi].max() >= min_iou:
                        j = int(np.argmax(ious[gi]))
                if j >= 0:
                    teeth[j]['labels'][a['d']] = 1
                elif add_gt_only and 'q' in a and 'n' in a:
                    lab = [0, 0, 0, 0]
                    lab[a['d']] = 1
                    # merge with an existing GT-only tooth that covers the same tooth
                    merged = False
                    for e in extra:
                        if e['q'] == a['q'] and e['n'] == a['n'] and \
                                iou_matrix([e['box']], [gboxes[gi]])[0, 0] > 0.5:
                            e['labels'][a['d']] = 1
                            merged = True
                    if not merged:
                        extra.append(dict(q=a['q'], n=a['n'], fdi=10 * a['q'] + a['n'], box=gboxes[gi],
                                          conf=1.0, labels=lab, gt_only=1))
        allt = teeth + extra
        if not allt:
            continue
        img = read_gray(r['path'])
        safe = r['key'].replace('/', '__')
        ids_by_fdi = {}
        for t in allt:
            t['crop_id'] = cid
            cid += 1
            crop = crop_tooth(img, t['box'], t['q'], H, W)
            p = out_dir / f'{safe}_{t["fdi"]}_{t["crop_id"]}.png'
            cv2.imwrite(str(p), crop)
            t['crop_path'] = str(p)
            if not t['gt_only']:
                ids_by_fdi[t['fdi']] = t['crop_id']
        for t in allt:
            tq, tn = twin_of(t['q'], t['n'])
            twin = ids_by_fdi.get(10 * tq + tn, -1)
            x1, y1, x2, y2 = t['box']
            rows.append(dict(crop_id=t['crop_id'], image_key=r['key'], split=split, q=t['q'], n=t['n'],
                             fdi=t['fdi'], x1=x1, y1=y1, x2=x2, y2=y2, det_conf=t['conf'],
                             gt_only=t['gt_only'], **{c: t['labels'][i] for i, c in enumerate(LABEL_COLS)},
                             crop_path=t['crop_path'], twin_crop_id=twin))
    return rows, cid


# ----------------------------------------------------------------------------- predictions
def match_gt_to_teeth(record, teeth, min_iou=0.3):
    """(gt index, disease, gt box xyxy, tooth index) for every GT disease box whose tooth was found:
    same tooth number first (as when labelling the crops), IoU >= min_iou so wrong-tooth pairs are dropped"""
    out = []
    if not teeth:
        return out
    for gi, a in enumerate(record['anns']):
        if 'd' not in a or 'q' not in a or 'n' not in a:
            continue
        g = xywh_to_xyxy(a['box'])
        same = [j for j, t in enumerate(teeth) if t['q'] == a['q'] and t['n'] == a['n']]
        if not same:
            continue
        ious = iou_matrix([g], [teeth[j]['box'] for j in same])[0]
        k = int(np.argmax(ious))
        if ious[k] >= min_iou:
            out.append((gi, int(a['d']), g, same[k]))
    return out


def fit_box_adapter(records, teeth_by_key, keys, min_count=20):
    """Box adapter, learned on TRAINING images only. The tooth finder learned the box style of the
    enumeration set, while the disease boxes were drawn separately. For each (disease, upper/lower jaw)
    we take the median offset of every box edge from the found tooth box to the annotated disease box,
    in units of tooth width/height. Returns ({(d, upper): [dx1, dy1, dx2, dy2]}, counts)."""
    keys = set(keys)
    offs = {}
    for r in records:
        if r['key'] not in keys:
            continue
        teeth = teeth_by_key.get(r['key'], [])
        for _, d, g, j in match_gt_to_teeth(r, teeth):
            t = teeth[j]['box']
            w, h = max(t[2] - t[0], 1.0), max(t[3] - t[1], 1.0)
            offs.setdefault((d, int(teeth[j]['q']) in (1, 2)), []).append(
                [(g[0] - t[0]) / w, (g[1] - t[1]) / h, (g[2] - t[2]) / w, (g[3] - t[3]) / h])
    adapter = {k: np.median(np.array(v), 0).tolist() for k, v in offs.items() if len(v) >= min_count}
    return adapter, {k: len(v) for k, v in offs.items()}


def adapt_box(box, d, q, adapter):
    if not adapter:
        return list(box)
    m = adapter.get((int(d), int(q) in (1, 2)))
    if m is None:
        return list(box)
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    return [x1 + m[0] * w, y1 + m[1] * h, x2 + m[2] * w, y2 + m[3] * h]


def box_iou_stats(records, teeth_by_key, keys, adapter=None):
    """IoU between found tooth boxes and the annotated disease boxes they match (raw or adapted)"""
    keys = set(keys)
    vals = []
    for r in records:
        if r['key'] not in keys:
            continue
        teeth = teeth_by_key.get(r['key'], [])
        for _, d, g, j in match_gt_to_teeth(r, teeth, min_iou=0.1):
            b = adapt_box(teeth[j]['box'], d, teeth[j]['q'], adapter)
            vals.append(float(iou_matrix([g], [b])[0, 0]))
    v = np.array(vals) if vals else np.zeros(1)
    return dict(pairs=len(vals), median_iou=round(float(np.median(v)), 3),
                below_050=round(float((v < 0.5).mean()), 3), below_075=round(float((v < 0.75).mean()), 3))


def oracle_boxes(records, teeth_by_key, keys):
    """ANALYSIS ONLY: for each found tooth and disease, the annotated box it matches (uses test labels)"""
    keys = set(keys)
    out = {}
    for r in records:
        if r['key'] not in keys:
            continue
        teeth = teeth_by_key.get(r['key'], [])
        for _, d, g, j in match_gt_to_teeth(r, teeth, min_iou=0.1):
            out[(r['key'], int(teeth[j]['fdi']), d)] = list(g)
    return out


def _rows_as_teeth(rows):
    return [dict(q=int(r['q']), n=int(r['n']), box=[r['x1'], r['y1'], r['x2'], r['y2']], crop_id=int(r['crop_id']))
            for r in rows]


def box_targets(records, man, min_iou=0.3):
    """Training targets for the learned box head: for every found tooth and every disease it has, the offset of
    each edge of the annotated disease box from the tooth box, in tooth widths/heights (original image frame)."""
    rows_by_key = {}
    for r in man[man.gt_only == 0].to_dict('records'):
        rows_by_key.setdefault(r['image_key'], []).append(r)
    out = {}
    for rec in records:
        teeth = _rows_as_teeth(rows_by_key.get(rec['key'], []))
        for _, d, g, j in match_gt_to_teeth(rec, teeth, min_iou):
            t = teeth[j]['box']
            w, h = max(t[2] - t[0], 1.0), max(t[3] - t[1], 1.0)
            row = out.setdefault(teeth[j]['crop_id'], {'crop_id': teeth[j]['crop_id']})
            iou = float(iou_matrix([g], [t])[0, 0])
            if row.get(f'iou{d}', -1) >= iou:   # two boxes of one disease on one tooth: keep the better match
                continue
            row.update({f'iou{d}': iou, f'm{d}': 1.0, f't{d}_x1': (g[0] - t[0]) / w, f't{d}_y1': (g[1] - t[1]) / h,
                        f't{d}_x2': (g[2] - t[2]) / w, f't{d}_y2': (g[3] - t[3]) / h})
    cols = ['crop_id'] + [f'm{d}' for d in range(4)] + [f't{d}_{k}' for d in range(4) for k in ('x1', 'y1', 'x2', 'y2')]
    df = pd.DataFrame(list(out.values()))
    for c in cols:
        if c not in df:
            df[c] = 0.0
    return df[cols].fillna(0.0)


def refined_box(r, off):
    w, h = r['x2'] - r['x1'], r['y2'] - r['y1']
    return [r['x1'] + off[0] * w, r['y1'] + off[1] * h, r['x2'] + off[2] * w, r['y2'] + off[3] * h]


def refined_iou_stats(records, rows_by_key, keys, refine):
    """IoU between the learned per-disease boxes and the annotated disease boxes they match"""
    keys = set(keys)
    vals = []
    for rec in records:
        if rec['key'] not in keys:
            continue
        rows = rows_by_key.get(rec['key'], [])
        teeth = _rows_as_teeth(rows)
        for _, d, g, j in match_gt_to_teeth(rec, teeth, min_iou=0.1):
            off = refine.get(teeth[j]['crop_id'])
            b = refined_box(rows[j], off[d]) if off is not None else teeth[j]['box']
            vals.append(float(iou_matrix([g], [b])[0, 0]))
    v = np.array(vals) if vals else np.zeros(1)
    return dict(pairs=len(vals), median_iou=round(float(np.median(v)), 3),
                below_050=round(float((v < 0.5).mean()), 3), below_075=round(float((v < 0.75).mean()), 3))


def preds_from_teeth(tooth_rows, probs_by_crop, conf_pow=0.5, adapter=None, oracle=None, refine=None):
    """Every detected tooth becomes 4 candidate boxes (one per disease). refine = learned per-disease box offsets
    from the classifier; adapter = median offsets; oracle (analysis only) swaps in the matched annotated box."""
    out = []
    for r in tooth_rows:
        if r['gt_only']:
            continue
        p = probs_by_crop.get(r['crop_id'])
        if p is None:
            continue
        w = max(r['det_conf'], 1e-6) ** conf_pow
        base = [r['x1'], r['y1'], r['x2'], r['y2']]
        off = refine.get(r['crop_id']) if refine is not None else None
        for d in range(4):
            box = refined_box(r, off[d]) if off is not None else adapt_box(base, d, r['q'], adapter)
            if oracle is not None:
                box = oracle.get((r['image_key'], int(r['fdi']), d), box)
            out.append(dict(image_key=r['image_key'], box=box, q=r['q'], n=r['n'], d=d,
                            score=float(p[d]) * w, src='tooth', cid=int(r['crop_id'])))
    return out


def _iou_pairs(A, B):
    """IoU of A[i] with B[i] (two aligned (n, 4) arrays of x1, y1, x2, y2)"""
    A, B = np.asarray(A, float).reshape(-1, 4), np.asarray(B, float).reshape(-1, 4)
    iw = np.clip(np.minimum(A[:, 2], B[:, 2]) - np.maximum(A[:, 0], B[:, 0]), 0, None)
    ih = np.clip(np.minimum(A[:, 3], B[:, 3]) - np.maximum(A[:, 1], B[:, 1]), 0, None)
    inter = iw * ih
    ua = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1]) + (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1]) - inter
    return inter / np.maximum(ua, 1e-9)


def transplant_boxes(cls_preds, donors, rows_by_key, min_cont=0.5, min_iou=0.5):
    """Our tooth-level predictions moved onto the detectors' box for the same tooth: the detectors place the box,
    our classifier scores it (detector boxes + classifier scores, as in DCR, Cheng et al. 2018). The DENTEX disease
    boxes are boxes around the diseased tooth, so a detector box on a tooth is a good box for any disease of it.
    For a prediction (tooth t, disease d) the donor is the highest-scoring detector box of disease d lying mostly inside
    t (containment >= min_cont), else the highest-scoring detector box of any disease on t. It is used only if it
    overlaps our own box with IoU >= min_iou; otherwise our own box stays. Scores never change.
    cls_preds: from preds_from_teeth (they carry the crop id 'cid'); donors: fused detector boxes."""
    best = {}   # (crop id, disease or -1 for any) -> (score, box)
    by_img = {}
    for p in donors:
        by_img.setdefault(p['image_key'], []).append(p)
    for key, ps in by_img.items():
        rows = [r for r in rows_by_key.get(key, []) if not r.get('gt_only', 0)]
        if not rows or not ps:
            continue
        idx = assign_by_containment([p['box'] for p in ps], [[r['x1'], r['y1'], r['x2'], r['y2']] for r in rows],
                                    min_cont)
        for p, j in zip(ps, idx):
            if j < 0:
                continue
            cid, s = int(rows[j]['crop_id']), float(p['score'])
            for k in (int(p['d']), -1):
                cur = best.get((cid, k))
                if cur is None or s > cur[0]:
                    best[(cid, k)] = (s, p['box'])
    cand = []
    for p in cls_preds:
        c = p.get('cid')
        b = None if c is None else (best.get((int(c), int(p['d']))) or best.get((int(c), -1)))
        cand.append(None if b is None else b[1])
    hit = [i for i, b in enumerate(cand) if b is not None]
    ok = np.zeros(len(cls_preds), bool)
    if hit:
        iou = _iou_pairs([cand[i] for i in hit], [cls_preds[i]['box'] for i in hit])
        ok[np.asarray(hit)[iou >= min_iou]] = True
    out = []
    for i, p in enumerate(cls_preds):
        if ok[i]:
            q = dict(p)
            q['box'] = [float(v) for v in cand[i]]
            q['src'] = 'tooth, detector box'
            out.append(q)
        else:
            out.append(p)
    return out


def preds_from_disdet(dis_by_key, teeth_by_key, probs_by_key_fdi=None, mix=0.0, min_iou=0.3):
    """Plain detector boxes, FDI taken from the matched tooth. If probs are given the score is
    det_conf^(1-mix) * p_class^mix (symmetry-aware rescoring)."""
    out = []
    for key, boxes in dis_by_key.items():
        teeth = teeth_by_key.get(key, [])
        for b in boxes:
            t, _ = assign_tooth(b['box'], teeth, min_iou)
            q, n = (t['q'], t['n']) if t else (1, 1)
            s = b['conf']
            if probs_by_key_fdi is not None and mix > 0 and t is not None:
                p = probs_by_key_fdi.get((key, t['fdi']))
                if p is not None:
                    s = (max(b['conf'], 1e-6) ** (1 - mix)) * (max(float(p[b['cls']]), 1e-6) ** mix)
            out.append(dict(image_key=key, box=list(b['box']), q=q, n=n, d=int(b['cls']), score=float(s),
                            src='det'))
    return out


def nms_merge(preds, iou_thr=0.6):
    """Class-wise NMS per image on a list of predictions."""
    out = []
    groups = {}
    for p in preds:
        groups.setdefault((p['image_key'], p['d']), []).append(p)
    for _, ps in groups.items():
        ps = sorted(ps, key=lambda z: -z['score'])
        boxes = np.array([p['box'] for p in ps])
        keep = []
        alive = np.ones(len(ps), bool)
        for i in range(len(ps)):
            if not alive[i]:
                continue
            keep.append(ps[i])
            if i + 1 < len(ps):
                ious = iou_matrix(boxes[i:i + 1], boxes[i + 1:])[0]
                alive[i + 1:] &= ious < iou_thr
        out.extend(keep)
    return out


def top_k_per_image(preds, k=100):
    by = {}
    for p in preds:
        by.setdefault(p['image_key'], []).append(p)
    out = []
    for v in by.values():
        out.extend(sorted(v, key=lambda z: -z['score'])[:k])
    return out


# ----------------------------------------------------------------------------- evaluation
def coco_eval(gt_items, pred_items, cat_ids, image_keys, verbose=False):
    """gt_items: dict(image_key, box xyxy, cat). pred_items: dict(image_key, box xyxy, cat, score).
    Returns AP, AP50, AP75, AR (COCO AR@100, averaged over IoU 0.5:0.95), per-class AP and
    best F1 per class at IoU 0.5."""
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    kid = {k: i + 1 for i, k in enumerate(sorted(set(image_keys)))}
    anns = []
    for i, g in enumerate(gt_items):
        x1, y1, x2, y2 = g['box']
        anns.append(dict(id=i + 1, image_id=kid[g['image_key']], category_id=int(g['cat']),
                         bbox=[x1, y1, x2 - x1, y2 - y1], area=(x2 - x1) * (y2 - y1), iscrowd=0))
    ds = dict(images=[dict(id=v) for v in kid.values()], annotations=anns,
              categories=[dict(id=int(c)) for c in cat_ids])
    res = []
    for p in pred_items:
        if p['image_key'] not in kid:
            continue
        x1, y1, x2, y2 = p['box']
        res.append(dict(image_id=kid[p['image_key']], category_id=int(p['cat']),
                        bbox=[x1, y1, x2 - x1, y2 - y1], score=float(p['score'])))
    out = dict(AP=0.0, AP50=0.0, AP75=0.0, AR=0.0, per_class_AP={}, best_F1_50={})
    if not res or not anns:
        return out
    ctx = contextlib.redirect_stdout(io.StringIO()) if not verbose else contextlib.nullcontext()
    with ctx:
        gt = COCO()
        gt.dataset = ds
        gt.createIndex()
        dt = gt.loadRes(res)
        E = COCOeval(gt, dt, 'bbox')
        E.params.maxDets = [1, 10, 100]
        E.evaluate()
        E.accumulate()
        E.summarize()
    st = E.stats
    out.update(AP=100 * max(st[0], 0), AP50=100 * max(st[1], 0), AP75=100 * max(st[2], 0),
               AR=max(st[8], 0))
    prec = E.eval['precision']            # [T, R, K, A, M]
    rec_thr = np.asarray(E.params.recThrs)
    for k, c in enumerate(E.params.catIds):
        p_all = prec[:, :, k, 0, -1]
        v = p_all[p_all > -1]
        out['per_class_AP'][int(c)] = 100 * float(v.mean()) if v.size else float('nan')
        p50 = prec[0, :, k, 0, -1]
        ok = p50 > -1
        if ok.any():
            pp, rr = p50[ok], rec_thr[ok]
            f1 = np.where(pp + rr > 0, 2 * pp * rr / np.maximum(pp + rr, 1e-9), 0)
            out['best_F1_50'][int(c)] = float(f1.max())
    return out


def dentex_eval(gt_records, preds, image_keys=None):
    """DENTEX-style: the same predicted boxes are scored three times, once with quadrant labels,
    once with enumeration labels and once with diagnosis labels (COCO AP, AP50, AP75, AR)."""
    keys = image_keys or [r['key'] for r in gt_records]
    keyset = set(keys)
    gts = []
    for r in gt_records:
        if r['key'] not in keyset:
            continue
        for a in r['anns']:
            if 'd' in a:
                gts.append(dict(image_key=r['key'], box=xywh_to_xyxy(a['box']),
                                q=a.get('q', 1), n=a.get('n', 1), d=a['d']))
    res = {}
    for task, field, cats in (('quadrant', 'q', [1, 2, 3, 4]), ('enumeration', 'n', list(range(1, 9))),
                              ('diagnosis', 'd', [1, 2, 3, 4])):
        off = 1 if field == 'd' else 0
        g = [dict(image_key=x['image_key'], box=x['box'], cat=x[field] + off) for x in gts]
        p = [dict(image_key=x['image_key'], box=x['box'], cat=x[field] + off, score=x['score']) for x in preds]
        res[task] = coco_eval(g, p, cats, keys)
    return res


def fast_ap(gt_items, pred_items, cat_ids, image_keys):
    """COCO AP and AP50 only (all areas, 100 detections): same numbers as coco_eval, ~4x faster (for bootstraps)"""
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    kid = {k: i + 1 for i, k in enumerate(sorted(set(image_keys)))}
    anns = [dict(id=i + 1, image_id=kid[g['image_key']], category_id=int(g['cat']),
                 bbox=[g['box'][0], g['box'][1], g['box'][2] - g['box'][0], g['box'][3] - g['box'][1]],
                 area=(g['box'][2] - g['box'][0]) * (g['box'][3] - g['box'][1]), iscrowd=0) for i, g in enumerate(gt_items)]
    res = [dict(image_id=kid[x['image_key']], category_id=int(x['cat']), score=float(x['score']),
                bbox=[x['box'][0], x['box'][1], x['box'][2] - x['box'][0], x['box'][3] - x['box'][1]])
           for x in pred_items if x['image_key'] in kid]
    if not res or not anns:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO()
        gt.dataset = dict(images=[dict(id=v) for v in kid.values()], annotations=anns,
                          categories=[dict(id=int(c)) for c in cat_ids])
        gt.createIndex()
        E = COCOeval(gt, gt.loadRes(res), 'bbox')
        E.params.areaRng, E.params.areaRngLbl, E.params.maxDets = [[0, 1e10]], ['all'], [100]
        E.evaluate()
        E.accumulate()
    p = E.eval['precision'][:, :, :, 0, 0]
    ap = p[p > -1].mean() if (p > -1).any() else 0.0
    p50 = p[0]
    ap50 = p50[p50 > -1].mean() if (p50 > -1).any() else 0.0
    return 100 * float(ap), 100 * float(ap50)


COCO_IOUS = np.linspace(.5, 0.95, int(np.round((0.95 - .5) / .05)) + 1, endpoint=True)   # as pycocotools
COCO_RECS = np.linspace(.0, 1.00, int(np.round((1.00 - .0) / .01)) + 1, endpoint=True)


def coco_iou(d, g):
    """pycocotools bbIou for xywh boxes without crowd, with its exact float operations. D x G."""
    dx, dy, dw, dh = (d[:, i:i + 1] for i in range(4))
    gx, gy, gw, gh = (g[None, :, i] for i in range(4))
    w = np.minimum(dw + dx, gw + gx) - np.maximum(dx, gx)
    h = np.minimum(dh + dy, gh + gy) - np.maximum(dy, gy)
    i = w * h
    with np.errstate(divide='ignore', invalid='ignore'):
        o = i / (dw * dh + gw * gh - i)
    return np.where((w > 0) & (h > 0), o, 0.0)


class CocoBoot:
    """Diagnosis AP / AP50 exactly as fast_ap (pycocotools, area 'all', best 100 per X-ray and class), but fast
    for bootstraps: every X-ray is matched once (matching only looks inside one X-ray), and a resample of X-rays
    only concatenates the cached matches, in the order pycocotools would see them."""

    def __init__(self, gt_records, preds, keys, cats=(1, 2, 3, 4), max_det=100):
        self.keys, self.cats = list(keys), tuple(cats)
        ks = set(self.keys)
        gt_by = {r['key']: r for r in gt_records if r['key'] in ks}
        P = {}
        for x in preds:
            if x['image_key'] in ks:
                P.setdefault((x['image_key'], int(x['d']) + 1), []).append(x)
        self.cache = {}
        for k in ks:
            anns = [a for a in gt_by[k]['anns'] if 'd' in a] if k in gt_by else []
            for c in self.cats:
                gb = [xywh_to_xyxy(a['box']) for a in anns if a['d'] + 1 == c]   # same round trip as fast_ap
                g = np.array([[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in gb], float).reshape(-1, 4)
                ps = P.get((k, c), [])
                d = np.array([[p['box'][0], p['box'][1], p['box'][2] - p['box'][0], p['box'][3] - p['box'][1]]
                              for p in ps], float).reshape(-1, 4)
                s = np.array([float(p['score']) for p in ps], float)
                self.cache[(k, c)] = self._match(g, d, s, max_det)

    @staticmethod
    def _match(g, d, s, max_det):
        """pycocotools evaluateImg: detections by score (stable), best max_det, then per IoU threshold each
        detection takes the unmatched box with the highest IoU >= threshold (the last one on ties)"""
        order = np.argsort(-s, kind='mergesort')[:max_det]
        s, d = s[order], d[order]
        m = np.zeros((len(COCO_IOUS), len(s)), bool)
        if len(s) and len(g):
            ious = coco_iou(d, g)
            for t, thr in enumerate(COCO_IOUS):
                thr = min(thr, 1 - 1e-10)
                free = np.ones(len(g), bool)
                for j in range(len(s)):
                    ok = free & (ious[j] >= thr)
                    if ok.any():
                        cand = np.flatnonzero(ok)
                        v = ious[j, cand]
                        free[cand[np.flatnonzero(v == v.max())[-1]]] = False
                        m[t, j] = True
        return s, m, len(g)

    def ap(self, sample=None):
        """(AP, AP50) in % over the X-rays sample (indices into keys, repeats allowed; None = all of them)"""
        idx = range(len(self.keys)) if sample is None else sample
        named = sorted((f'{self.keys[j]}#{i}', self.keys[j]) for i, j in enumerate(idx))   # pycocotools image order
        ks = [k for _, k in named]
        per_cat = []
        for c in self.cats:
            parts = [self.cache[(k, c)] for k in ks]
            ngt = sum(p[2] for p in parts)
            if ngt == 0:
                continue                      # no box of this class in the resample: left out, as pycocotools
            S = np.concatenate([p[0] for p in parts])
            q = np.zeros((len(COCO_IOUS), len(COCO_RECS)))
            if len(S):
                M = np.concatenate([p[1] for p in parts], axis=1)[:, np.argsort(-S, kind='mergesort')]
                tp = np.cumsum(M, axis=1).astype(float)
                fp = np.cumsum(~M, axis=1).astype(float)
                rc = tp / ngt
                pr = tp / (fp + tp + np.spacing(1))
                pr = np.maximum.accumulate(pr[:, ::-1], axis=1)[:, ::-1]
                nd = pr.shape[1]
                for t in range(len(COCO_IOUS)):
                    ii = np.searchsorted(rc[t], COCO_RECS, side='left')
                    ok = ii < nd
                    q[t, ok] = pr[t, ii[ok]]
            per_cat.append(q)
        if not per_cat:
            return 0.0, 0.0
        P = np.stack(per_cat)
        return 100 * float(P.mean()), 100 * float(P[:, 0].mean())


def paired_bootstrap(gt_records, preds_by_name, keys, ref, n_boot=500, seed=0):
    """Image-level paired bootstrap of the diagnosis AP: every resample draws len(keys) X-rays with replacement
    and scores ALL models on that same set, so model differences are paired. Returns, per model, the observed
    AP/AP50 difference to `ref` and its 95% interval. Same resamples for the same keys and seed."""
    keys = list(keys)
    boots = {n: CocoBoot(gt_records, P, keys) for n, P in preds_by_name.items()}
    obs = {n: b.ap() for n, b in boots.items()}
    rng = np.random.default_rng(seed)
    diffs = {n: [] for n in preds_by_name if n != ref}
    for _ in range(n_boot):
        sample = rng.integers(0, len(keys), len(keys))
        r0 = boots[ref].ap(sample)
        for n in diffs:
            a = boots[n].ap(sample)
            diffs[n].append((a[0] - r0[0], a[1] - r0[1]))
    out = {}
    for n, v in diffs.items():
        v = np.array(v)
        out[n] = dict(AP=round(float(obs[n][0]), 2), dAP=round(float(obs[n][0] - obs[ref][0]), 2),
                      dAP_ci=f'[{np.percentile(v[:, 0], 2.5):+.1f}, {np.percentile(v[:, 0], 97.5):+.1f}]',
                      dAP_lo=round(float(np.percentile(v[:, 0], 2.5)), 2), dAP_hi=round(float(np.percentile(v[:, 0], 97.5)), 2),
                      AP50=round(float(obs[n][1]), 2), dAP50=round(float(obs[n][1] - obs[ref][1]), 2),
                      dAP50_ci=f'[{np.percentile(v[:, 1], 2.5):+.1f}, {np.percentile(v[:, 1], 97.5):+.1f}]',
                      p_better=round(float((v[:, 0] > 0).mean()), 3))
    return out


# ----------------------------------------------------------------------------- ultralytics-style metric
# The YOLO-based DENTEX papers report the numbers printed by ultralytics' validator, which differ from COCO:
# predictions are matched to boxes in order of IoU (COCO: in order of confidence), all of them count
# (COCO: best 100 per image) and AP is the area under a linearly interpolated precision envelope.
UL_IOUV = np.linspace(0.5, 0.95, 10).astype(np.float32).astype(float)   # torch.linspace in float32, as ultralytics


def ul_match(pred_cls, true_cls, iou):
    """ultralytics BaseValidator.match_predictions (default, non-scipy). iou: labels x predictions."""
    correct = np.zeros((len(pred_cls), len(UL_IOUV)), bool)
    iou = iou * (np.asarray(true_cls)[:, None] == np.asarray(pred_cls)[None, :])
    for i, thr in enumerate(UL_IOUV):
        m = np.array(np.nonzero(iou >= thr)).T
        if m.shape[0]:
            if m.shape[0] > 1:
                m = m[iou[m[:, 0], m[:, 1]].argsort()[::-1]]
                m = m[np.unique(m[:, 1], return_index=True)[1]]
                m = m[np.unique(m[:, 0], return_index=True)[1]]
            correct[m[:, 1].astype(int), i] = True
    return correct


def ul_compute_ap(recall, precision, new_formula=True):
    """ultralytics compute_ap. new_formula: version >= 8.4.37 (precision drops to 0 after the last recall).
    Old formula (YOLOv5 ... ultralytics 8.4.36, what the 2024-2026 papers used): a straight line from the last
    point to (recall 1, precision 0), which adds area whenever recall stays below 1."""
    if new_formula:
        mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
        mpre = np.concatenate(([1.0], precision, [0.0], [0.0]))
    else:
        mrec = np.concatenate(([0.0], recall, [1.0]))
        mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    trap = getattr(np, 'trapezoid', None) or np.trapz
    return float(trap(np.interp(x, mrec, mpre), x))


def ul_ap_per_class(tp, conf, pred_cls, target_cls, new_formula=True, eps=1e-16):
    """AP per class (classes present in the labels) x 10 IoU thresholds, as ultralytics ap_per_class"""
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]
    classes, nt = np.unique(target_cls, return_counts=True)
    ap = np.zeros((len(classes), tp.shape[1]))
    for ci, c in enumerate(classes):
        k = pred_cls == c
        if k.sum() == 0 or nt[ci] == 0:
            continue
        fpc = (1 - tp[k]).cumsum(0)
        tpc = tp[k].cumsum(0)
        recall = tpc / (nt[ci] + eps)
        precision = tpc / (tpc + fpc)
        for j in range(tp.shape[1]):
            ap[ci, j] = ul_compute_ap(recall[:, j], precision[:, j], new_formula)
    return classes, ap


def ul_eval(gt_records, preds, image_keys):
    """Diagnosis mAP50-95 / mAP50 computed the way ultralytics' validator does, on the given X-rays.
    Keys without suffix: current ultralytics (>= 8.4.37, what model.val() prints on Kaggle now);
    '_old': the AP formula of YOLOv5 ... ultralytics 8.4.36, i.e. how the peer papers' numbers were made."""
    keys = sorted(set(image_keys))
    keyset = set(keys)
    gt_by = {r['key']: [(xywh_to_xyxy(a['box']), int(a['d'])) for a in r['anns'] if 'd' in a]
             for r in gt_records if r['key'] in keyset}
    p_by = {}
    for p in preds:
        if p['image_key'] in keyset:
            p_by.setdefault(p['image_key'], []).append(p)
    tps, confs, pcls, tcls = [], [], [], []
    for k in keys:
        g = gt_by.get(k, [])
        gc = np.array([c for _, c in g], int)
        tcls.append(gc)
        P = p_by.get(k, [])
        if not P:
            continue
        pc = np.array([int(p['d']) for p in P], int)
        if len(g):
            tp = ul_match(pc, gc, iou_matrix(np.array([b for b, _ in g], float), np.array([p['box'] for p in P], float)))
        else:
            tp = np.zeros((len(P), len(UL_IOUV)), bool)
        tps.append(tp); confs.append(np.array([float(p['score']) for p in P])); pcls.append(pc)
    tc = np.concatenate(tcls) if tcls else np.zeros(0, int)
    out = {}
    if not tps or not len(tc):
        for tag in ('', '_old'):
            out.update({'mAP' + tag: 0.0, 'mAP50' + tag: 0.0, 'per_class_AP50' + tag: {}})
        return out
    tp, cf, pc = np.concatenate(tps), np.concatenate(confs), np.concatenate(pcls)
    for tag, new in (('', True), ('_old', False)):
        classes, ap = ul_ap_per_class(tp, cf, pc, tc, new)
        out['mAP' + tag] = 100 * float(ap.mean())
        out['mAP50' + tag] = 100 * float(ap[:, 0].mean())
        out['per_class_AP50' + tag] = {DIS[c]: round(100 * float(ap[i, 0]), 1) for i, c in enumerate(classes)}
    return out


# ----------------------------------------------------------------------------- self-training pool
POOL_TAGS = ('enum', 'quad', 'unl')
POOL_EXPECTED = {'enum': 634, 'quad': 693, 'unl': 1571}   # DENTEX X-rays without diagnosis labels


def pool_tag(rel):
    """which DENTEX subset an image belongs to, from its path inside the data folder: the X-rays without diagnosis
    labels are the enumeration set, the quadrant-only set and the unlabelled set (None = anything else)"""
    s = '/' + str(rel).lower().replace('\\', '/')
    if any(t in s for t in ('valid', '/test', 'disease', '_decoded', '__macosx', '/._')):
        return None
    if 'unlabel' in s:
        return 'unl'
    if 'enumeration' in s:
        return 'enum'
    if 'quadrant' in s:
        return 'quad'
    return None


def find_pool(roots):
    """{tag: sorted image paths} of the X-rays without diagnosis labels; each subset is taken from the first
    folder in `roots` that has it, so a subset found twice is not used twice"""
    out = {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        here = {}
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                t = pool_tag(p.relative_to(root))
                if t:
                    here.setdefault(t, set()).add(str(p))
        for t, v in here.items():
            out.setdefault(t, sorted(v))
    return out


def thumb_of(im):
    """64x32 grey thumbnail used to find copies of the same X-ray"""
    import cv2
    return cv2.resize(im, (64, 32), interpolation=cv2.INTER_AREA).astype(np.float32)


def thumb_file(path):
    return thumb_of(read_gray(path))


def export_pool_image(task):
    """(source, jpg, long_side) -> (W, H, scale, thumbnail), or None if the file is not a readable image.
    The jpg is made exactly like resize_long's."""
    import cv2
    src, dst, long_side = task
    try:
        im = read_gray(src)
    except Exception:
        return None
    h, w = im.shape[:2]
    s = min(1.0, long_side / max(h, w))
    if not Path(dst).exists():
        small = cv2.resize(im, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA) if s < 1 else im
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(dst), small, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return w, h, s, thumb_of(im)


def build_crops_chunk(task):
    """build_crops for one chunk of X-rays (a picklable task for pmap); returns the rows"""
    return build_crops(*task)[0]


def pmap(fn, tasks, workers=4, chunksize=4):
    """map fn over tasks in worker processes ('spawn': safe in a notebook that already uses CUDA and threads);
    falls back to one process if that fails"""
    tasks = list(tasks)
    if workers > 1 and len(tasks) > 8:
        try:
            import multiprocessing as mp
            from concurrent.futures import ProcessPoolExecutor
            with ProcessPoolExecutor(workers, mp_context=mp.get_context('spawn')) as ex:
                return list(ex.map(fn, tasks, chunksize=chunksize))
        except Exception as e:
            print('   parallel map failed, running in one process:', str(e)[:200])
    return [fn(t) for t in tasks]


def _standardize(x):
    x = np.asarray(x, np.float64).reshape(len(x), -1)
    return (x - x.mean(1, keepdims=True)) / np.maximum(x.std(1, keepdims=True), 1e-6)


def rms_nearest(a, b=None):
    """RMS difference between standardised thumbnails (zero mean, unit variance, so a copy saved with other
    brightness or contrast still counts) from each one in a to its nearest one in b (to the nearest EARLIER one in a
    when b is None). sqrt(2 - 2 * correlation): ~0.05 for the same X-ray saved twice. Returns (distance, index)."""
    A = _standardize(a)
    B = A if b is None else _standardize(b)
    D = A.shape[1]
    if len(A) == 0 or len(B) == 0:
        return np.full(len(A), np.inf), np.full(len(A), -1)
    d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    d2 = np.maximum(d2, 0) / D
    if b is None:
        d2[np.triu_indices(len(A))] = np.inf     # only earlier images count
    j = d2.argmin(1)
    return np.sqrt(d2[np.arange(len(A)), j]), j


def assign_by_containment(boxes, tooth_boxes, min_cont=None):
    """index of the tooth each box belongs to: the tooth box holding the largest part of it. A box outside every
    tooth box goes to the tooth with the nearest centre, or, with min_cont, every box with less than that fraction
    inside its best tooth box gets -1 (also -1 when there are no teeth)"""
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    T = np.asarray(tooth_boxes, float).reshape(-1, 4)
    if len(T) == 0:
        return np.full(len(boxes), -1)
    if len(boxes) == 0:
        return np.zeros(0, int)
    x1 = np.maximum(boxes[:, None, 0], T[None, :, 0]); y1 = np.maximum(boxes[:, None, 1], T[None, :, 1])
    x2 = np.minimum(boxes[:, None, 2], T[None, :, 2]); y2 = np.minimum(boxes[:, None, 3], T[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = np.maximum((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]), 1e-9)
    cont = inter / area[:, None]
    j = cont.argmax(1)
    c = (boxes[:, :2] + boxes[:, 2:]) / 2
    tc = (T[:, :2] + T[:, 2:]) / 2
    near = ((c[:, None, :] - tc[None]) ** 2).sum(-1).argmin(1)
    if min_cont is not None:
        return np.where(cont.max(1) >= min_cont, j, -1)
    return np.where(cont.max(1) > 0, j, near)


def tooth_scores(preds, rows_by_key, min_cont=0.3):
    """{crop_id: [4 scores]}: per tooth and disease, the best score of the predicted boxes that belong to that tooth
    (assign_by_containment; a box with less than min_cont of its area inside any tooth box is ignored, e.g. an impacted
    tooth the tooth detector missed). preds: dict(image_key, box, d, score) as made by preds_from_disdet."""
    out = {}
    by = {}
    for p in preds:
        by.setdefault(p['image_key'], []).append(p)
    for key, rows in rows_by_key.items():
        rows = [r for r in rows if not r.get('gt_only', 0)]
        for r in rows:
            out[int(r['crop_id'])] = np.zeros(4)
        ps = by.get(key, [])
        if not ps or not rows:
            continue
        idx = assign_by_containment([p['box'] for p in ps], [[r['x1'], r['y1'], r['x2'], r['y2']] for r in rows],
                                    min_cont)
        for p, j in zip(ps, idx):
            if j < 0:
                continue
            v = out[int(rows[j]['crop_id'])]
            v[p['d']] = max(v[p['d']], float(p['score']))
    return out


def f1_threshold(gt_records, preds, keys, d=None, iou=0.5):
    """score threshold with the best F1 at IoU >= iou (each annotated box matched once, highest scores first, per
    X-ray and disease). d = one disease, None = all four pooled. Returns dict(thr, f1, p, r, n_gt)."""
    keys = set(keys)
    gts = {}
    for r in gt_records:
        if r['key'] not in keys:
            continue
        for a in r['anns']:
            if 'd' in a and (d is None or a['d'] == d):
                gts.setdefault((r['key'], a['d']), []).append(xywh_to_xyxy(a['box']))
    n_gt = sum(len(v) for v in gts.values())
    cand = sorted([p for p in preds if p['image_key'] in keys and (d is None or p['d'] == d)], key=lambda z: -z['score'])
    used = {k: np.zeros(len(v), bool) for k, v in gts.items()}
    tp = np.zeros(len(cand))
    for i, p in enumerate(cand):
        g = gts.get((p['image_key'], p['d']))
        if not g:
            continue
        ious = iou_matrix([p['box']], g)[0]
        ious[used[(p['image_key'], p['d'])]] = -1
        j = int(np.argmax(ious))
        if ious[j] >= iou:
            used[(p['image_key'], p['d'])][j] = True
            tp[i] = 1
    if not len(cand) or n_gt == 0:
        return dict(thr=1.0, f1=0.0, p=0.0, r=0.0, n_gt=n_gt)
    ctp = np.cumsum(tp)
    prec, rec = ctp / np.arange(1, len(cand) + 1), ctp / n_gt
    f1 = np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-9), 0)
    k = int(np.argmax(f1))
    return dict(thr=float(cand[k]['score']), f1=float(f1[k]), p=float(prec[k]), r=float(rec[k]), n_gt=n_gt)


def yolo_label_lines(boxes, W, H):
    """YOLO label lines 'cls cx cy w h' (normalised) for (d, x1, y1, x2, y2) boxes in original pixels"""
    out = []
    for d, x1, y1, x2, y2 in boxes:
        x1, x2 = min(max(x1, 0), W), min(max(x2, 0), W)
        y1, y2 = min(max(y1, 0), H), min(max(y2, 0), H)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        out.append(f'{int(d)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}')
    return out


# ----------------------------------------------------------------------------- box fusion
try:   # numba is in the Kaggle image; without it the same algorithm runs in numpy (~6x slower, same result)
    from numba import njit as _njit
except Exception:
    _njit = None


def _wbf_group_np(S, B, thr):
    """one X-ray and disease: boxes B (sorted by weighted score S, highest first) -> clusters as running sums"""
    n = len(S)
    fused, acc = np.empty((n, 4)), np.empty((n, 4))
    ssum, cnt, area, top = np.empty(n), np.zeros(n, np.int64), np.empty(n), np.empty(n, np.int64)
    k = 0
    for i in range(n):
        b = B[i]
        j = -1
        ab = (b[2] - b[0]) * (b[3] - b[1])
        if k:
            f = fused[:k]
            iw = np.minimum(f[:, 2], b[2]) - np.maximum(f[:, 0], b[0])
            ih = np.minimum(f[:, 3], b[3]) - np.maximum(f[:, 1], b[1])
            inter = np.clip(iw, 0, None) * np.clip(ih, 0, None)
            iou = inter / np.maximum(area[:k] + ab - inter, 1e-9)
            j = int(np.argmax(iou))
            if iou[j] <= thr:
                j = -1
        s = S[i]
        if j >= 0:
            acc[j] += s * b
            ssum[j] += s
            cnt[j] += 1
            fb = acc[j] / max(ssum[j], 1e-12)
            fused[j] = fb
            area[j] = (fb[2] - fb[0]) * (fb[3] - fb[1])
        else:
            fused[k], acc[k], ssum[k], cnt[k], area[k], top[k] = b, s * b, s, 1, ab, i
            k += 1
    return fused[:k], ssum[:k], cnt[:k], top[:k]


if _njit is not None:
    @_njit(cache=False)
    def _wbf_group_nb(S, B, thr):
        n = S.shape[0]
        fused = np.empty((n, 4))
        acc = np.empty((n, 4))
        ssum = np.empty(n)
        cnt = np.zeros(n, np.int64)
        area = np.empty(n)
        top = np.empty(n, np.int64)
        k = 0
        for i in range(n):
            bx1, by1, bx2, by2 = B[i, 0], B[i, 1], B[i, 2], B[i, 3]
            ab = (bx2 - bx1) * (by2 - by1)
            best = -1.0
            j = -1
            for c in range(k):
                iw = min(fused[c, 2], bx2) - max(fused[c, 0], bx1)
                ih = min(fused[c, 3], by2) - max(fused[c, 1], by1)
                inter = max(iw, 0.0) * max(ih, 0.0)
                den = area[c] + ab - inter
                if den < 1e-9:
                    den = 1e-9
                iou = inter / den
                if iou > best:
                    best = iou
                    j = c
            if j >= 0 and best <= thr:
                j = -1
            s = S[i]
            if j >= 0:
                for q in range(4):
                    acc[j, q] += s * B[i, q]
                ssum[j] += s
                cnt[j] += 1
                den = ssum[j] if ssum[j] > 1e-12 else 1e-12
                for q in range(4):
                    fused[j, q] = acc[j, q] / den
                area[j] = (fused[j, 2] - fused[j, 0]) * (fused[j, 3] - fused[j, 1])
            else:
                for q in range(4):
                    fused[k, q] = B[i, q]
                    acc[k, q] = s * B[i, q]
                ssum[k] = s
                cnt[k] = 1
                area[k] = ab
                top[k] = i
                k += 1
        return fused[:k].copy(), ssum[:k].copy(), cnt[:k].copy(), top[:k].copy()
else:
    _wbf_group_nb = None


def wbf(pred_lists, weights=None, iou_thr=0.55):
    """Weighted boxes fusion (Solovyev et al. 2021; same result as ensemble_boxes 1.0.9 with conf_type='avg'),
    per X-ray and disease. Boxes of the models are clustered (IoU > iou_thr with the running fused box), the fused
    box is the score-weighted mean of its cluster and the score is the mean weighted score times
    min(#models, #boxes) / sum(weights), so a box that only one model found keeps a lower score.
    The FDI labels come from the best box of the cluster."""
    w = np.ones(len(pred_lists)) if weights is None else np.asarray(weights, float)
    groups = {}
    for t, P in enumerate(pred_lists):
        wt = w[t]
        for p in P:
            groups.setdefault((p['image_key'], int(p['d'])), []).append((float(p['score']) * wt, p))
    global _wbf_group_nb
    fn = _wbf_group_nb if _wbf_group_nb is not None else _wbf_group_np
    out = []
    n_models, wsum = len(w), float(w.sum())
    for (key, d), items in groups.items():
        items.sort(key=lambda z: -z[0])
        n = len(items)
        S = np.fromiter((x[0] for x in items), float, n)
        B = np.array([x[1]['box'] for x in items], float).reshape(n, 4)
        try:
            fused, ssum, cnt, top = fn(S, B, float(iou_thr))
        except Exception:
            if fn is _wbf_group_np:
                raise
            print('   numba WBF failed, using the numpy version (same result, slower)')
            _wbf_group_nb, fn = None, _wbf_group_np
            fused, ssum, cnt, top = fn(S, B, float(iou_thr))
        for j in range(len(ssum)):
            t = items[top[j]][1]
            out.append(dict(image_key=key, box=[float(v) for v in fused[j]], q=t.get('q', 1), n=t.get('n', 1),
                            d=d, score=float(ssum[j] / cnt[j] * min(n_models, cnt[j]) / wsum), src='wbf'))
    return out


def diag_eval_fast(gt_records, preds, image_keys):
    """diagnosis AP (0.5:0.95), AP50 and AP per disease {1..4: AP}: the same numbers as
    dentex_eval(...)['diagnosis'] (pycocotools, area 'all', best 100 per X-ray and disease), about 4x faster"""
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    keys = set(image_keys)
    kid = {k: i + 1 for i, k in enumerate(sorted(keys))}
    anns = []
    for r in gt_records:
        if r['key'] not in keys:
            continue
        for a in r['anns']:
            if 'd' in a:
                x, y, w, h = a['box']
                anns.append(dict(id=len(anns) + 1, image_id=kid[r['key']], category_id=int(a['d']) + 1,
                                 bbox=[x, y, w, h], area=w * h, iscrowd=0))
    res = [dict(image_id=kid[p['image_key']], category_id=int(p['d']) + 1, score=float(p['score']),
                bbox=[p['box'][0], p['box'][1], p['box'][2] - p['box'][0], p['box'][3] - p['box'][1]])
           for p in preds if p['image_key'] in kid]
    out = dict(AP=0.0, AP50=0.0, per_class_AP={})
    if not res or not anns:
        return out
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO()
        gt.dataset = dict(images=[dict(id=v) for v in kid.values()], annotations=anns,
                          categories=[dict(id=c) for c in (1, 2, 3, 4)])
        gt.createIndex()
        E = COCOeval(gt, gt.loadRes(res), 'bbox')
        E.params.areaRng, E.params.areaRngLbl, E.params.maxDets = [[0, 1e10]], ['all'], [100]
        E.evaluate()
        E.accumulate()
    p = E.eval['precision'][:, :, :, 0, 0]
    v = p[p > -1]
    out['AP'] = 100 * float(v.mean()) if v.size else 0.0
    p50 = p[0][p[0] > -1]
    out['AP50'] = 100 * float(p50.mean()) if p50.size else 0.0
    for k, c in enumerate(E.params.catIds):
        pk = p[:, :, k]
        pk = pk[pk > -1]
        out['per_class_AP'][int(c)] = 100 * float(pk.mean()) if pk.size else float('nan')
    return out


def summarize(res, name=''):
    d = res['diagnosis']
    pc = {DIS[c - 1]: round(v, 1) for c, v in d['per_class_AP'].items()}
    f1 = {DIS[c - 1]: round(v, 3) for c, v in d['best_F1_50'].items()}
    return dict(model=name,
                diag_AP=round(d['AP'], 2), diag_AP50=round(d['AP50'], 2), diag_AP75=round(d['AP75'], 2),
                diag_AR=round(d['AR'], 4),
                enum_AP=round(res['enumeration']['AP'], 2), enum_AP50=round(res['enumeration']['AP50'], 2),
                quad_AP=round(res['quadrant']['AP'], 2), quad_AP50=round(res['quadrant']['AP50'], 2),
                per_class_AP=pc, best_F1_iou50=f1)

# ----------------------------------------------------------------------------- v15 audit
def orb_inliers(task):
    """(path_a, path_b) -> number of ORB keypoint matches between two X-rays (1024 px, CLAHE) that one RANSAC
    homography explains. The same X-ray gives hundreds; two X-rays of the same patient (the same teeth, fillings and
    crowns) usually many; different patients few. -1 if an image cannot be read."""
    import cv2
    pa, pb = task

    def load(p):
        im = read_gray(p)
        s = 1024 / max(im.shape[:2])
        if s < 1:
            im = cv2.resize(im, (int(round(im.shape[1] * s)), int(round(im.shape[0] * s))), interpolation=cv2.INTER_AREA)
        return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(im)
    try:
        a, b = load(pa), load(pb)
        orb = cv2.ORB_create(nfeatures=1500)
        ka, da = orb.detectAndCompute(a, None)
        kb, db = orb.detectAndCompute(b, None)
        if da is None or db is None or len(ka) < 10 or len(kb) < 10:
            return 0
        good = [m[0] for m in cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(da, db, k=2)
                if len(m) == 2 and m[0].distance < 0.75 * m[1].distance]
        if len(good) < 8:
            return 0
        src = np.float32([ka[g.queryIdx].pt for g in good])
        dst = np.float32([kb[g.trainIdx].pt for g in good])
        _, mask = cv2.findHomography(src, dst, cv2.RANSAC, 10.0)
        return int(mask.sum()) if mask is not None else 0
    except Exception:
        return -1


def tooth_ious(gt_teeth, pred_teeth):
    """per annotated tooth (dicts with q, n and an xyxy box): IoU with the predicted tooth of the same FDI number,
    0 when the prediction has no such tooth"""
    pred = {(int(t['q']), int(t['n'])): t['box'] for t in pred_teeth}
    out = []
    for t in gt_teeth:
        b = pred.get((int(t['q']), int(t['n'])))
        out.append(float(iou_matrix([t['box']], [b])[0, 0]) if b is not None else 0.0)
    return out


def pair_montage(pairs, out_path, h=200):
    """[(path_a, path_b, label)] -> one JPEG with each pair side by side and its label (to check near copies by eye)"""
    import cv2
    rows = []
    for pa, pb, label in pairs:
        ims = []
        for p in (pa, pb):
            try:
                im = read_gray(p)
                im = cv2.resize(im, (int(round(im.shape[1] * h / im.shape[0])), h), interpolation=cv2.INTER_AREA)
            except Exception:
                im = np.zeros((h, 2 * h), np.uint8)
            ims.append(im)
        wmax = 2 * h * 2
        canvas = np.full((h + 28, wmax + 10), 255, np.uint8)
        x = 0
        for im in ims:
            w = min(im.shape[1], wmax // 2)
            canvas[28:28 + h, x:x + w] = im[:, :w]
            x += wmax // 2 + 10
        cv2.putText(canvas, str(label)[:120], (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, 0, 1, cv2.LINE_AA)
        rows.append(canvas)
    if not rows:
        return None
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), np.vstack(rows), [cv2.IMWRITE_JPEG_QUALITY, 85])
    return str(out_path)


In [ ]:
%%writefile train_arch.py
#!/usr/bin/env python
"""TwinArch: contralateral-aware dental arch reasoning for tooth-level diagnosis.

One X-ray = one set of teeth. Every tooth crop is encoded ONCE by a small CNN. Crops from the
image-right side (quadrants 2 and 3) are mirrored before encoding, so a tooth and its contralateral
twin (16<->26, 36<->46, ...) end up in the same orientation and their features can be compared
directly. Then:

  single : head(f_i + FDI_i)                                  no context (plain crop classifier)
  twin   : head([f_i + FDI_i, |f_i - f_twin|, has_twin])      pairwise asymmetry only
  arch   : relation-biased transformer over all teeth          arch context, no explicit asymmetry
  full   : arch + asymmetry token input + asymmetry in head    OURS

The transformer's attention gets a learned bias per tooth relation (self, twin, neighbour,
antagonist, same quadrant, other), so anatomy is built into the attention instead of being learned
from scratch on 705 X-rays.

2 GPUs:  torchrun --standalone --nproc_per_node=2 train_arch.py ...
1 GPU :  python train_arch.py ...
"""
import os, json, math, time, random, argparse
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

LABELS = ['y_caries', 'y_deep', 'y_peri', 'y_impacted']
MEAN, STD = 0.449, 0.226
TWIN_Q = {1: 2, 2: 1, 3: 4, 4: 3}
ANTAG_Q = {1: 4, 4: 1, 2: 3, 3: 2}
REL_SELF, REL_TWIN, REL_NEIGH, REL_ANTAG, REL_QUAD, REL_OTHER = range(6)
N_REL = 6


# ----------------------------------------------------------------------------- args / ddp
def get_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--manifest', required=True)
    ap.add_argument('--train-keys', default='', help='image keys to train on (required unless --predict-only)')
    ap.add_argument('--val-keys', default='', help='image keys for checkpoint selection (required unless --predict-only)')
    ap.add_argument('--pred-keys', nargs='*', default=[], help='name=path pairs')
    ap.add_argument('--variant', default='full', choices=['single', 'twin', 'arch', 'full'])
    ap.add_argument('--backbone', default='convnext_nano.in12k_ft_in1k')
    ap.add_argument('--epochs', type=int, default=20)
    ap.add_argument('--img-bs', type=int, default=2, help='X-rays per GPU per step')
    ap.add_argument('--lr', type=float, default=1e-4)
    ap.add_argument('--head-lr', type=float, default=5e-4)
    ap.add_argument('--wd', type=float, default=0.05)
    ap.add_argument('--dim', type=int, default=256)
    ap.add_argument('--layers', type=int, default=2)
    ap.add_argument('--heads', type=int, default=4)
    ap.add_argument('--max-teeth', type=int, default=48)
    ap.add_argument('--crop-h', type=int, default=384)
    ap.add_argument('--crop-w', type=int, default=224)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--out', required=True)
    ap.add_argument('--no-pretrained', action='store_true')
    ap.add_argument('--max-images', type=int, default=0, help='smoke test: cap X-rays per split')
    ap.add_argument('--no-cache', action='store_true', help='read crops from disk every time')
    ap.add_argument('--drop-path', type=float, default=0.1, help='stochastic depth in the encoder')
    ap.add_argument('--ema', type=float, default=0.997, help='weight averaging decay, 0 = off')
    ap.add_argument('--box-targets', default='', help='csv from the notebook: annotated-box offsets per tooth')
    ap.add_argument('--box-w', type=float, default=4.0, help='weight of the learned-box loss')
    ap.add_argument('--box-src', default='both', choices=['token', 'map', 'both'],
                    help='learned boxes from the pooled tooth token, from the spatial feature map, or both '
                         '(two detached heads trained side by side; pred files get b* and s* columns)')
    ap.add_argument('--predict-only', action='store_true', help='no training: load --weights and write predictions')
    ap.add_argument('--weights', default='', help='state dict for --predict-only (a best.pt of an earlier run)')
    ap.add_argument('--tta', type=int, default=1, help='test-time augmentation: average over this many fixed views '
                                                        '(1 = off, up to 5: identity, zoom in/out, rotate +/-3 deg)')
    ap.add_argument('--pool-ratio', type=float, default=1.0,
                    help='self-training: pseudo-labelled X-rays (manifest rows with w < 1) drawn per annotated X-ray in '
                         'each epoch; 0 = the whole pool every epoch')
    a = ap.parse_args(argv)
    if not a.predict_only and not (a.train_keys and a.val_keys):
        ap.error('--train-keys and --val-keys are required for training')
    if a.predict_only and not a.weights:
        ap.error('--predict-only needs --weights')
    a.tta = max(1, min(int(a.tta), len(TTA_VIEWS)))
    return a


def setup_ddp():
    world = int(os.environ.get('WORLD_SIZE', '1'))
    if world > 1:
        backend = 'nccl' if torch.cuda.is_available() else 'gloo'
        local = int(os.environ.get('LOCAL_RANK', 0))
        try:
            if torch.cuda.is_available():
                dist.init_process_group(backend=backend, device_id=torch.device(f'cuda:{local}'))
            else:
                dist.init_process_group(backend=backend)
        except TypeError:          # older torch without device_id
            dist.init_process_group(backend=backend)
        rank, local = dist.get_rank(), int(os.environ.get('LOCAL_RANK', 0))
        if torch.cuda.is_available():
            torch.cuda.set_device(local)
        return rank, world, local
    return 0, 1, 0


def barrier(world):
    if world > 1:
        dist.barrier()


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def read_keys(p):
    return set(x.strip() for x in Path(p).read_text().splitlines() if x.strip())


# ----------------------------------------------------------------------------- numpy helpers
def load_crop(path, H, W):
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None:
        return np.zeros((H, W), np.float32)
    if im.shape != (H, W):
        im = cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)
    return im.astype(np.float32) / 255.0


def preload(paths, H, W):
    """read every crop once into RAM as uint8; DataLoader workers share it after fork"""
    cache = {}
    for p in paths:
        im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if im is None:
            im = np.zeros((H, W), np.uint8)
        elif im.shape != (H, W):
            im = cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)
        cache[str(p)] = im
    return cache


def sample_photo(rng):
    """exposure settings drawn ONCE per X-ray and shared by all its teeth. Exposure belongs to the image,
    and drawing it per tooth would fake left-right density differences that TwinArch then learns to ignore."""
    use_cb = rng.random() < 0.8
    return dict(contrast=rng.uniform(0.8, 1.25) if use_cb else 1.0,
                bright=rng.uniform(-0.08, 0.08) if use_cb else 0.0,
                gamma=rng.uniform(0.75, 1.35) if rng.random() < 0.3 else None,
                blur=rng.uniform(0.3, 1.0) if rng.random() < 0.2 else None,
                noise=0.02 if rng.random() < 0.2 else 0.0)


def augment(a, rng, photo):
    """returns (augmented crop, 2x3 affine used or None)"""
    H, W = a.shape
    M = None
    if rng.random() < 0.9:     # small geometric jitter stays per tooth, like box noise from the tooth detector
        M = cv2.getRotationMatrix2D((W / 2, H / 2), rng.uniform(-7, 7), rng.uniform(0.9, 1.1))
        M[:, 2] += (rng.uniform(-0.05, 0.05) * W, rng.uniform(-0.05, 0.05) * H)
        a = cv2.warpAffine(a, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if photo['contrast'] != 1.0 or photo['bright'] != 0.0:
        a = np.clip((a - 0.5) * photo['contrast'] + 0.5 + photo['bright'], 0, 1)
    if photo['gamma'] is not None:
        a = np.clip(a, 0, 1) ** photo['gamma']
    if photo['blur'] is not None:
        a = cv2.GaussianBlur(a, (3, 3), photo['blur'])
    if photo['noise'] > 0:     # sensor noise is independent per pixel anyway
        a = np.clip(a + rng.normal(0, photo['noise'], a.shape).astype(np.float32), 0, 1)
    return a.astype(np.float32), M


TTA_VIEWS = [None, (0.0, 1.06), (0.0, 0.94), (3.0, 1.0), (-3.0, 1.0)]   # (rotation deg, scale), inside training jitter


def tta_matrix(view, H, W):
    return None if view is None else cv2.getRotationMatrix2D((W / 2, H / 2), view[0], view[1])


def tooth_frame(w, h, upper, H, W, mx=0.15, my_crown=0.08, my_apex=0.20):
    """where the found tooth box sits inside its letterboxed crop (same geometry as dentex_utils.crop_tooth)"""
    top, bot = (my_apex, my_crown) if upper else (my_crown, my_apex)
    RW, RH = w * (1 + 2 * mx), h * (1 + top + bot)
    s = min(H / RH, W / RW)
    ox, oy = (W - RW * s) / 2, (H - RH * s) / 2
    fx1, fy1 = ox + mx * w * s, oy + top * h * s
    return fx1, fy1, fx1 + w * s, fy1 + h * s


def flip_offsets(o):
    """edge offsets [dx1, dy1, dx2, dy2] under a left-right mirror of the crop (its own inverse)"""
    return np.stack([-o[..., 2], o[..., 1], -o[..., 0], o[..., 3]], -1)


def warp_offsets(o, frame, M):
    """move the annotated box with the same affine as the crop, measured against the fixed tooth frame"""
    fx1, fy1, fx2, fy2 = frame
    FW, FH = max(fx2 - fx1, 1e-3), max(fy2 - fy1, 1e-3)
    x1, y1, x2, y2 = fx1 + o[0] * FW, fy1 + o[1] * FH, fx2 + o[2] * FW, fy2 + o[3] * FH
    pts = np.array([[x1, y1, 1], [x2, y1, 1], [x1, y2, 1], [x2, y2, 1]], np.float64) @ np.asarray(M, np.float64).T
    nx1, ny1 = pts[:, 0].min(), pts[:, 1].min()
    nx2, ny2 = pts[:, 0].max(), pts[:, 1].max()
    return np.array([(nx1 - fx1) / FW, (ny1 - fy1) / FH, (nx2 - fx2) / FW, (ny2 - fy2) / FH], np.float32)


def twin_index(qs, ns, gt_only):
    """index of the contralateral twin inside the same X-ray (prefer detected teeth), else -1"""
    T = len(qs)
    out = np.full(T, -1, np.int64)
    for i in range(T):
        cand = [j for j in range(T) if j != i and qs[j] == TWIN_Q[int(qs[i])] and ns[j] == ns[i]]
        if cand:
            cand.sort(key=lambda j: gt_only[j])
            out[i] = cand[0]
    return out


def relations(qs, ns):
    qs = np.asarray(qs, np.int64)
    ns = np.asarray(ns, np.int64)
    qi, qj = qs[:, None], qs[None, :]
    ni, nj = ns[:, None], ns[None, :]
    twin_q = np.vectorize(TWIN_Q.get)(qs)[:, None] if len(qs) else qs[:, None]
    antag_q = np.vectorize(ANTAG_Q.get)(qs)[:, None] if len(qs) else qs[:, None]
    R = np.full((len(qs), len(qs)), REL_OTHER, np.int64)
    R[qi == qj] = REL_QUAD
    R[(qj == antag_q) & (nj == ni)] = REL_ANTAG
    R[(qi == qj) & (np.abs(ni - nj) == 1)] = REL_NEIGH
    R[(qj == twin_q) & (nj == ni)] = REL_TWIN
    np.fill_diagonal(R, REL_SELF)
    return R


# ----------------------------------------------------------------------------- dataset
class ArchDS(Dataset):
    """one item = one X-ray with all its tooth crops"""
    def __init__(self, df, H, W, train, max_teeth=48, seed=0, cache=None, boxes=None, view=None):
        self.groups = [g.reset_index(drop=True) for _, g in df.groupby('image_key', sort=True)]
        self.H, self.W, self.train, self.max_teeth, self.seed = H, W, train, max_teeth, seed
        self.cache = cache
        self.boxes = boxes    # {crop_id: (offsets [4 diseases, 4 edges], mask [4])} in the original image frame
        self.view = tta_matrix(view, H, W) if not train else None   # fixed test-time view
        # X-rays whose teeth carry the teacher's soft labels (weight w < 1) instead of annotations
        self.is_pool = [bool('w' in g and (g['w'] < 1).any()) for g in self.groups]

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, i):
        g = self.groups[i]
        if len(g) > self.max_teeth:
            g = g.sort_values(['gt_only', 'det_conf'], ascending=[False, False]).head(self.max_teeth)
            g = g.reset_index(drop=True)
        rng = np.random.default_rng() if self.train else None
        photo = sample_photo(rng) if self.train else None
        qs, ns = g['q'].values.astype(int), g['n'].values.astype(int)
        crops = []
        T = len(g)
        bt, bm = np.zeros((T, 4, 4), np.float32), np.zeros((T, 4), np.float32)
        ws, hs = (g['x2'] - g['x1']).values, (g['y2'] - g['y1']).values
        for i_t, (p, q, cid) in enumerate(zip(g['crop_path'], qs, g['crop_id'].values)):
            if self.cache is not None and p in self.cache:
                a = self.cache[p].astype(np.float32) / 255.0
            else:
                a = load_crop(p, self.H, self.W)
            tb = self.boxes.get(int(cid)) if self.boxes is not None else None
            if tb is not None:
                bt[i_t], bm[i_t] = tb
            if q in (2, 3):                 # canonical orientation: mirror the image-right side
                a = np.ascontiguousarray(a[:, ::-1])
                bt[i_t] = flip_offsets(bt[i_t])
            if self.train:
                a, M = augment(a, rng, photo)
                if M is not None and bm[i_t].any():
                    fr = tooth_frame(max(ws[i_t], 1.0), max(hs[i_t], 1.0), q in (1, 2), self.H, self.W)
                    for d in np.flatnonzero(bm[i_t]):
                        bt[i_t, d] = warp_offsets(bt[i_t, d], fr, M)
            elif self.view is not None:
                a = cv2.warpAffine(a, self.view, (self.W, self.H), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            crops.append((a - MEAN) / STD)
        return dict(x=np.stack(crops)[:, None].astype(np.float32), fdi=((qs - 1) * 8 + (ns - 1)).astype(np.int64),
                    twin=twin_index(qs, ns, g['gt_only'].values), rel=relations(qs, ns),
                    y=g[LABELS].values.astype(np.float32), ids=g['crop_id'].values.astype(np.int64),
                    gt_only=g['gt_only'].values.astype(np.int64), bt=bt, bm=bm * (1 - g['gt_only'].values[:, None]),
                    w=(g['w'].values if 'w' in g else np.ones(T)).astype(np.float32))


def collate(batch):
    B = len(batch)
    T = max(len(b['fdi']) for b in batch)
    H, W = batch[0]['x'].shape[-2:]
    x = np.zeros((B, T, 1, H, W), np.float32)
    fdi = np.zeros((B, T), np.int64)
    twin = np.full((B, T), -1, np.int64)
    rel = np.full((B, T, T), REL_OTHER, np.int64)
    y = np.zeros((B, T, 4), np.float32)
    ids = np.full((B, T), -1, np.int64)
    gto = np.ones((B, T), np.int64)
    mask = np.zeros((B, T), bool)
    bt, bm = np.zeros((B, T, 4, 4), np.float32), np.zeros((B, T, 4), np.float32)
    w = np.zeros((B, T), np.float32)
    for i, b in enumerate(batch):
        t = len(b['fdi'])
        x[i, :t], fdi[i, :t], twin[i, :t], rel[i, :t, :t] = b['x'], b['fdi'], b['twin'], b['rel']
        y[i, :t], ids[i, :t], gto[i, :t], mask[i, :t] = b['y'], b['ids'], b['gt_only'], True
        bt[i, :t], bm[i, :t], w[i, :t] = b['bt'], b['bm'], b['w']
    return {k: torch.from_numpy(v) for k, v in
            dict(x=x, fdi=fdi, twin=twin, rel=rel, y=y, ids=ids, gt_only=gto, mask=mask, bt=bt, bm=bm, w=w).items()}


class MixSampler(torch.utils.data.Sampler):
    """Self-training epochs: every annotated X-ray once plus `ratio` times as many pseudo-labelled ones, shuffled
    together and split evenly over the GPUs. The pseudo-labelled X-rays are taken in order from one random permutation
    of the pool after another, so all of them are used before any repeats."""
    def __init__(self, is_pool, ratio, world, rank, seed):
        is_pool = np.asarray(is_pool, bool)
        self.lab, self.pool = np.flatnonzero(~is_pool), np.flatnonzero(is_pool)
        n = len(self.pool) if ratio <= 0 else int(round(ratio * len(self.lab)))
        self.n_pool = min(len(self.pool), n)
        self.world, self.rank, self.seed, self.epoch = world, rank, seed, 0
        self.total = (len(self.lab) + self.n_pool) // world * world   # same number of X-rays on every GPU

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return self.total // self.world

    def pool_part(self):
        out, start = [], self.epoch * self.n_pool
        while len(out) < self.n_pool:
            k, off = divmod(start + len(out), len(self.pool))
            perm = np.random.default_rng(self.seed + 7919 * (k + 1)).permutation(self.pool)
            out.extend(perm[off: off + self.n_pool - len(out)].tolist())
        return np.asarray(out, np.int64)

    def __iter__(self):
        idx = np.concatenate([self.lab, self.pool_part()]) if self.n_pool else self.lab.copy()
        np.random.default_rng(self.seed + 1000 * (self.epoch + 1)).shuffle(idx)   # identical on every rank
        return iter(idx[:self.total][self.rank::self.world].tolist())


# ----------------------------------------------------------------------------- model
class RelAttention(nn.Module):
    def __init__(self, dim, heads, drop=0.1):
        super().__init__()
        self.h, self.dh = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.rel_bias = nn.Embedding(N_REL, heads)
        nn.init.zeros_(self.rel_bias.weight)
        self.drop = nn.Dropout(drop)

    def forward(self, x, rel, mask):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-1, -2)) / math.sqrt(self.dh)
        att = att + self.rel_bias(rel).permute(0, 3, 1, 2).to(att.dtype)
        att = att.masked_fill(~mask[:, None, None, :], torch.finfo(att.dtype).min)
        att = self.drop(att.softmax(-1))
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, dim, heads, drop=0.1):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = RelAttention(dim, heads, drop)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * dim, dim))
        for lin in (self.attn.proj, self.mlp[-1]):   # block starts as identity
            nn.init.zeros_(lin.weight)
            nn.init.zeros_(lin.bias)

    def forward(self, x, rel, mask):
        x = x + self.attn(self.n1(x), rel, mask)
        return x + self.mlp(self.n2(x))


def make_head(in_d, n_cls, drop, zero_last=False):
    head = nn.Sequential(nn.LayerNorm(in_d), nn.Dropout(drop), nn.Linear(in_d, 256), nn.GELU(),
                         nn.Dropout(drop), nn.Linear(256, n_cls))
    if zero_last:
        nn.init.zeros_(head[-1].weight)
        nn.init.zeros_(head[-1].bias)
    return head


class TwinArch(nn.Module):
    def __init__(self, backbone, variant, pretrained=True, dim=256, layers=2, heads=4, n_cls=4, drop=0.2,
                 drop_path=0.0, box_src='token'):
        super().__init__()
        import timm
        self.variant = variant
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=1,
                                     drop_path_rate=drop_path)
        self.pad_to = 16   # encoder always sees 16, 32, 48 or 64 crops, so GPU kernels are set up once
        self.proj = nn.Sequential(nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.fdi = nn.Embedding(32, dim)
        nn.init.normal_(self.fdi.weight, std=0.02)
        use_diff = variant in ('twin', 'full')
        if use_diff:
            self.missing = nn.Parameter(torch.zeros(dim))
        if variant == 'full':
            self.diff_in = nn.Linear(dim + 1, dim)
            nn.init.zeros_(self.diff_in.weight)
            nn.init.zeros_(self.diff_in.bias)
        if variant in ('arch', 'full'):
            self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(layers)])
            self.norm = nn.LayerNorm(dim)
        self.head = make_head(dim, n_cls, drop)
        self.n_cls = n_cls
        # learned boxes: edge offsets per disease, read off the tooth features WITHOUT changing them (detached),
        # so the diagnosis is trained exactly as without boxes. 'token' reads the pooled tooth token, 'map' the
        # encoder's spatial feature map (where in the crop things are), 'both' trains the two side by side.
        self.box_src = box_src
        self.box_head = self.box_map = None
        if box_src in ('token', 'both'):
            self.box_head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 256), nn.GELU(), nn.Linear(256, n_cls * 4))
            nn.init.zeros_(self.box_head[-1].weight)
            nn.init.zeros_(self.box_head[-1].bias)
        if box_src in ('map', 'both'):
            C = self.enc.num_features
            self.box_reduce = nn.Sequential(nn.Conv2d(C, 64, 1), nn.GELU(), nn.AdaptiveAvgPool2d((6, 4)), nn.Flatten())
            m_in = 64 * 6 * 4 + dim
            self.box_map = nn.Sequential(nn.LayerNorm(m_in), nn.Linear(m_in, 256), nn.GELU(), nn.Linear(256, n_cls * 4))
            nn.init.zeros_(self.box_map[-1].weight)
            nn.init.zeros_(self.box_map[-1].bias)
        if use_diff:   # asymmetry evidence is added on top of the crop evidence, starting at zero
            self.head_asym = make_head(dim + 1, n_cls, drop, zero_last=True)

    def forward(self, x, fdi, twin, rel, mask):
        B, T = mask.shape
        xs = x[mask]                                           # each tooth encoded once
        n = xs.shape[0]
        pad = (-n) % self.pad_to if self.pad_to > 1 else 0
        if pad:   # blank crops; ConvNeXt has no batch statistics, so they do not touch real teeth
            xs = torch.cat([xs, xs.new_zeros((pad,) + tuple(xs.shape[1:]))], 0)
        fm = self.enc.forward_features(xs)                      # [n(+pad), C, h, w]
        f = self.proj(self.enc.forward_head(fm, pre_logits=True)[:n])
        P = f.new_zeros(B, T, f.shape[-1])
        P[mask] = f
        tok = P + self.fdi(fdi).to(P.dtype)
        if self.variant in ('twin', 'full'):
            idx = twin.clamp(min=0).unsqueeze(-1).expand(-1, -1, P.shape[-1])
            has = (twin >= 0).to(P.dtype).unsqueeze(-1)
            diff = has * (P - torch.gather(P, 1, idx)).abs() + (1 - has) * self.missing.to(P.dtype)
            asym = torch.cat([diff, has], -1)
        if self.variant == 'full':
            tok = tok + self.diff_in(asym)
        if self.variant in ('arch', 'full'):
            for blk in self.blocks:
                tok = blk(tok, rel, mask)
            tok = self.norm(tok)
        out = self.head(tok)
        if self.variant in ('twin', 'full'):
            out = out + self.head_asym(asym)
        # detach: in v4 the box loss trained the encoder too and pulled the diagnosis down (AP50 64 -> 56)
        boxes = {}
        if self.box_head is not None:
            boxes['token'] = self.box_head(tok.detach()).view(B, T, self.n_cls, 4)
        if self.box_map is not None:
            sp = self.box_reduce(fm[:n].detach())
            S = sp.new_zeros(B, T, sp.shape[-1])
            S[mask] = sp
            boxes['map'] = self.box_map(torch.cat([S, tok.detach()], -1)).view(B, T, self.n_cls, 4)
        return out, boxes                                       # [B, T, 4], {source: [B, T, 4, 4]}


# ----------------------------------------------------------------------------- helpers
class WeightEMA:
    """running average of the weights; validation, checkpoint and predictions use the averaged copy"""
    def __init__(self, model, decay):
        import copy
        self.decay, self.n = decay, 0
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))   # short warm-up so early weights are not frozen in
        src = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1 - d)
            else:
                v.copy_(src[k])


def amp_ctx(enabled):
    if enabled:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    import contextlib
    return contextlib.nullcontext()


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


def macro_ap(y, p):
    from sklearn.metrics import average_precision_score
    aps = [average_precision_score(y[:, k], p[:, k]) for k in range(y.shape[1]) if y[:, k].sum() > 0]
    return (float(np.mean(aps)) if aps else 0.0), aps


def to_dev(b, device):
    return {k: v.to(device, non_blocking=True) for k, v in b.items()}


@torch.no_grad()
def predict(core, df, args, device, use_amp, cache=None, tta=1):
    """probabilities per tooth; with tta > 1 they are averaged over that many fixed views (boxes: identity view)"""
    out = predict_view(core, df, args, device, use_amp, cache, None)
    if tta > 1 and len(out[0]):
        ps = [out[1]]
        for view in TTA_VIEWS[1:tta]:
            o = predict_view(core, df, args, device, use_amp, cache, view)
            assert np.array_equal(o[0], out[0])
            ps.append(o[1])
        out = (out[0], np.mean(ps, 0), out[2], out[3])
    return out


@torch.no_grad()
def predict_view(core, df, args, device, use_amp, cache=None, view=None):
    ds = ArchDS(df, args.crop_h, args.crop_w, False, args.max_teeth, cache=cache, view=view)
    dl = DataLoader(ds, batch_size=args.img_bs * 2, shuffle=False, num_workers=args.workers, collate_fn=collate)
    core.eval()
    ids, probs, ys, boxes = [], [], [], {}
    for b in dl:
        b = to_dev(b, device)
        with amp_ctx(use_amp):
            logit, bxs = core(b['x'], b['fdi'], b['twin'], b['rel'], b['mask'])
        p = torch.sigmoid(logit.float())
        keep = b['mask'] & (b['gt_only'] == 0)
        q = (b['fdi'][keep].cpu().numpy() // 8) + 1
        flip = np.isin(q, (2, 3))
        for src, box in bxs.items():
            bx = box.float()[keep].cpu().numpy()                     # canonical frame
            bx[flip] = flip_offsets(bx[flip])                        # back to the original image frame
            boxes.setdefault(src, []).append(bx)
        ids.append(b['ids'][keep].cpu().numpy())
        probs.append(p[keep].cpu().numpy())
        ys.append(b['y'][keep].cpu().numpy())
    if not ids:
        return np.zeros(0, np.int64), np.zeros((0, 4)), np.zeros((0, 4)), {}
    return (np.concatenate(ids), np.concatenate(probs), np.concatenate(ys),
            {k: np.concatenate(v) for k, v in boxes.items()})


def pred_frame(ids, probs, bxs):
    """one row per tooth: probabilities, and the learned box offsets (b* = token head, s* = spatial map head)"""
    cols = dict(crop_id=ids, p_caries=probs[:, 0], p_deep=probs[:, 1], p_peri=probs[:, 2], p_impacted=probs[:, 3])
    for src, pre in (('token', 'b'), ('map', 's')):
        if src in bxs:
            for d in range(4):
                for k, e in enumerate(('x1', 'y1', 'x2', 'y2')):
                    cols[f'{pre}{d}_{e}'] = bxs[src][:, d, k]
    return pd.DataFrame(cols)


def pred_keys_of(spec, args):
    name, kp = spec.split('=', 1)
    keys = sorted(read_keys(kp))
    return name, (keys[:args.max_images] if args.max_images else keys)


def run_predict_only(core, df, args, device, use_amp, rank, world, out):
    """score every --pred-keys set with the weights in --weights; each GPU takes every world-th X-ray"""
    core.load_state_dict(torch.load(args.weights, map_location=device))
    for spec in args.pred_keys:
        name, keys = pred_keys_of(spec, args)
        mine = set(keys[rank::world])
        dfp = df[df['image_key'].isin(mine) & (df['gt_only'] == 0)]
        t0 = time.time()
        ids, probs, _, bxs = predict(core, dfp, args, device, use_amp, None, tta=args.tta)
        part = out / (f'pred_{name}.part{rank}.csv' if world > 1 else f'pred_{name}.csv')
        pred_frame(ids, probs, bxs).to_csv(part, index=False)
        print(f'[train_arch] rank {rank}: {name} {len(ids)} teeth of {len(mine)} X-rays, {args.tta} view(s), '
              f'{time.time() - t0:.0f}s', flush=True)
    barrier(world)
    if rank == 0:
        for spec in args.pred_keys:
            name = spec.split('=', 1)[0]
            if world > 1:
                parts = [out / f'pred_{name}.part{r}.csv' for r in range(world)]
                pd.concat([pd.read_csv(q) for q in parts], ignore_index=True).to_csv(out / f'pred_{name}.csv', index=False)
                for q in parts:
                    q.unlink()
            print(f'[train_arch] wrote {out / f"pred_{name}.csv"}', flush=True)
        (out / 'log.json').write_text(json.dumps(dict(args=vars(args), predict_only=True), indent=1))


@torch.no_grad()
def profile(core, args, device, use_amp, n_teeth=28):
    """params, GFLOPs and latency for the classifier part on one X-ray with n_teeth teeth"""
    core.eval()
    pad_was, core.pad_to = core.pad_to, 1
    T = n_teeth
    x = torch.randn(1, T, 1, args.crop_h, args.crop_w, device=device)
    fdi = torch.arange(T, device=device).clamp(max=31).unsqueeze(0)
    twin = torch.full((1, T), -1, dtype=torch.long, device=device)
    rel = torch.full((1, T, T), REL_OTHER, dtype=torch.long, device=device)
    mask = torch.ones(1, T, dtype=torch.bool, device=device)
    out = dict(params_M=sum(p.numel() for p in core.parameters()) / 1e6, teeth=T)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc:
            core(x, fdi, twin, rel, mask)
        out['GFLOPs_per_xray'] = fc.get_total_flops() / 1e9
    except Exception as e:
        out['GFLOPs_per_xray'] = None
        out['flop_error'] = str(e)[:200]
    if device.type == 'cuda':
        for _ in range(3):
            with amp_ctx(use_amp):
                core(x, fdi, twin, rel, mask)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20):
            with amp_ctx(use_amp):
                core(x, fdi, twin, rel, mask)
        torch.cuda.synchronize()
        out['ms_per_xray'] = (time.time() - t0) / 20 * 1000
    core.pad_to = pad_was
    return out


# ----------------------------------------------------------------------------- main
def main(argv=None):
    args = get_args(argv)
    rank, world, local = setup_ddp()
    cuda = torch.cuda.is_available()
    device = torch.device(f'cuda:{local}' if cuda else 'cpu')
    use_amp = cuda
    torch.backends.cudnn.benchmark = False   # batch shapes change every step (teeth per X-ray)
    seed_all(args.seed + rank)
    out = Path(args.out)
    if rank == 0:
        out.mkdir(parents=True, exist_ok=True)
    barrier(world)

    df = pd.read_csv(args.manifest)
    if 'w' in df:      # per-tooth loss weight: 1 = annotated, < 1 = the teacher's soft labels (self-training)
        df['w'] = df['w'].fillna(1.0)
    if not os.path.exists(df['crop_path'].iloc[0]):   # crops unpacked somewhere else: <manifest dir>/<split>/<file>
        base = Path(args.manifest).parent
        df['crop_path'] = [str(base / Path(p).parent.name / Path(p).name) for p in df['crop_path']]
    missing = [p for p in df['crop_path'].iloc[::max(1, len(df) // 200)] if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f'{len(missing)} sampled crops are missing, e.g. {missing[0]}')
    kw = dict(pretrained=not (args.no_pretrained or args.predict_only), dim=args.dim, layers=args.layers,
              heads=args.heads, drop_path=args.drop_path, box_src=args.box_src)
    if args.predict_only:
        core = TwinArch(args.backbone, args.variant, **kw).to(device)
        run_predict_only(core, df, args, device, use_amp, rank, world, out)
        barrier(world)
        if world > 1:
            dist.destroy_process_group()
        return

    tr_keys, va_keys = read_keys(args.train_keys), read_keys(args.val_keys)
    if args.max_images:
        tr_keys = set(sorted(tr_keys)[:args.max_images])
        va_keys = set(sorted(va_keys)[:args.max_images])
    tr = df[df['image_key'].isin(tr_keys)]
    va = df[df['image_key'].isin(va_keys) & (df['gt_only'] == 0)]   # GT-only crops are for training only
    weighted = 'w' in tr and bool((tr['w'] < 1).any())
    lab = tr[tr['w'] >= 1] if weighted else tr       # class balance from the annotated teeth only
    pos = lab[LABELS].values.sum(0)
    pw = np.clip(np.sqrt((len(lab) - pos) / np.maximum(pos, 1)), 1.0, 10.0)
    if rank == 0:
        print(f'[train_arch] variant={args.variant} backbone={args.backbone} world={world} '
              f'train_xrays={lab.image_key.nunique()} teeth={len(lab)} val_xrays={va.image_key.nunique()} '
              f'pos={np.round(pos, 1).tolist()} pos_weight={np.round(pw, 2).tolist()}', flush=True)
        if weighted:
            pl = tr[tr['w'] < 1]
            print(f'[train_arch] + {pl.image_key.nunique()} pseudo-labelled X-rays ({len(pl)} teeth, weight '
                  f'{pl.w.mean():.2f}, mean soft label {np.round(pl[LABELS].values.mean(0), 3).tolist()}), '
                  f'{args.pool_ratio:g} per annotated X-ray each epoch', flush=True)
    pos_weight = torch.tensor(pw, dtype=torch.float32, device=device)

    if rank == 0:
        model = TwinArch(args.backbone, args.variant, **kw)
    barrier(world)
    if rank != 0:
        model = TwinArch(args.backbone, args.variant, **kw)
    model = model.to(device)
    core = model
    if world > 1:
        model = nn.parallel.DistributedDataParallel(model, device_ids=[local] if cuda else None)
        core = model.module

    cache = None
    if not args.no_cache:
        t_c = time.time()
        keys_all = set(tr_keys) | set(va_keys)
        for spec in args.pred_keys:
            keys_all |= read_keys(spec.split('=', 1)[1])
        cdf = df[df['image_key'].isin(keys_all)]
        if 'w' in cdf:      # pseudo-labelled X-rays are too many for RAM: they are read from disk
            cdf = cdf[cdf['w'] >= 1]
        cache = preload(cdf['crop_path'].unique(), args.crop_h, args.crop_w)
        if rank == 0:
            print(f'[train_arch] cached {len(cache)} crops in RAM in {time.time() - t_c:.0f}s', flush=True)
    boxes = None
    if args.box_targets and os.path.exists(args.box_targets):
        bt_df = pd.read_csv(args.box_targets)
        tcols = [f't{d}_{k}' for d in range(4) for k in ('x1', 'y1', 'x2', 'y2')]
        mcols = [f'm{d}' for d in range(4)]
        boxes = {int(c): (t.reshape(4, 4).astype(np.float32), mk.astype(np.float32))
                 for c, t, mk in zip(bt_df['crop_id'], bt_df[tcols].values, bt_df[mcols].values)}
        if rank == 0:
            print(f'[train_arch] box targets for {len(boxes)} teeth', flush=True)
    ds_tr = ArchDS(tr, args.crop_h, args.crop_w, True, args.max_teeth, seed=args.seed + 100 * rank, cache=cache,
                   boxes=boxes)
    if any(ds_tr.is_pool):
        sampler = MixSampler(ds_tr.is_pool, args.pool_ratio, world, rank, args.seed)
        if rank == 0:
            print(f'[train_arch] each epoch: {len(sampler.lab)} annotated + {sampler.n_pool} pseudo-labelled X-rays',
                  flush=True)
    else:
        sampler = DistributedSampler(ds_tr, num_replicas=world, rank=rank, shuffle=True, seed=args.seed,
                                     drop_last=True) if world > 1 else None
    dl_tr = DataLoader(ds_tr, batch_size=args.img_bs, sampler=sampler, shuffle=sampler is None,
                       num_workers=args.workers, pin_memory=cuda, drop_last=True, collate_fn=collate,
                       persistent_workers=args.workers > 0)
    enc_ids = {id(p) for p in core.enc.parameters()}
    # box heads are clipped on their own: a large box gradient must not shrink the diagnosis update
    box_ids = {id(p) for nm in ('box_head', 'box_reduce', 'box_map') if getattr(core, nm, None) is not None
               for p in getattr(core, nm).parameters()}
    main_params = [p for p in core.parameters() if id(p) not in box_ids]
    box_params = [p for p in core.parameters() if id(p) in box_ids]
    opt = torch.optim.AdamW([{'params': list(core.enc.parameters()), 'lr': args.lr},
                             {'params': [p for p in core.parameters() if id(p) not in enc_ids], 'lr': args.head_lr}],
                            weight_decay=args.wd)
    steps, warm = max(1, args.epochs * len(dl_tr)), max(1, len(dl_tr))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
    scaler = make_scaler(use_amp)
    ema = WeightEMA(core, args.ema) if (args.ema > 0 and rank == 0) else None   # ranks hold identical weights

    best, log = -1.0, []
    for ep in range(args.epochs):
        t0 = time.time()
        if sampler is not None:
            sampler.set_epoch(ep)
        model.train()
        tot, nb = 0.0, 0
        for b in dl_tr:
            b = to_dev(b, device)
            with amp_ctx(use_amp):
                logits, boxes = model(b['x'], b['fdi'], b['twin'], b['rel'], b['mask'])
            m = b['mask']
            x, y = logits.float()[m], b['y'][m]
            if weighted:
                # annotated teeth: class-balanced BCE as before. Pseudo-labelled teeth: plain BCE on the teacher's
                # soft label (it is already class-balanced) times their weight w
                wm = b['w'][m]
                is_lab = (wm >= 1).float()[:, None]
                pwm = is_lab * pos_weight[None, :] + (1 - is_lab)
                bce = pwm * y * F.softplus(-x) + (1 - y) * F.softplus(x)
                loss = (bce.mean(-1) * wm).sum() / wm.sum().clamp(min=1e-6)
            else:
                loss = F.binary_cross_entropy_with_logits(x, y, pos_weight=pos_weight)
            bmk = b['bm'] * m.unsqueeze(-1).float()                     # only real teeth that have that disease
            for box in boxes.values():                                  # detached heads: no effect on diagnosis
                l1 = F.smooth_l1_loss(box.float(), b['bt'], beta=0.02, reduction='none').sum(-1)
                loss = loss + args.box_w * (l1 * bmk).sum() / bmk.sum().clamp(min=1.0)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(main_params, 1.0)
            if box_params:
                torch.nn.utils.clip_grad_norm_(box_params, 1.0)
            scale_before = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            if scaler.get_scale() >= scale_before:   # a skipped step lowers the scale; keep the schedule in sync
                sched.step()
            if ema is not None:
                ema.update(core)
            tot += float(loss.detach())
            nb += 1
        if rank == 0:
            evm = ema.module if ema is not None else core
            ids, probs, ys, _ = predict(evm, va, args, device, use_amp, cache)
            mAP, per = macro_ap(ys, probs) if len(ids) else (0.0, [])
            if mAP > best:
                best = mAP
                torch.save(evm.state_dict(), out / 'best.pt')
            rec = dict(epoch=ep, loss=round(tot / max(nb, 1), 4), val_macro_ap=round(mAP, 4),
                       per_class=[round(v, 4) for v in per], sec=round(time.time() - t0, 1))
            log.append(rec)
            print(json.dumps(rec), flush=True)
        barrier(world)

    if rank == 0:
        if not (out / 'best.pt').exists():
            torch.save(core.state_dict(), out / 'best.pt')
        core.load_state_dict(torch.load(out / 'best.pt', map_location=device))
        prof = profile(core, args, device, use_amp)
        print('[train_arch] efficiency', json.dumps(prof), flush=True)
        for spec in args.pred_keys:
            name, keys = pred_keys_of(spec, args)
            dfp = df[df['image_key'].isin(set(keys)) & (df['gt_only'] == 0)]
            ids, probs, _, bxs = predict(core, dfp, args, device, use_amp, cache, tta=args.tta)
            pred_frame(ids, probs, bxs).to_csv(out / f'pred_{name}.csv', index=False)
            print(f'[train_arch] wrote {out / f"pred_{name}.csv"} rows={len(ids)}', flush=True)
        (out / 'log.json').write_text(json.dumps(dict(args=vars(args), best_val_macro_ap=best, efficiency=prof,
                                                      log=log), indent=1))
    barrier(world)
    if world > 1:
        dist.destroy_process_group()


if __name__ == '__main__':
    main()

In [ ]:
import importlib
sys.path.insert(0, os.getcwd())
import dentex_utils as U
importlib.reload(U)
print('helpers loaded:', U.DIS)


def show(obj):
    """print tables as plain text so they also land in the Kaggle run log (display() output does not)"""
    import pandas as pd
    with pd.option_context('display.max_columns', None, 'display.width', 250, 'display.max_colwidth', 80):
        if isinstance(obj, pd.DataFrame):      # keep the index when it carries names (groupby results)
            print(obj.to_string(index=not isinstance(obj.index, pd.RangeIndex)), flush=True)
        elif isinstance(obj, pd.Series):
            print(obj.to_string(), flush=True)
        else:
            print(obj, flush=True)

## 4. Data
Uses the attached `dentex_needed.zip` from the prep notebook (falls back to downloading from Hugging Face if nothing is attached). Heavy files live in `/kaggle/tmp`; only compact files are saved because Kaggle keeps ~500 output files per version. If the parser complains, send Claude the folder listing printed below.

In [ ]:
# Reuse whatever an earlier version of this notebook already finished (attach it with Add Input -> Your Work).
def find_inputs(name):
    """every attached file/folder called `name`, at any depth (Kaggle nests datasets and notebook outputs differently)"""
    hits = set(glob.glob(f'{INPUT_ROOT}/{name}')) | set(glob.glob(f'{INPUT_ROOT}/**/{name}', recursive=True))
    return sorted(hits, key=lambda p: (p.count('/'), p))

prev = [p for p in find_inputs(WORK_NAME) if Path(p).is_dir()]
if FINAL and RUN_PROTOCOL_B and not RUN_PROTOCOL_A and not prev and not (WORK / 'records.pkl').exists():
    raise RuntimeError('Protocol B only: attach the output of the earlier paper run (v10 for the v11 trial; Add Input '
                       '-> Your Work), otherwise the detectors, crops and classifiers would be made again.')
if prev and not (WORK / 'records.pkl').exists():
    def _skip_bulky(d, names):   # Kaggle unzips crops.zip when a version is re-uploaded as a dataset
        if Path(d).name == 'crops':
            return list(names)
        return [n for n in names if Path(n).suffix.lower() in ('.png', '.jpg', '.jpeg')]
    # several earlier outputs attached: merge them all, the most complete one last so its files win
    def nfiles(p):
        n = sum(1 for f in Path(p).rglob('*') if f.is_file() and 'crops' not in f.relative_to(p).parts[:-1]
                and f.suffix.lower() not in ('.png', '.jpg', '.jpeg'))
        if (Path(p) / 'state.zip').is_file():     # v15 outputs keep most files inside state.zip
            import zipfile
            with zipfile.ZipFile(Path(p) / 'state.zip') as zf:
                n += len(zf.namelist())
        return n
    for p in sorted(prev, key=nfiles):
        print('Resuming from', p)
        shutil.copytree(p, WORK, dirs_exist_ok=True, ignore=_skip_bulky)
    TIMES = json.loads(TIMES_P.read_text()) if TIMES_P.exists() else {}
else:
    print('Fresh run' if not prev else 'WORK already populated, nothing to copy')

# v15: a session ends by packing everything the next one needs into state.zip (Kaggle keeps only ~500 output files,
# cell 16). Unpack it here; when the output was re-uploaded as a dataset, Kaggle may already have unzipped it into
# a folder called state.
def unpack_state():
    z = WORK / 'state.zip'
    if z.is_file():
        shutil.unpack_archive(str(z), str(WORK), format='zip')
        z.unlink()
        print('unpacked the saved state (state.zip)')
    for d in (WORK / 'state', WORK / 'state.zip'):
        if d.is_dir():
            for f in [q for q in d.rglob('*') if q.is_file()]:
                t = WORK / f.relative_to(d)
                t.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(f), str(t))
            shutil.rmtree(d, ignore_errors=True)
            print(f'moved the unzipped saved state ({d.name}/) into place')
unpack_state()
if (WORK / 'timing.json').exists():
    TIMES = json.loads(TIMES_P.read_text())

MANIFEST = SCR / 'crops' / 'manifest.csv'
if (WORK / 'crops.zip').exists() and not MANIFEST.exists():
    with stage('unpack_crops'):
        shutil.unpack_archive(str(WORK / 'crops.zip'), str(SCR))
print('files saved in WORK so far:', sum(1 for p in WORK.rglob('*') if p.is_file()),
      '(the session packs them into one state.zip at the end: Kaggle keeps only ~500 output files)')

In [ ]:
REPO = 'ibrahimhamamci/DENTEX'
FILES = ['DENTEX/training_data.zip', 'DENTEX/validation_data.zip', 'DENTEX/test_data.zip',
         'DENTEX/validation_triple.json']
# raw X-rays are only needed to build crops or to train a YOLO detector
need_raw = (not MANIFEST.exists() or
            (RUN_PROTOCOL_A and RUN_YOLO_BASELINE_A) or     # the detectors predict on the X-rays (val settings)
            (RUN_PROTOCOL_B and RUN_YOLO_BASELINE_B) or
            SELF_TRAIN)                                     # the student YOLO trains on the annotated X-rays too
# the 1571 unlabelled X-rays and the quadrant-only set are only used for self-training
POOL_SKIP = () if SELF_TRAIN else ('unlabel', '/quadrant/')
attached = [p for p in find_inputs('dentex_needed.zip') if os.path.isfile(p)]
official_zips = [p for n in ('training_data.zip', 'validation_data.zip', 'test_data.zip')
                 for p in [q for q in find_inputs(n) if os.path.isfile(q)][:1]]
official_json = [p for p in find_inputs('validation_triple.json') if os.path.isfile(p)]
unzipped = sorted({str(Path(INPUT_ROOT) / Path(p).relative_to(INPUT_ROOT).parts[0])
                   for pat in ('*disease*.json', '*triple*.json')
                   for p in glob.glob(f'{INPUT_ROOT}/**/{pat}', recursive=True)
                   if WORK_NAME not in p})
RAW_X = RAW / 'x'
if need_raw and unzipped and not attached and not official_zips:
    # you uploaded DENTEX yourself and Kaggle unzipped it: read it in place, nothing to copy
    RAW_X = Path(unzipped[0])
    print('using the DENTEX folders you uploaded:', RAW_X)
elif need_raw and not (RAW / '.done').exists():
    (RAW / 'x').mkdir(parents=True, exist_ok=True)
    if official_zips:
        with stage('extract_uploaded_dentex'):
            print('using the official DENTEX zips you uploaded:', official_zips)
            for z in official_zips:
                print('  extracted', U.safe_extract(z, RAW / 'x', skip=POOL_SKIP), 'files from', Path(z).name)
            for j in official_json[:1]:
                shutil.copy(j, RAW / 'x' / Path(j).name)
            (RAW / '.done').touch()
    elif attached:
        with stage('unpack_attached_dentex'):
            print('using the attached DENTEX copy:', attached[0])
            shutil.unpack_archive(attached[0], str(RAW / 'x'))
            (RAW / '.done').touch()
    else:
        with stage('download+extract'):
            print('No attached copy found (run the prep notebook once to skip this), downloading from Hugging Face')
            from huggingface_hub import hf_hub_download
            print(f'free disk at {RAW}: {shutil.disk_usage(RAW).free/1e9:.0f} GB (need ~16 GB peak)')
            token = None
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:
                pass
            for f in FILES:
                p = hf_hub_download(REPO, f, repo_type='dataset', local_dir=str(RAW / 'dl'), token=token)
                print(f'downloaded {f}: {os.path.getsize(p)/1e9:.2f} GB')
                if p.endswith('.zip'):
                    n = U.safe_extract(p, RAW / 'x', skip=POOL_SKIP)
                    print(f'  extracted {n} files')
                    os.remove(p)
                else:
                    shutil.copy(p, RAW / 'x' / Path(p).name)
            (RAW / '.done').touch()
elif not need_raw:
    print('crops and detectors already done, raw X-rays not needed in this session')
else:
    print('raw data already here')
if Path(RAW_X).exists():
    for p in sorted({str(q.parent.relative_to(RAW_X)) for q in Path(RAW_X).rglob('*') if q.is_file()})[:40]:
        print('  ', p)

In [ ]:
REC_P = WORK / 'records.pkl'
if REC_P.exists():
    SETS = pickle.loads(REC_P.read_bytes())
else:
    SETS = U.discover(RAW_X)
    REC_P.write_bytes(pickle.dumps(SETS))
for k in ['enum', 'dis_train', 'dis_val', 'dis_test']:
    if k not in SETS:
        print(f'!! {k} not found')
# The 250 released test labels (LabelMe, Turkish) use a different, treatment-style label set: cürük, gömülü,
# lezyon, kanal (root canal), küretaj (curettage), cekim (extraction), kirik (fracture), saglam (healthy).
# There is no 'deep caries' and the organisers never published a mapping (GitHub issue #8 is unanswered),
# so these labels cannot reproduce the 4-class leaderboard. We keep them out of every reported number.
USE_LABELME_TEST = False
if 'dis_test' in SETS and 'labelme' in SETS['dis_test']['cats'].get('c3', {}) and not USE_LABELME_TEST:
    print('Ignoring the released LabelMe test labels (different label set, no official mapping).')
    SETS.pop('dis_test')
HAS_TEST = 'dis_test' in SETS
for k, v in SETS.items():
    n_ann = sum(len(i['anns']) for i in v['images'])
    print(f'{k:10s} images={len(v["images"]):4d} boxes={n_ann:6d}  cats3={v["cats"]["c3"]}')
for k in ['dis_train', 'dis_val', 'dis_test']:
    if k in SETS:
        cnt = np.bincount([a['d'] for r in SETS[k]['images'] for a in r['anns'] if 'd' in a], minlength=4)
        qc = np.bincount([a.get('q', 0) for r in SETS[k]['images'] for a in r['anns'] if 'd' in a], minlength=5)[1:]
        print(k, dict(zip(U.DIS, cnt.tolist())), '| same boxes per quadrant 1-4:', qc.tolist())
if not HAS_TEST:
    print('Protocol A = train on the 705, pick settings on a held-out 10% of them, '
          'report on the official 50-image validation set (proper 4-class labels).')

## Audit (v15): copies inside DENTEX, and which X-rays the tooth detector had seen


In [ ]:
# v15 legitimacy audit (CPU only, ~3 min once, then cached in WORK). DENTEX's subsets were cut from one archive of
# X-rays, so the same X-ray can sit in two of them. Three questions, answered from the pixels:
#  1. Is a diagnosis X-ray (705 training, 50 official validation) a copy of another one? A copy across a split boundary
#     would let a test X-ray's labels into training, so such validation / test X-rays are not scored (drop_copies).
#  2. Which of the enumeration X-rays (the tooth detector's training data) are copies of diagnosis X-rays? The tooth
#     detector of v5-v14 learned from all of them, so it had seen the teeth (not the diagnoses) of many validation and
#     test X-rays. From v15 those are cropped by a second tooth detector that learns only from the rest (ENUM_CLEAN).
#  3. Are two different diagnosis X-rays perhaps of the same patient? DENTEX has no patient ids, so this is a check
#     only: ORB keypoints + RANSAC on the most similar pairs, and a picture of them (audit_v15_pairs.jpg in the output).
AUDIT_P = WORK / 'audit_v15.pkl'
_NP = max(1, min(4, os.cpu_count() or 1))
ANN_RECS = [r for s in ('dis_train', 'dis_val', 'dis_test') if s in SETS for r in SETS[s]['images']]
ANN_KEYS = [r['key'] for r in ANN_RECS]
REC_BY_KEY = {r['key']: r for r in ANN_RECS}
ENUM_RECS = SETS['enum']['images'] if 'enum' in SETS else []
DUP_HARD, DUP_SOFT = CFG['DUP_HARD'], CFG['DUP_SOFT']
if CFG.get('CLEAN_EVAL') and not ENUM_RECS:
    print('!! no enumeration X-rays found: the clean tooth detector cannot be trained, CLEAN_EVAL is switched off')
    CFG['CLEAN_EVAL'] = False

AUDIT = pickle.loads(AUDIT_P.read_bytes()) if AUDIT_P.exists() else None
if AUDIT is not None and (AUDIT['ann_keys'] != ANN_KEYS or AUDIT['enum_keys'] != [r['key'] for r in ENUM_RECS]):
    AUDIT = None     # made for other X-rays
if AUDIT is None:
    with stage('audit_v15'):
        th_a = np.stack(U.pmap(U.thumb_file, [r['path'] for r in ANN_RECS], workers=_NP)).astype(np.float32)
        th_e = (np.stack(U.pmap(U.thumb_file, [r['path'] for r in ENUM_RECS], workers=_NP)).astype(np.float32)
                if ENUM_RECS else np.zeros((0,) + th_a.shape[1:], np.float32))

        def _dist(A, B):   # sqrt(2 - 2 * correlation) of the standardised thumbnails, as U.rms_nearest
            A, B = U._standardize(A), U._standardize(B)
            return np.sqrt(np.maximum(2 - 2 * (A @ B.T) / A.shape[1], 0)).astype(np.float32)
        d_aa = _dist(th_a, th_a)
        np.fill_diagonal(d_aa, np.inf)
        d_ea = _dist(th_e, th_a) if len(th_e) else np.zeros((0, len(th_a)), np.float32)
        # the most similar diagnosis X-ray of every diagnosis X-ray, and a few enumeration copies as a known positive
        nn = d_aa.argmin(1)
        cand = sorted({(int(min(i, j)), int(max(i, j))) for i, j in enumerate(nn)}, key=lambda p: d_aa[p])
        cand = cand[:CFG.get('AUDIT_MAX_PAIRS', 800)]
        e_pairs = list(zip(range(len(d_ea)), d_ea.min(1), d_ea.argmin(1))) if len(d_ea) else []
        ctrl = [(int(i), int(j)) for i, m, j in e_pairs if m < DUP_HARD][:15]
        # enumeration X-rays that look somewhat like a diagnosis X-ray: a copy cropped or processed differently would
        # hide here, so ORB checks them too (they are kept away from the clean tooth detector if they match)
        e_near = [(int(i), int(j)) for i, m, j in e_pairs if DUP_SOFT <= m < 0.45]
        tasks = ([(ANN_RECS[i]['path'], ANN_RECS[j]['path']) for i, j in cand]
                 + [(ENUM_RECS[i]['path'], ANN_RECS[j]['path']) for i, j in ctrl + e_near])
        inl = U.pmap(U.orb_inliers, tasks, workers=_NP, chunksize=8) if CFG.get('AUDIT_ORB', True) else [-1] * len(tasks)
        n1, n2 = len(cand), len(cand) + len(ctrl)
        AUDIT = dict(ann_keys=ANN_KEYS, enum_keys=[r['key'] for r in ENUM_RECS], th_ann=th_a, d_aa=d_aa,
                     e_min=d_ea.min(1) if len(d_ea) else np.zeros(0), e_arg=d_ea.argmin(1) if len(d_ea) else np.zeros(0, int),
                     orb_pairs=[(i, j, float(d_aa[i, j]), int(n)) for (i, j), n in zip(cand, inl[:n1])],
                     orb_ctrl=[(i, j, int(n)) for (i, j), n in zip(ctrl, inl[n1:n2])],
                     orb_enum=[(i, j, int(n)) for (i, j), n in zip(e_near, inl[n2:])])
        AUDIT_P.write_bytes(pickle.dumps(AUDIT))

# 1. copies among the diagnosis X-rays -> groups (DUP_OF[key] = the other copies of the same X-ray)
_d = AUDIT['d_aa']
_ii, _jj = np.where(np.triu(_d < DUP_SOFT, 1))
AUDIT_CLOSE = sorted([(ANN_KEYS[i], ANN_KEYS[j], float(_d[i, j])) for i, j in zip(_ii, _jj)], key=lambda t: t[2])
_parent = {k: k for k in ANN_KEYS}
def _root(k):
    while _parent[k] != k:
        _parent[k] = _parent[_parent[k]]
        k = _parent[k]
    return k
for a, b, dd in AUDIT_CLOSE:
    if dd < DUP_HARD:
        _parent[_root(a)] = _root(b)
_groups = {}
for k in ANN_KEYS:
    _groups.setdefault(_root(k), set()).add(k)
DUP_OF = {k: g - {k} for g in _groups.values() if len(g) > 1 for k in g}

def drop_copies(train_keys, val_keys, test_keys):
    """validation X-rays with a copy among the training X-rays, and test X-rays with a copy among the training or
    validation X-rays, are left out of the scores (the models were trained on that copy and its labels)"""
    if not DUP_OF:
        return list(val_keys), list(test_keys), []
    trs = set(train_keys)
    seen = trs | set(val_keys)
    va = [k for k in val_keys if not (DUP_OF.get(k, set()) & trs)]
    te = [k for k in test_keys if not (DUP_OF.get(k, set()) & seen)]
    return va, te, sorted((set(val_keys) - set(va)) | (set(test_keys) - set(te)))

# ORB scale: the same X-ray (enumeration copies) gives a median of _ctrl inliers; a quarter of that (at least 40) flags
# a pair as possibly the same X-ray or the same patient
_oc = [t[2] for t in AUDIT['orb_ctrl'] if t[2] >= 0]
_ctrl = float(np.median(_oc)) if _oc else float('nan')
_flag = max(40.0, 0.25 * _ctrl) if _oc else 40.0
# 2. enumeration X-rays that are (near) copies of diagnosis X-rays
_em, _ea = np.asarray(AUDIT['e_min']), np.asarray(AUDIT['e_arg']).astype(int)
_orb_e = {i: n for i, j, n in AUDIT.get('orb_enum', []) if n >= _flag}
ENUM_CLEAN = [r for i, (r, m) in enumerate(zip(ENUM_RECS, _em)) if m >= DUP_SOFT and i not in _orb_e]   # the clean
                                                                    # tooth detector learns from these only
ENUM_COPY = {}       # diagnosis key -> its copy among the enumeration X-rays (whose tooth labels show what was seen)
for r, m, j in sorted(zip(ENUM_RECS, _em, _ea), key=lambda t: -t[1]):
    if m < DUP_HARD:
        ENUM_COPY[ANN_KEYS[j]] = r
_set = lambda k: k.split('/')[0]
_cnt = lambda keys: {s: sum(_set(k) == s for k in keys) for s in sorted({_set(k) for k in ANN_KEYS})}
print(f'\nAUDIT v15: {len(ANN_KEYS)} diagnosis X-rays {_cnt(ANN_KEYS)}, {len(ENUM_RECS)} enumeration X-rays')
_copies = [t for t in AUDIT_CLOSE if t[2] < DUP_HARD]
_near = [t for t in AUDIT_CLOSE if t[2] >= DUP_HARD]
print(f'1. copies among the diagnosis X-rays (thumbnail distance < {DUP_HARD}): {len(_copies)} pairs'
      + ''.join(f'\n     {a} = {b} ({dd:.3f})' for a, b, dd in _copies[:15]))
print(f'   near copies ({DUP_HARD} to {DUP_SOFT}): {len(_near)} pairs'
      + ''.join(f'\n     {a} ~ {b} ({dd:.3f})' for a, b, dd in _near[:15]))
_q = np.percentile(_d.min(1), [0, 1, 5, 50])
print('   distance of each diagnosis X-ray to its most similar other one: min %.3f, 1%% %.3f, 5%% %.3f, median %.3f'
      % tuple(_q))
print(f'2. enumeration X-rays that are copies of diagnosis X-rays: {int((_em < DUP_HARD).sum())} of {len(ENUM_RECS)} '
      f'(near copies {int(((_em >= DUP_HARD) & (_em < DUP_SOFT)).sum())}, matched by ORB {len(_orb_e)}). Diagnosis '
      f'X-rays whose teeth the v5-v14 tooth detector learned from: {_cnt(ENUM_COPY)}. The clean tooth detector learns '
      f'from the other {len(ENUM_CLEAN)}.')
_op = [t for t in AUDIT['orb_pairs'] if t[3] >= 0]
if _op:
    _n = np.array([t[3] for t in _op])
    _hits = sorted([t for t in _op if t[3] >= _flag and t[2] >= DUP_HARD], key=lambda t: -t[3])
    print(f'3. same-patient check (ORB + RANSAC inliers, {len(_op)} most similar pairs): median {np.median(_n):.0f}, '
          f'95% {np.percentile(_n, 95):.0f}, max {_n.max()}; the same X-ray (enumeration copies) gives a median of '
          f'{_ctrl:.0f}. Pairs with >= {_flag:.0f} inliers that are not copies (possibly the same patient): {len(_hits)}'
          + ''.join(f'\n     {ANN_KEYS[i]} ~ {ANN_KEYS[j]} (distance {dd:.3f}, {n} inliers)' for i, j, dd, n in _hits[:15]))
    AUDIT_SAME_PATIENT = [(ANN_KEYS[i], ANN_KEYS[j], dd, n) for i, j, dd, n in _hits]
else:
    AUDIT_SAME_PATIENT = []
    print('3. same-patient check skipped')
pd.DataFrame([dict(kind='copy' if dd < DUP_HARD else 'near copy', key_a=a, key_b=b, distance=round(dd, 4))
              for a, b, dd in AUDIT_CLOSE]
             + [dict(kind='most similar pair, ORB', key_a=ANN_KEYS[i], key_b=ANN_KEYS[j], distance=round(dd, 4),
                     orb_inliers=n) for i, j, dd, n in AUDIT['orb_pairs']]
             ).to_csv(WORK / 'audit_v15_pairs.csv', index=False)
_mont = WORK / 'audit_v15_pairs.jpg'
if not _mont.exists():
    _show = [(REC_BY_KEY[a]['path'], REC_BY_KEY[b]['path'], f'copy: {a} | {b} | distance {dd:.3f}') for a, b, dd in _copies[:4]]
    _show += [(REC_BY_KEY[a]['path'], REC_BY_KEY[b]['path'], f'near copy: {a} | {b} | distance {dd:.3f}')
              for a, b, dd in _near[:6]]
    _show += [(ANN_RECS[i]['path'], ANN_RECS[j]['path'], f'most ORB matches: {ANN_KEYS[i]} | {ANN_KEYS[j]} | distance '
               f'{dd:.3f} | ORB {n}') for i, j, dd, n in sorted([t for t in _op if t[2] >= DUP_HARD], key=lambda t: -t[3])[:8]]
    _show += [(ENUM_RECS[i]['path'], ANN_RECS[j]['path'], f'positive control: enumeration copy of {ANN_KEYS[j]} | ORB {n}')
              for i, j, n in AUDIT['orb_ctrl'][:2]]
    U.pair_montage(_show, _mont)
print(f'   pairs: audit_v15_pairs.csv, pictures of the closest ones: {_mont.name} (both in the output)')


## 5. Detectors (both GPUs, DDP)
Tooth + FDI detector on the 634 enumeration X-rays (finds and numbers teeth); it crops the TRAINING X-rays. v15: a second tooth detector, same recipe, learns only from the enumeration X-rays that are not (near) copies of any diagnosis X-ray, and only it crops validation and test X-rays. The plain YOLO disease detector is trained only as the baseline row.


In [ ]:
TOOTH_ROOT = SCR / 'yolo_tooth'
TOOTH_W = WORK / 'weights' / 'tooth.pt'

def free_gpu():
    """hand back the GPU memory this notebook cached (YOLO inference) before a training subprocess starts"""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def finish_run(name, dest):
    """copy best.pt (+ results.csv) of an ultralytics run from scratch into WORK/weights"""
    run = SCR / 'runs' / name
    assert (run / 'weights' / 'best.pt').exists(), f'{name} did not finish, scroll up for the error'
    shutil.copy(run / 'weights' / 'best.pt', dest)
    if (run / 'results.csv').exists():
        shutil.copy(run / 'results.csv', WORK / 'weights' / f'{name}_results.csv')
    return dest

def train_tooth_detector(name, recs, dest):
    """YOLO tooth finder (32 FDI classes) on enumeration X-rays, 10% of them held out to pick the checkpoint"""
    tmeta = U.export_yolo([dict(r) for r in recs], TOOTH_ROOT, U.tooth_cls, CFG['LONG_SIDE'])
    keys = sorted(tmeta)
    random.Random(0).shuffle(keys)
    nval = max(30, len(keys) // 10)
    tr_txt = U.write_list(TOOTH_ROOT, f'{name}_train', keys[nval:], tmeta)
    va_txt = U.write_list(TOOTH_ROOT, f'{name}_val', keys[:nval], tmeta)
    yaml_p = U.write_yolo_yaml(TOOTH_ROOT, U.TOOTH_NAMES, tr_txt, va_txt, yaml_name=f'{name}.yaml')
    # fliplr=0: a horizontal flip would turn tooth 36 into a "46" without changing the label.
    # mosaic=0: tooth numbers depend on where a tooth sits in the whole arch.
    cmd = (f"yolo detect train model={CFG['TOOTH_MODEL']} data={yaml_p} imgsz={CFG['DET_IMGSZ']} "
           f"epochs={CFG['TOOTH_EPOCHS']} batch={8 if 'rtdetr' in CFG['TOOTH_MODEL'] else CFG['DET_BATCH']} device={DEVICE_STR} "
           f"workers={CFG['DET_WORKERS']} project={SCR/'runs'} name={name} exist_ok=True seed=0 "
           f"cos_lr=True patience=30 fliplr=0.0 mosaic=0.0 mixup=0.0 degrees=3 translate=0.05 "
           f"scale=0.2 hsv_h=0 hsv_s=0 hsv_v=0.3 plots=True")
    print(cmd)
    free_gpu()
    get_ipython().system(cmd)
    return finish_run(name, dest)

if not TOOTH_W.exists():
    with stage('tooth_detector'):
        train_tooth_detector('tooth', SETS['enum']['images'], TOOTH_W)
print('tooth detector:', TOOTH_W)
# v15: the same recipe on the enumeration X-rays that are not (near) copies of any diagnosis X-ray (audit, cell 08b).
# Only this one crops validation and test X-rays; the first one keeps cropping training X-rays (it never saw a
# diagnosis label, and training X-rays may well be in its training data).
TOOTH_CLEAN_W = WORK / 'weights' / 'tooth_clean.pt'
if CFG.get('CLEAN_EVAL') and not TOOTH_CLEAN_W.exists():
    with stage('tooth_detector_clean'):
        print(f'clean tooth detector: {len(ENUM_CLEAN)} of {len(ENUM_RECS)} enumeration X-rays')
        train_tooth_detector('tooth_clean', ENUM_CLEAN, TOOTH_CLEAN_W)
if CFG.get('CLEAN_EVAL'):
    print('clean tooth detector (validation and test X-rays):', TOOTH_CLEAN_W)


In [ ]:
DIS_ROOT = SCR / 'yolo_dis'
DIS_META_P = WORK / 'dis_meta.pkl'
DMETA = pickle.loads(DIS_META_P.read_bytes()) if DIS_META_P.exists() else None

def ensure_dis_images():
    """(re)build the resized disease-set jpgs in scratch; only called when a detector must train or predict"""
    global DMETA
    if DMETA is not None and all(Path(v[3]).exists() and str(v[3]).startswith(str(DIS_ROOT))
                                 for v in list(DMETA.values())[:25]):
        return DMETA
    with stage('export_disease_images'):
        recs = [r for k in ['dis_train', 'dis_val', 'dis_test'] if k in SETS for r in SETS[k]['images']]
        DMETA = U.export_yolo(recs, DIS_ROOT, U.disease_cls, CFG['LONG_SIDE'])
        DIS_META_P.write_bytes(pickle.dumps(DMETA))
        REC_P.write_bytes(pickle.dumps(SETS))     # now also stores image sizes/scales
    return DMETA

DIS_ROOT_HI = SCR / 'yolo_dis_hi'          # v14: the same X-rays exported with a longer side (DET_IMGSZ_XHI)
DIS_META_HI_P = WORK / 'dis_meta_hi.pkl'
DMETA_HI = pickle.loads(DIS_META_HI_P.read_bytes()) if DIS_META_HI_P.exists() else None

def ensure_dis_images_hi():
    """the annotated X-rays exported a second time with the long side at DET_IMGSZ_XHI (for the high-resolution
    student); the labels are the same (YOLO labels are relative to the image size)"""
    global DMETA_HI
    if DMETA_HI is not None and all(Path(v[3]).exists() and str(v[3]).startswith(str(DIS_ROOT_HI))
                                    for v in list(DMETA_HI.values())[:25]):
        return DMETA_HI
    with stage('export_disease_images_hi'):
        recs = [dict(r) for k in ['dis_train', 'dis_val', 'dis_test'] if k in SETS for r in SETS[k]['images']]
        DMETA_HI = U.export_yolo(recs, DIS_ROOT_HI, U.disease_cls, CFG['DET_IMGSZ_XHI'])
        DIS_META_HI_P.write_bytes(pickle.dumps(DMETA_HI))
    longs = np.array([max(v[0], v[1]) for v in DMETA_HI.values()])
    print(f'annotated X-rays, long side: min {longs.min()}, median {np.median(longs):.0f}, max {longs.max()} px; '
          f'{(longs > CFG["DET_IMGSZ_XHI"]).mean():.0%} are larger than {CFG["DET_IMGSZ_XHI"]} px (the 1280 jpgs keep '
          f'{np.median(np.minimum(1, 1280 / longs)):.0%} of the median width, these {np.median(np.minimum(1, CFG["DET_IMGSZ_XHI"] / longs)):.0%})')
    return DMETA_HI

KEYS = {k: [r['key'] for r in SETS[k]['images']] for k in ['dis_train', 'dis_val', 'dis_test'] if k in SETS}
if HAS_TEST:
    A_TRAIN, A_VAL, A_TEST = KEYS['dis_train'], KEYS['dis_val'], KEYS['dis_test']
else:   # fallback: hold out 10% of train for model selection, report on the official val set
    kk = sorted(KEYS['dis_train']); random.Random(123).shuffle(kk)
    A_VAL, A_TRAIN, A_TEST = kk[:70], kk[70:], KEYS['dis_val']
A_VAL, A_TEST, _dropped = drop_copies(A_TRAIN, A_VAL, A_TEST)    # v15 audit: X-rays whose copy the models trained on
if _dropped:
    print(f'protocol A: {len(_dropped)} validation / test X-rays have a copy among the training X-rays and are not '
          f'scored: {_dropped}')

def train_disease_yolo(name, train_keys, val_keys, epochs, seed=0, train_txt=None, extra='', imgsz=None, batch=None,
                       model=None, hires=False):
    """train_txt: a ready list of training images (the student's annotated + pseudo-labelled X-rays).
    imgsz / batch: another input size than DET_IMGSZ (the 1280 YOLO) and its batch.
    model: start from these weights instead of DIS_MODEL (another architecture, or a fine-tune of a trained YOLO).
    hires: use the higher-resolution export (ensure_dis_images_hi) for the training and validation lists."""
    w = WORK / 'weights' / f'{name}.pt'
    if w.exists():
        return w
    dm, root = (ensure_dis_images_hi(), DIS_ROOT_HI) if hires else (ensure_dis_images(), DIS_ROOT)
    tr_txt = str(train_txt) if train_txt else U.write_list(root, f'{name}_train', train_keys, dm)
    va_txt = U.write_list(root, f'{name}_val', val_keys, dm)
    yaml_p = U.write_yolo_yaml(root, U.DIS, tr_txt, va_txt, yaml_name=f'{name}.yaml')
    # disease labels do not depend on the side, so the usual flips/mosaic are fine here
    b = batch or CFG['DET_BATCH']
    for attempt in range(2):   # out of GPU memory at the start: once more with half the batch
        shutil.rmtree(SCR / 'runs' / name, ignore_errors=True)
        cmd = (f"yolo detect train model={model or CFG['DIS_MODEL']} data={yaml_p} imgsz={imgsz or CFG['DET_IMGSZ']} "
               f"epochs={epochs} batch={b} device={DEVICE_STR} workers={CFG['DET_WORKERS']} "
               f"project={SCR/'runs'} name={name} exist_ok=True seed={seed} cos_lr=True patience=30 plots=True {extra}")
        print(cmd)
        free_gpu()
        get_ipython().system(cmd)
        if (SCR / 'runs' / name / 'weights' / 'best.pt').exists() or attempt or b <= 4:
            break
        b //= 2
        print(f'!! {name} did not train, trying once more with batch {b}')
    return finish_run(name, w)

DIS_W_A = DIS_W_A2 = None
if RUN_PROTOCOL_A and RUN_YOLO_BASELINE_A:
    with stage('disease_detector_A'):
        DIS_W_A = train_disease_yolo('disA', A_TRAIN, A_VAL, CFG['DIS_EPOCHS'])
    print('disease detector A:', DIS_W_A)
    if CFG['ENSEMBLE_CONTROL']:   # same detector, another seed: is a fusion gain more than "two models beat one"?
        with stage('disease_detector_A_seed1'):
            DIS_W_A2 = train_disease_yolo('disA_s1', A_TRAIN, A_VAL, CFG['DIS_EPOCHS'], seed=1)

In [ ]:
from ultralytics import YOLO, RTDETR

def load_detector(weights, arch_name):
    return RTDETR(str(weights)) if 'rtdetr' in str(arch_name).lower() else YOLO(str(weights))

def yolo_boxes(weights, keys, conf, iou, agnostic, max_det, arch_name='yolo', meta=None, augment=False, imgsz=None):
    """Run a detector on the cached jpgs and return boxes in ORIGINAL pixel coordinates.
    meta: {key: (W, H, scale, jpg)} of other X-rays than the annotated ones (the self-training pool)."""
    dm = meta if meta is not None else ensure_dis_images()
    model = load_detector(weights, arch_name)
    keys = list(keys)
    out = {k: (np.zeros((0, 4)), np.zeros(0, int), np.zeros(0)) for k in keys}
    # newer ultralytics turns a list of paths into ONE batch (all ~750 X-rays at once -> CUDA OOM),
    # so feed small chunks; results come back in the same order as the paths
    CH = 16
    for i in range(0, len(keys), CH):
        chunk = keys[i:i + CH]
        res = model.predict(source=[dm[k][3] for k in chunk], imgsz=imgsz or CFG['DET_IMGSZ'], conf=conf, iou=iou,
                            agnostic_nms=agnostic, max_det=max_det, device=0, batch=CH, verbose=False,
                            augment=augment)
        for k, r in zip(chunk, res):
            s = dm[k][2]
            b = r.boxes
            out[k] = (b.xyxy.cpu().numpy() / s, b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy())
    return out

TEETH_P = WORK / 'teeth.pkl'
if TEETH_P.exists():
    TEETH = pickle.loads(TEETH_P.read_bytes())
else:
    with stage('tooth_inference'):
        allk = [k for v in KEYS.values() for k in v]
        raw = yolo_boxes(TOOTH_W, allk, conf=0.10, iou=0.5, agnostic=True, max_det=64, arch_name=CFG['TOOTH_MODEL'])
        TEETH = {k: U.teeth_from_yolo(*raw[k]) for k in allk}
        TEETH_P.write_bytes(pickle.dumps(TEETH))
nt = [len(v) for v in TEETH.values()]
print(f'teeth per X-ray: mean {np.mean(nt):.1f}, min {min(nt)}, max {max(nt)}')

# v15: the clean tooth detector on the same X-rays (it crops the validation and test X-rays, cell 12)
TEETH_CLEAN_P = WORK / 'teeth_clean.pkl'
TEETH_E = TEETH
if CFG.get('CLEAN_EVAL'):
    if TEETH_CLEAN_P.exists():
        TEETH_E = pickle.loads(TEETH_CLEAN_P.read_bytes())
    else:
        with stage('tooth_inference_clean'):
            allk = [k for v in KEYS.values() for k in v]
            raw = yolo_boxes(TOOTH_CLEAN_W, allk, conf=0.10, iou=0.5, agnostic=True, max_det=64, arch_name=CFG['TOOTH_MODEL'])
            TEETH_E = {k: U.teeth_from_yolo(*raw[k]) for k in allk}
            TEETH_CLEAN_P.write_bytes(pickle.dumps(TEETH_E))
    nt = [len(v) for v in TEETH_E.values()]
    print(f'clean tooth detector, teeth per X-ray: mean {np.mean(nt):.1f}, min {min(nt)}, max {max(nt)}')
    # what the first tooth detector got from having seen these X-rays: on the diagnosis X-rays whose copy is an
    # enumeration X-ray, that copy's tooth labels say where every tooth really is
    if ENUM_COPY:
        tq = []
        for label, T_ in (('v5-v14 tooth detector (learned from these X-rays)', TEETH),
                          ('v15 clean tooth detector (never saw them)', TEETH_E)):
            ious = []
            for k, e in ENUM_COPY.items():
                a = REC_BY_KEY[k]
                sx = a['w'] / e['w'] if a.get('w') and e.get('w') else 1.0
                sy = a['h'] / e['h'] if a.get('h') and e.get('h') else 1.0
                gt_t = [dict(q=t['q'], n=t['n'], box=[t['box'][0] * sx, t['box'][1] * sy, (t['box'][0] + t['box'][2]) * sx,
                                                    (t['box'][1] + t['box'][3]) * sy])
                        for t in e['anns'] if 'q' in t and 'n' in t]
                ious += U.tooth_ious(gt_t, T_.get(k, []))
            ious = np.asarray(ious) if ious else np.zeros(1)
            hit = ious[ious > 0]
            tq.append(dict(detector=label, xrays=len(ENUM_COPY), teeth=int(len(ious)),
                           found_iou50=round(float((ious >= 0.5).mean()), 3), found_iou75=round(float((ious >= 0.75).mean()), 3),
                           found_iou90=round(float((ious >= 0.9).mean()), 3),
                           median_iou=round(float(np.median(hit)) if len(hit) else 0.0, 3)))
        print('Tooth boxes on the diagnosis X-rays that have an enumeration copy (that copy\'s labels are the truth; '
              'right FDI number and IoU at least 0.5 / 0.75 / 0.9):')
        show(pd.DataFrame(tq))

DET_EVAL = dict(conf=0.001, iou=0.7, max_det=300)   # ultralytics' own validation settings

def validator_boxes(weights, keys, tag, augment=False, meta=None, root=None, conf=None, imgsz=None):
    """The plain detector's boxes exactly as its own validator makes them when it computes mAP: multi-label NMS
    (model.predict keeps one class per box, the validator up to all four), conf 0.001, iou 0.7, up to 300 per X-ray,
    letterboxed batches. Read back from the validator's predictions.json, in original pixels. Also returns the
    diagnosis mAP50-95 / mAP50 the validator printed for these X-rays.
    augment: ultralytics' test-time augmentation (the X-ray at 3 scales, one of them mirrored).
    meta / root: other X-rays than the annotated ones (the self-training pool) and their YOLO folder.
    conf: a higher score floor than the validator's 0.001 (for the pool, where only confident boxes matter).
    imgsz: the detector's own input size (the 1280 YOLO), smaller batches above 1024."""
    dm = meta if meta is not None else ensure_dis_images()
    root = root or DIS_ROOT
    keys = list(keys)
    txt = U.write_list(root, f'{tag}_list', keys, dm)
    yaml_p = U.write_yolo_yaml(root, U.DIS, txt, txt, yaml_name=f'{tag}.yaml')
    run = SCR / 'runs' / f'{tag}_val'
    shutil.rmtree(run, ignore_errors=True)
    ev = dict(DET_EVAL, conf=conf) if conf else DET_EVAL
    sz = imgsz or CFG['DET_IMGSZ']
    m = load_detector(weights, 'yolo').val(data=str(yaml_p), split='val', imgsz=sz, batch=16 if sz <= 1024 else 8,
                                           device=0, plots=False, save_json=True, verbose=False, augment=augment,
                                           project=str(SCR / 'runs'), name=f'{tag}_val', exist_ok=True, **ev)
    pj = Path(getattr(m, 'save_dir', run)) / 'predictions.json'
    stem2key = {Path(dm[k][3]).stem: k for k in keys}
    out = {k: [] for k in keys}
    for p in json.loads(pj.read_text()):
        k = stem2key.get(Path(str(p.get('file_name', ''))).stem) or stem2key.get(str(p['image_id']))
        if k is None:
            continue
        s = dm[k][2]
        x, y, w, h = p['bbox']
        out[k].append(dict(box=[x / s, y / s, (x + w) / s, (y + h) / s], cls=int(p['category_id']) - 1,
                           conf=float(p['score'])))
    return out, dict(mAP=round(100 * float(m.box.map), 2), mAP50=round(100 * float(m.box.map50), 2))

def disease_boxes(weights, splits, cache_name, augment=False, meta=None, root=None, conf=None, imgsz=None):
    """plain-detector boxes for the X-rays of {split name: keys}, one validator run per split.
    Returns (boxes by X-ray, {split: mAP50-95 / mAP50 that ultralytics printed})."""
    p = WORK / f'{cache_name}_validator.pkl'
    if p.exists():
        return pickle.loads(p.read_bytes())
    boxes, printed, ok = {}, {}, True
    for name, keys in splits.items():
        try:
            b, printed[name] = validator_boxes(weights, keys, f'{cache_name}_{name}', augment=augment, meta=meta,
                                               root=root, conf=conf, imgsz=imgsz)
        except Exception as e:
            ok = False
            print(f'!! ultralytics validator failed for {cache_name} {name} ({str(e)[:200]}). Using model.predict '
                  'boxes instead: one class per box, a slightly weaker baseline. Tell Claude.')
            ev = dict(DET_EVAL, conf=conf) if conf else DET_EVAL
            raw = yolo_boxes(weights, keys, agnostic=False, meta=meta, augment=augment, imgsz=imgsz, **ev)
            b = {k: [dict(box=list(map(float, bb)), cls=int(c), conf=float(s)) for bb, c, s in zip(*raw[k])] for k in keys}
            printed[name] = dict(mAP=None, mAP50=None)
        boxes.update(b)
    if ok:
        p.write_bytes(pickle.dumps((boxes, printed)))
    return boxes, printed

DISBOX_A = DISBOX_A2 = None
PRINTED_A = {}
if RUN_PROTOCOL_A and DIS_W_A is not None:
    with stage('disease_inference_A'):
        DISBOX_A, PRINTED_A = disease_boxes(DIS_W_A, {'val': A_VAL, 'test': A_TEST}, 'disbox_A')
        if DIS_W_A2 is not None:
            DISBOX_A2, _ = disease_boxes(DIS_W_A2, {'val': A_VAL, 'test': A_TEST}, 'disbox_A_s1')

## 6. Tooth crops and contralateral twins
Every detected tooth is cropped (extra margin on the root side so periapical lesions stay in view) and linked to its twin: 1x↔2x, 3x↔4x, same tooth number. v15: a second set of crops of every diagnosis X-ray, made with the clean tooth detector; all predictions on validation and test X-rays use these.


In [ ]:
if not MANIFEST.exists():
    with stage('crops'):
        rows, nid = [], 0
        for split in ['dis_train', 'dis_val', 'dis_test']:
            if split not in SETS:
                continue
            r, nid = U.build_crops(SETS[split]['images'], TEETH, SCR / 'crops', split, CFG['CROP_H'],
                                   CFG['CROP_W'], add_gt_only=(split == 'dis_train'), start_id=nid)
            rows += r
            print(split, len(r), 'tooth crops')
        pd.DataFrame(rows).to_csv(MANIFEST, index=False)
if not (WORK / 'crops.zip').exists():
    with stage('zip_crops'):   # ~30k crops saved as ONE file so the version keeps them
        shutil.make_archive(str(WORK / 'crops'), 'zip', root_dir=str(SCR), base_dir='crops')
MAN = pd.read_csv(MANIFEST)
print('crops:', len(MAN), '| with twin:', f"{(MAN.twin_crop_id >= 0).mean():.1%}",
      '| GT-only (missed by detector):', int(MAN.gt_only.sum()))
print('positives per class:', MAN[U.LABEL_COLS].sum().to_dict())

BOXT = SCR / 'crops' / 'box_targets.csv'   # annotated-box offsets per tooth, training X-rays only
if not BOXT.exists():
    U.box_targets(SETS['dis_train']['images'], MAN).to_csv(BOXT, index=False)
print('teeth with a learned-box target:', len(pd.read_csv(BOXT)))

KEYDIR = WORK / 'keys'; KEYDIR.mkdir(exist_ok=True)
def write_keys(name, keys):
    p = KEYDIR / f'{name}.txt'
    p.write_text('\n'.join(keys) + '\n')
    return str(p)
ROWS = MAN[MAN.gt_only == 0].to_dict('records')
ROWS_BY_KEY = {}
for r in ROWS:
    ROWS_BY_KEY.setdefault(r['image_key'], []).append(r)

# v15: CLEAN crops. Every validation and test X-ray is cropped by the clean tooth detector (cell 09), with ids from
# EVAL_CROP_ID0 on; the crops above (first tooth detector, with GT-only teeth) are only ever used to TRAIN classifiers.
# From here on ROWS / ROWS_BY_KEY / TEETH (everything that makes predictions) are the clean ones.
MAN_TRAIN, MANIFEST_TRAIN, TEETH_TRAIN = MAN, MANIFEST, TEETH
MANIFEST_E, MAN_E = MANIFEST, MAN
if CFG.get('CLEAN_EVAL'):
    MANIFEST_E = SCR / 'crops_e' / 'manifest.csv'
    if not MANIFEST_E.exists():
        with stage('crops_clean'):
            CH = 50   # X-rays per task, at most 32 teeth each: 2,000 ids per task (fixed, so the ids never change)
            tasks = []
            for split in ['dis_train', 'dis_val', 'dis_test']:
                if split in SETS:
                    recs = SETS[split]['images']
                    for i in range(0, len(recs), CH):
                        tasks.append((recs[i:i + CH], {r['key']: TEETH_E.get(r['key'], []) for r in recs[i:i + CH]},
                                      str(SCR / 'crops_e'), split, CFG['CROP_H'], CFG['CROP_W'], False,
                                      CFG['EVAL_CROP_ID0'] + len(tasks) * 2_000))
            rows_e = [x for part in U.pmap(U.build_crops_chunk, tasks, workers=_NP, chunksize=1) for x in part]
            pd.DataFrame(rows_e).to_csv(MANIFEST_E, index=False)
    MAN_E = pd.read_csv(MANIFEST_E)
    TEETH = TEETH_E
    ROWS = MAN_E[MAN_E.gt_only == 0].to_dict('records')
    ROWS_BY_KEY = {}
    for r in ROWS:
        ROWS_BY_KEY.setdefault(r['image_key'], []).append(r)
    print(f'clean crops (validation and test X-rays): {len(MAN_E)} teeth of {MAN_E.image_key.nunique()} X-rays')

def train_frame(train_keys, eval_keys):
    """the crops a classifier of one split learns from: the first detector's crops of its training X-rays, the clean
    ones of its validation (early stopping) and test X-rays"""
    trs, evs = set(train_keys), set(eval_keys) - set(train_keys)
    return pd.concat([MAN_TRAIN[MAN_TRAIN.image_key.isin(trs)], MAN_E[MAN_E.image_key.isin(evs)]], ignore_index=True)

def train_manifest(tag, train_keys, eval_keys):
    if not CFG.get('CLEAN_EVAL'):
        return MANIFEST
    p = SCR / 'manifests' / f'{tag}.csv'
    p.parent.mkdir(parents=True, exist_ok=True)
    train_frame(train_keys, eval_keys).to_csv(p, index=False)
    return p


## 6b. Self-training pool
The DENTEX X-rays without diagnosis labels: found in the attached data or downloaded, copies of annotated X-rays removed, teeth found and cropped exactly like the annotated ones. Nothing here runs unless a later step needs it.


In [ ]:
# Self-training pool: the DENTEX X-rays that have no diagnosis labels (634 enumeration + 693 quadrant-only + 1571
# unlabelled). Everything here is made only when a later step needs it (nothing runs if the results are cached).
POOL_ROOT = SCR / 'yolo_pool'            # resized jpgs + empty label files (ultralytics wants a labels folder)
POOL_P = WORK / 'pool.pkl'               # the pool after removing copies of annotated X-rays
POOL_TEETH_P = WORK / 'teeth_pool.pkl'   # tooth detector output on the pool
POOL_MAN_P = WORK / 'pool_manifest.csv'  # tooth crops of the pool; their crop ids are fixed the first time
POOL_CROPS = SCR / 'pool_crops'
POOL = pickle.loads(POOL_P.read_bytes()) if POOL_P.exists() else None
POOL_META = None                         # {key: (W, H, scale, jpg)} once this session has the jpgs
NPROC = max(1, min(4, os.cpu_count() or 1))

def hf_get(f):
    """one file of the DENTEX dataset from Hugging Face (into scratch)"""
    from huggingface_hub import hf_hub_download
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    print(f'free disk at {RAW}: {shutil.disk_usage(RAW).free/1e9:.0f} GB')
    return hf_hub_download(REPO, f, repo_type='dataset', local_dir=str(RAW / 'dl'), token=token)

def pool_sources():
    """{subset: image paths}: from the DENTEX folders this session already has, the rest from training_data.zip
    (an attached copy, else Hugging Face; ~11 GB, about a minute on Kaggle)"""
    (RAW / 'pool').mkdir(parents=True, exist_ok=True)
    roots = [Path(RAW_X), RAW / 'pool']
    found = U.find_pool(roots)
    missing = [t for t in U.POOL_TAGS if not found.get(t)]
    if missing:
        with stage('pool_download'):
            zips = [q for q in find_inputs('training_data.zip') if os.path.isfile(q)]
            z = zips[0] if zips else hf_get('DENTEX/training_data.zip')
            n = U.safe_extract(z, RAW / 'pool', skip=(), keep=lambda name: U.pool_tag(name) in missing)
            print(f'extracted {n} files of {missing} from {z}')
            if not zips:
                os.remove(z)
        found = U.find_pool(roots)
    for t in U.POOL_TAGS:
        n = len(found.get(t, []))
        print(f'   pool {t}: {n} X-rays' + ('' if n == U.POOL_EXPECTED[t] else f' (DENTEX has {U.POOL_EXPECTED[t]})'))
    return found

def ensure_pool():
    """the pool X-rays of this session as resized jpgs (made like the annotated ones). The first time, copies of
    annotated X-rays (any split) and repeats inside the pool are removed, and the list is saved."""
    global POOL, POOL_META
    if POOL_META is not None:
        return POOL
    if POOL is not None and all(Path(r['src']).exists() for r in POOL):
        cand = POOL
    else:
        found = pool_sources()
        cand = [dict(key=f'pool_{t}/{Path(p).stem}', tag=t, src=p) for t in U.POOL_TAGS for p in found.get(t, [])]
        if POOL is not None:    # new session: same pool as before, files found again
            where = {c['key']: c['src'] for c in cand}
            lost = [r['key'] for r in POOL if r['key'] not in where]
            assert not lost, f'{len(lost)} pool X-rays of the earlier run are missing now, e.g. {lost[:3]}'
            for r in POOL:
                r['src'] = where[r['key']]
            cand = POOL
    with stage('pool_images'):
        tasks = [(r['src'], str(POOL_ROOT / 'images' / 'all' / (r['key'].replace('/', '__') + '.jpg')),
                  CFG['LONG_SIDE']) for r in cand]
        res = U.pmap(U.export_pool_image, tasks, workers=NPROC)
        bad = [r['src'] for r, x in zip(cand, res) if x is None]
        if bad:
            print(f'!! {len(bad)} pool files are not readable images and are skipped, e.g. {bad[:3]}')
        ok = [i for i, x in enumerate(res) if x is not None]
        assert ok, 'no readable pool X-rays found'
        cand, tasks, res = [cand[i] for i in ok], [tasks[i] for i in ok], [res[i] for i in ok]
        for r, t, (w, h, s, _) in zip(cand, tasks, res):
            r.update(w=int(w), h=int(h), scale=float(s), jpg=t[1])
        if POOL is None:
            dm = ensure_dis_images()
            lab = sorted(dm)
            th_lab = U.pmap(U.thumb_file, [dm[k][3] for k in lab], workers=NPROC)
            th_pool = [x[3] for x in res]
            d_lab, j_lab = U.rms_nearest(th_pool, th_lab)
            d_in, j_in = U.rms_nearest(th_pool)
            DUP = 0.17   # thumbnail correlation above 0.985 = the same X-ray (saved twice ~0.05, other X-rays > 0.3)
            copy_lab, copy_in = d_lab < DUP, (d_in < DUP) & ~(d_lab < DUP)
            print(f'pool: {len(cand)} X-rays, {int(copy_lab.sum())} are copies of annotated X-rays and '
                  f'{int(copy_in.sum())} repeat another pool X-ray: both removed')
            for i in np.flatnonzero(copy_lab)[:10]:
                print(f'   {cand[i]["key"]} = {lab[j_lab[i]]} (difference {d_lab[i]:.3f})')
            q = np.percentile(d_lab, [0, 1, 5, 50]) if len(d_lab) else [np.nan] * 4
            print('   difference to the closest annotated X-ray (copies < %.2f): min %.3f, 1%% %.3f, 5%% %.3f, median %.3f'
                  % ((DUP,) + tuple(q)))
            POOL = [r for r, a, b in zip(cand, copy_lab, copy_in) if not (a or b)]
            POOL_P.write_bytes(pickle.dumps(POOL))
        (POOL_ROOT / 'labels' / 'all').mkdir(parents=True, exist_ok=True)
        for r in POOL:
            (POOL_ROOT / 'labels' / 'all' / (r['key'].replace('/', '__') + '.txt')).touch()
        POOL_META = {r['key']: (r['w'], r['h'], r['scale'], r['jpg']) for r in POOL}
        # v15 audit: how close does any pool X-ray come to a diagnosis X-ray? (copies, < DUP_HARD, were removed above)
        if AUDIT is not None and len(AUDIT.get('th_ann', [])):
            at = {c['key']: x[3] for c, x in zip(cand, res)}
            thp = np.stack([at[r['key']] for r in POOL if r['key'] in at])
            dmin, jmin = U.rms_nearest(thp, AUDIT['th_ann'])
            soft = [(r['key'], ANN_KEYS[j], d) for r, d, j in zip([r for r in POOL if r['key'] in at], dmin, jmin)
                    if d < CFG['DUP_SOFT']]
            print(f'pool check (v15): the pool X-ray closest to a diagnosis X-ray is at {dmin.min():.3f} '
                  f'(copies < {CFG["DUP_HARD"]}); {len(soft)} pool X-rays are closer than {CFG["DUP_SOFT"]}'
                  + ''.join(f'\n   {a} ~ {b} ({d:.3f})' for a, b, d in sorted(soft, key=lambda t: t[2])[:10]))
    print('pool X-rays:', len(POOL), {t: sum(r['tag'] == t for r in POOL) for t in U.POOL_TAGS})
    return POOL

POOL_ROOT_HI = SCR / 'yolo_pool_hi'
POOL_META_HI = None

def ensure_pool_hi():
    """the pool X-rays exported a second time with the long side at DET_IMGSZ_XHI (high-resolution student)"""
    global POOL_META_HI
    if POOL_META_HI is not None:
        return POOL_META_HI
    ensure_pool()
    with stage('pool_images_hi'):
        tasks = [(r['src'], str(POOL_ROOT_HI / 'images' / 'all' / (r['key'].replace('/', '__') + '.jpg')),
                  CFG['DET_IMGSZ_XHI']) for r in POOL]
        res = U.pmap(U.export_pool_image, tasks, workers=NPROC)
        POOL_META_HI = {r['key']: (x[0], x[1], x[2], t[1]) for r, t, x in zip(POOL, tasks, res) if x is not None}
        (POOL_ROOT_HI / 'labels' / 'all').mkdir(parents=True, exist_ok=True)
        for k in POOL_META_HI:
            (POOL_ROOT_HI / 'labels' / 'all' / (k.replace('/', '__') + '.txt')).touch()
    return POOL_META_HI

def pool_keys():
    return [r['key'] for r in (POOL if POOL is not None else ensure_pool())]

def pool_teeth():
    """tooth detector boxes on the pool (same settings as on the annotated X-rays)"""
    if POOL_TEETH_P.exists():
        return pickle.loads(POOL_TEETH_P.read_bytes())
    ensure_pool()
    with stage('pool_tooth_inference'):
        keys = pool_keys()
        raw = yolo_boxes(TOOTH_W, keys, conf=0.10, iou=0.5, agnostic=True, max_det=64, arch_name=CFG['TOOTH_MODEL'],
                         meta=POOL_META)
        teeth = {k: U.teeth_from_yolo(*raw[k]) for k in keys}
        POOL_TEETH_P.write_bytes(pickle.dumps(teeth))
    nt = [len(v) for v in teeth.values()]
    print(f'pool teeth per X-ray: mean {np.mean(nt):.1f}, min {min(nt)}, max {max(nt)}')
    return teeth

def pool_manifest(make_crops=True):
    """tooth crops of the pool (from the full-resolution X-rays, like the annotated ones). Crop ids do not depend on
    how the work is split, so predictions saved in an earlier session still match."""
    man_scr = POOL_CROPS / 'manifest.csv'
    if man_scr.exists():
        return pd.read_csv(man_scr)
    if not make_crops and POOL_MAN_P.exists():
        return pd.read_csv(POOL_MAN_P)
    teeth = pool_teeth()
    ensure_pool()
    with stage('pool_crops'):
        recs = [dict(key=r['key'], path=r['src'], anns=[]) for r in POOL]
        CH = 100   # X-rays per task; at most 32 teeth each, so 10000 ids per task are enough
        tasks = [(recs[i:i + CH], {r['key']: teeth.get(r['key'], []) for r in recs[i:i + CH]}, str(POOL_CROPS), 'pool',
                  CFG['CROP_H'], CFG['CROP_W'], False, 10_000_000 + (i // CH) * 10_000) for i in range(0, len(recs), CH)]
        rows = [x for part in U.pmap(U.build_crops_chunk, tasks, workers=NPROC, chunksize=1) for x in part]
        man = pd.DataFrame(rows)
        if POOL_MAN_P.exists():
            old = pd.read_csv(POOL_MAN_P)
            assert old.crop_id.tolist() == man.crop_id.tolist() and old.image_key.tolist() == man.image_key.tolist(), \
                'pool crops differ from the earlier run'
        else:
            man.to_csv(POOL_MAN_P, index=False)
        man.to_csv(man_scr, index=False)
    print(f'pool crops: {len(man)} teeth of {man.image_key.nunique()} X-rays')
    return man


## 7. TwinArch (both GPUs, torchrun) + smoke test
The smoke test trains one epoch on 40 X-rays so any error shows up in a few minutes instead of after hours.

In [ ]:
import timm, copy
try:
    _ = timm.create_model(CFG['CLS_BACKBONE'], pretrained=True, num_classes=0, in_chans=1)   # cache weights once
except Exception as e:
    print('backbone not available:', e, '-> falling back to convnext_tiny')
    CFG['CLS_BACKBONE'] = 'convnext_tiny.fb_in22k_ft_in1k'
    _ = timm.create_model(CFG['CLS_BACKBONE'], pretrained=True, num_classes=0, in_chans=1)
TRAIN_SCRIPT = str(Path('train_arch.py').resolve())
FAILED = []   # classifier runs that did not finish even after one retry (reported at the end)

def cls_done(out, preds):
    out = Path(out)
    return (out / 'log.json').exists() and all((out / f'pred_{n}.csv').exists() for n in preds)

def launcher():
    return f'torchrun --standalone --nproc_per_node={NGPU}' if NGPU > 1 else sys.executable

def run_cls(variant, train_k, val_k, preds, out, seed=0, epochs=None, extra='', retries=1, manifest=None):
    """train one classifier; returns its folder, or None if it failed twice (the rest of the run goes on)"""
    out = Path(out)
    done = lambda: cls_done(out, preds)
    if done():
        print('already trained:', out)
        return out
    pk = ' '.join(f'{n}={p}' for n, p in preds.items())
    cmd = (f"{launcher()} {TRAIN_SCRIPT} --manifest {manifest or MANIFEST} --train-keys {train_k} --val-keys {val_k} "
           f"--pred-keys {pk} --variant {variant} --backbone {CFG['CLS_BACKBONE']} "
           f"--epochs {epochs or CFG['CLS_EPOCHS']} --img-bs {CFG['CLS_IMG_BS']} --crop-h {CFG['CROP_H']} "
           f"--crop-w {CFG['CROP_W']} --seed {seed} --out {out} --workers 2 --box-targets {BOXT} "
           f"--box-src {CFG['BOX_SRC']} {CFG['CLS_EXTRA']} {extra}")
    for attempt in range(1 + retries):
        free_gpu()
        print(cmd)
        get_ipython().system(cmd)
        if done():
            return out
        print(f'!! {out.name} did not finish (attempt {attempt + 1}), scroll up for the error'
              + ('. Retrying once.' if attempt < retries else '. Skipping it, the rest of the run goes on.'))
    FAILED.append(out.name)
    return None

BOX_COLS = {'learned, token head': 'b', 'learned, map head': 's'}   # column prefixes in pred_*.csv

def load_boxes(out, name, prefix='b'):
    """learned per-disease box offsets written by train_arch.py (None if that head was not trained)"""
    d = pd.read_csv(Path(out) / f'pred_{name}.csv')
    cols = [f'{prefix}{k}_{e}' for k in range(4) for e in ('x1', 'y1', 'x2', 'y2')]
    if not all(c in d for c in cols):
        return None
    return {int(c): v.reshape(4, 4) for c, v in zip(d.crop_id, d[cols].values.astype(float))}

def load_probs(out, name, keys=None):
    d = pd.read_csv(Path(out) / f'pred_{name}.csv')
    if keys is not None:   # a reused classifier must have scored exactly the teeth of today's crops
        want = {r['crop_id'] for k in keys for r in ROWS_BY_KEY.get(k, [])}
        got = set(d.crop_id.astype(int))
        if want != got:
            raise RuntimeError(f'{Path(out).name}/pred_{name}.csv does not match the current crops '
                               f'({len(got)} vs {len(want)} teeth). Bump CLS_TAG to retrain it.')
    return {int(c): np.array(v, float) for c, v in zip(d.crop_id, d[['p_caries', 'p_deep', 'p_peri', 'p_impacted']].values)}

def make_preds(mode, keys, probs, disbox=None, conf_pow=0.5, adapter=None, oracle=None, mix=0.0, refine=None, topk=None):
    """mode 'tooth' : our method (tooth boxes x classifier probabilities), boxes learned, adapted or raw.
       mode 'det'   : plain YOLO disease detector baseline, FDI taken from the matched tooth.
       mode 'fusion': ANALYSIS, detector boxes with the score mixed with the tooth classifier's probability.
       topk: keep only the best topk boxes per X-ray (None = all; the scorers apply their own limits)."""
    keys = set(keys)
    if mode in ('det', 'fusion'):
        pf = probs_by_fdi(probs) if mode == 'fusion' else None
        P = U.preds_from_disdet({k: v for k, v in disbox.items() if k in keys}, TEETH, pf, mix=mix)
    else:
        rows = [r for k in keys for r in ROWS_BY_KEY.get(k, [])]
        P = U.preds_from_teeth(rows, probs, conf_pow, adapter, oracle, refine)
    return U.top_k_per_image(P, topk) if topk else P

def probs_by_fdi(probs):
    return {(r['image_key'], int(r['fdi'])): probs[r['crop_id']] for r in ROWS if r['crop_id'] in probs}

def coco(P):
    """the boxes the COCO / DENTEX scorer gets: all of them (pycocotools itself keeps the best 100 per X-ray and
    class), or the best COCO_TOPK per X-ray if that is set"""
    k = CFG.get('COCO_TOPK')
    return U.top_k_per_image(P, k) if k else P

def val_ap(gt, P, keys):
    """diagnosis AP on validation X-rays (the same number dentex_eval gives, computed for the diagnosis only)"""
    return U.diag_eval_fast(gt, coco(P), keys)['AP']

def score_row(gt, P, keys, name):
    """one table row: DENTEX/COCO scores, plus the diagnosis mAP the way ultralytics' validator computes it
    (all boxes) with its current and its old AP formula"""
    s = U.summarize(U.dentex_eval(gt, coco(P), keys), name)
    ul = U.ul_eval(gt, P, keys)
    s.update(ul_AP=round(ul['mAP'], 2), ul_AP50=round(ul['mAP50'], 2), ul_old_AP=round(ul['mAP_old'], 2),
             ul_old_AP50=round(ul['mAP50_old'], 2), ul_old_per_class_AP50=ul['per_class_AP50_old'])
    return s

def select_and_test(probs_val, probs_test, gt_val, gt_test, val_keys, test_keys, name, adapter=None,
                    refine_val=None, refine_test=None):
    """Only one knob (how much the tooth-detector confidence counts) is chosen, on validation images.
    Returns the test row and the predictions {val, test} of the chosen setting."""
    best = None
    for cp in (0.0, 0.5, 1.0):
        pv = make_preds('tooth', val_keys, probs_val, conf_pow=cp, adapter=adapter, refine=refine_val, topk=None)
        ap = val_ap(gt_val, pv, val_keys)
        if best is None or ap > best[0]:
            best = (ap, cp, pv)
    pt = make_preds('tooth', test_keys, probs_test, conf_pow=best[1], adapter=adapter, refine=refine_test, topk=None)
    s = score_row(gt_test, pt, test_keys, name)
    s.update(chosen=f'conf_pow={best[1]}', val_diag_AP=round(best[0], 2))
    return s, dict(val=best[2], test=pt, cfg=dict(conf_pow=best[1]))

def model_rows(out, gt_val, gt_test, val_keys, test_keys, name, adapter):
    """One row per box source for the same scores: learned boxes (token / map head), median adapter, raw tooth
    boxes. The box source is a setting like any other, so it is picked on VALIDATION: that row gets
    picked=True. Returns (rows, {val, test} predictions of the picked row)."""
    pv, pt = load_probs(out, 'val', val_keys), load_probs(out, 'test', test_keys)
    rows, preds = [], []
    for label, pre in BOX_COLS.items():
        bv, bt = load_boxes(out, 'val', pre), load_boxes(out, 'test', pre)
        if bv is not None and bt is not None:
            s, p = select_and_test(pv, pt, gt_val, gt_test, val_keys, test_keys, f'{name}, {label}',
                                   refine_val=bv, refine_test=bt)
            s['boxes'] = label; rows.append(s); preds.append(p)
    if adapter:
        s, p = select_and_test(pv, pt, gt_val, gt_test, val_keys, test_keys, name + ', median box adapter', adapter=adapter)
        s['boxes'] = 'median adapter'; rows.append(s); preds.append(p)
    s, p = select_and_test(pv, pt, gt_val, gt_test, val_keys, test_keys, name + ', raw tooth boxes')
    s['boxes'] = 'tooth'; rows.append(s); preds.append(p)
    k = int(np.argmax([r['val_diag_AP'] for r in rows]))
    for i, r in enumerate(rows):
        r['picked'] = i == k
    preds[k]['cfg']['boxes'] = rows[k]['boxes']   # how to make the same predictions on other X-rays (the pool)
    return rows, preds[k]

def det_row(disbox, gt_val, gt_test, val_keys, test_keys, name):
    """plain detector baseline row; its validation predictions are kept for the fusion"""
    pv = make_preds('det', val_keys, {}, disbox, topk=None)
    pt = make_preds('det', test_keys, {}, disbox, topk=None)
    s = score_row(gt_test, pt, test_keys, name)
    s.update(chosen='-', variant='plain YOLO', boxes='detector', picked=True, val_diag_AP=round(val_ap(gt_val, pv, val_keys), 2))
    return s, dict(val=pv, test=pt)

# WBF settings tried on validation. Two or three models in the order (ours, YOLO[, YOLO]) or (YOLO, YOLO): v10 picked
# a classifier weight of half a YOLO's every time, the edge of its grid, so lower ones are tried too.
FUSION_GRID = {2: [(w, thr) for w in ((1, 1), (2, 1), (1, 2), (1, 3)) for thr in (0.5, 0.6)],
               3: [(w, thr) for w in ((1, 1, 1), (2, 1, 1), (0.5, 1, 1), (0.25, 1, 1)) for thr in (0.5, 0.6)]}

def kinds_grid(kinds):
    """WBF settings for any mix of members, by kind: 'c' = our classifier, 'd' = a detector, 's' = a self-trained
    student, '?' = may be left out (weight 0). A classifier gets 0.25, 0.5 or 1 x a detector's weight, students 1 or
    2 x their kind's."""
    out = []
    students = any('s' in k for k in kinds)
    optional = [i for i, k in enumerate(kinds) if '?' in k]
    for wc in ((0.25, 0.5, 1.0) if any(k[0] == 'c' for k in kinds) else (1.0,)):
        for ws in ((1.0, 2.0) if students else (1.0,)):
            w = [(wc if k[0] == 'c' else 1.0) * (ws if 's' in k else 1.0) for k in kinds]
            for drop in ((False, True) if optional else (False,)):
                wd = tuple(0.0 if (drop and i in optional) else x for i, x in enumerate(w))
                out += [(wd, thr) for thr in (0.5, 0.6)]
    return out

def grid_v12(kinds, transplant=True):
    """v12 WBF settings, as kinds_grid plus: our classifiers may sit on the DETECTORS' boxes for their tooth
    (U.transplant_boxes: the detectors place the box, the classifier scores it). Then a classifier no longer blurs
    the fused box, so it may also count more (up to 2 x a detector). Entries are (weights, iou, detector boxes)."""
    has_c, has_d = any(k[0] == 'c' for k in kinds), any(k[0] == 'd' for k in kinds)
    students = any('s' in k for k in kinds)
    optional = [i for i, k in enumerate(kinds) if '?' in k]
    out = []
    for tp in ((False, True) if (transplant and CFG.get('TRANSPLANT', True) and has_c and has_d) else (False,)):
        for wc in (((0.25, 0.5, 1.0, 2.0) if tp else (0.25, 0.5, 1.0)) if has_c else (1.0,)):
            for ws in ((1.0, 2.0) if students else (1.0,)):
                w = [(wc if k[0] == 'c' else 1.0) * (ws if 's' in k else 1.0) for k in kinds]
                for drop in ((False, True) if optional else (False,)):
                    wd = tuple(0.0 if (drop and i in optional) else x for i, x in enumerate(w))
                    out += [(wd, thr, tp) for thr in (0.5, 0.6)]
    return out

def wbf_members(models, w, thr, part):
    """WBF of the members with a weight above 0 (a weight of 0 = that member is left out)"""
    keep = [i for i, x in enumerate(w) if x > 0]
    return U.wbf([models[i][part] for i in keep], weights=[w[i] for i in keep], iou_thr=thr)

DONOR_IOU = 0.55   # the detectors' boxes that classifiers borrow: their plain WBF (equal weights, this IoU)

def with_detector_boxes(models, kinds, parts=('val', 'test'), rows_by_key=None):
    """the same members, with every classifier member ('c...' kind) moved onto the boxes of the detector members
    of this fusion (U.transplant_boxes); detectors are unchanged"""
    rbk = rows_by_key if rows_by_key is not None else ROWS_BY_KEY
    dets = [m for m, k in zip(models, kinds) if k[0] == 'd']
    out = [dict(m) for m in models]
    for part in parts:
        donors = U.wbf([m[part] for m in dets], iou_thr=DONOR_IOU)
        for m, k in zip(out, kinds):
            if k[0] == 'c':
                m[part] = U.transplant_boxes(m[part], donors, rbk)
    return out

def fusion_row(models, gt_val, gt_test, val_keys, test_keys, name, kinds=None, per_class=False, grid=None):
    """weighted boxes fusion of several models' predictions (each {val, test}; ours first when it takes part).
    The weights and the IoU threshold are picked on VALIDATION X-rays: one setting for all diseases, or with
    per_class=True one setting per disease (each disease's own validation AP decides).
    grid: (weights, iou[, detector boxes]) settings to try; default as in v11 (grid_v12 adds detector boxes)."""
    grid = grid or (kinds_grid(kinds) if kinds else FUSION_GRID[len(models)])
    grid = [g if len(g) == 3 else (g[0], g[1], False) for g in grid]
    variants = {False: models}
    if any(g[2] for g in grid):
        variants[True] = with_detector_boxes(models, kinds)
    vals, best, best_d = {}, None, {}
    for w, thr, tp in grid:
        fv = wbf_members(variants[tp], w, thr, 'val')
        ev = U.diag_eval_fast(gt_val, coco(fv), val_keys)
        vals[(w, thr, tp)] = fv
        if best is None or ev['AP'] > best[0]:
            best = (ev['AP'], w, thr, tp)
        for d in range(4):
            ap = ev['per_class_AP'].get(d + 1, float('nan'))
            if ap == ap and (d not in best_d or ap > best_d[d][0]):
                best_d[d] = (ap, (w, thr, tp))
    desc = lambda w, thr, tp: f"{':'.join(f'{x:g}' for x in w)} iou {thr}" + (', detector boxes' if tp else '')
    if not per_class:
        ft = wbf_members(variants[best[3]], best[1], best[2], 'test')
        s = score_row(gt_test, ft, test_keys, name)
        s.update(chosen='WBF ' + desc(*best[1:]), val_diag_AP=round(best[0], 2), boxes='fused')
        return s, dict(val=vals[best[1:]], test=ft, cfg=dict(w=best[1], thr=best[2], tp=best[3]))
    cfg_d = {d: best_d[d][1] if d in best_d else best[1:] for d in range(4)}
    tests = {c: wbf_members(variants[c[2]], c[0], c[1], 'test') for c in set(cfg_d.values())}
    fv = [p for d in range(4) for p in vals[cfg_d[d]] if p['d'] == d]
    ft = [p for d in range(4) for p in tests[cfg_d[d]] if p['d'] == d]
    s = score_row(gt_test, ft, test_keys, name)
    s.update(chosen='per disease: ' + ', '.join(desc(*cfg_d[d]).replace(' iou ', '/') for d in range(4)),
             val_diag_AP=round(val_ap(gt_val, fv, val_keys), 2), boxes='fused')
    return s, dict(val=fv, test=ft, cfg=dict(per_class=cfg_d))

def metric_check(printed, splits):
    """the same detector on the same X-rays, scored four ways.
    printed: {split: what ultralytics' validator printed}; splits: [(label, split, gt_records, keys, predictions)]"""
    cols = {}
    for label, name, gt, keys, P in splits:
        ul = U.ul_eval(gt, P, keys)
        co = U.dentex_eval(gt, coco(P), keys)['diagnosis']
        pr = printed.get(name, {})
        cols[(label, 'AP')] = [pr.get('mAP'), round(ul['mAP'], 2), round(ul['mAP_old'], 2), round(co['AP'], 2)]
        cols[(label, 'AP50')] = [pr.get('mAP50'), round(ul['mAP50'], 2), round(ul['mAP50_old'], 2), round(co['AP50'], 2)]
    idx = ['ultralytics model.val() itself', 'our copy of it (same formula, same boxes)',
           'our copy, old AP formula (ultralytics <= 8.4.36, the papers)', 'COCO / DENTEX (our main metric)']
    return pd.DataFrame(cols, index=idx)

def efficiency(out):
    try:
        return json.loads((Path(out) / 'log.json').read_text()).get('efficiency', {})
    except Exception:
        return {}

def yolo_cost(weights, arch_name='yolo', shape=(512, 1024), n=30):
    """params (M), GFLOPs and model ms per X-ray (fp16, batch 1) at one fixed letterboxed panoramic size,
    timed after warm-up with cuda synchronize (a varying input size re-tunes the GPU kernels every image)"""
    if weights is None:
        return {}
    out = {}
    try:
        det = load_detector(weights, arch_name)
        out['params_M'] = sum(p.numel() for p in det.model.parameters()) / 1e6
        try:
            from ultralytics.utils.torch_utils import get_flops
            out['GFLOPs'] = float(get_flops(det.model, list(shape))) or None
        except Exception as e:
            out['flop_error'] = str(e)[:80]
        if torch.cuda.is_available():
            m = copy.deepcopy(det.model)
            try:
                m = m.fuse()
            except Exception:
                pass
            m = m.cuda().half().eval()
            x = torch.zeros(1, 3, *shape, device='cuda', dtype=torch.half)
            with torch.no_grad():
                for _ in range(10):
                    m(x)
                torch.cuda.synchronize(); t0 = time.time()
                for _ in range(n):
                    m(x)
                torch.cuda.synchronize()
            out['ms_per_xray'] = (time.time() - t0) / n * 1000
            del m, x
            free_gpu()
    except Exception as e:
        out['error'] = str(e)[:100]
    return out

def bootstrap_table(gt_records, preds_by_name, keys, ref, n_boot=500):
    """paired image-level bootstrap: every model and the reference are scored on the same resampled image sets"""
    bs = U.paired_bootstrap(gt_records, {k: coco(v) for k, v in preds_by_name.items()}, keys, ref=ref, n_boot=n_boot)
    return pd.DataFrame([dict(model=k, **v) for k, v in bs.items()])

def show_boot(df):
    show(df.drop(columns=[c for c in ('dAP_lo', 'dAP_hi') if c in df]))

UL_COLS = ['model', 'seed', 'diag_AP', 'diag_AP50', 'ul_AP', 'ul_AP50', 'ul_old_AP', 'ul_old_AP50']

if CFG['SMOKE_TEST'] and not (WORK / 'cls' / f'smoke_{CLS_TAG}' / 'pred_val.csv').exists():
    with stage('smoke_test'):
        tr_k = write_keys('smoke_train', A_TRAIN[:40]); va_k = write_keys('smoke_val', A_VAL[:10])
        run_cls('full', tr_k, va_k, {'val': va_k}, WORK / 'cls' / f'smoke_{CLS_TAG}', epochs=1,
                extra='--max-images 40', retries=0)
        assert (WORK / 'cls' / f'smoke_{CLS_TAG}' / 'pred_val.csv').exists(), 'smoke test failed, scroll up for the error'
        FAILED.clear()
        print('smoke test passed: TwinArch trains with DDP on', NGPU, 'GPU(s)')
        print('efficiency on this GPU:', efficiency(WORK / 'cls' / f'smoke_{CLS_TAG}'))


## 7b. The 1280 YOLO, test-time augmentation, self-training (three rounds) and the big fusions (one split at a time)
The teachers' settings (fusion weights, thresholds) come from the split's validation X-rays; its test X-rays are only scored. Every fusion of ours is compared with the same detectors fused without our classifiers, every self-trained model with the same model without the pool, every fine-tuned student with the student itself.


In [ ]:
# Everything v11 adds, one split at a time (each protocol B split, and protocol A):
#  * a sharper YOLO: the same YOLO11m trained at 1280 instead of 1024 (one more model to fuse, and a control)
#  * test-time augmentation for every model, the baselines included
#  * self-training. TEACHER = our crop classifier + the split's YOLOs (seed 0, seed 1, 1280), fused with WBF (weights
#    and IoU picked on the split's validation X-rays), with TTA. It labels the DENTEX X-rays that have no diagnosis
#    labels: boxes above a per-disease threshold (best F1 on validation) for a student YOLO (at 1280), and a soft
#    score per tooth and disease for a student classifier. Both students start again from COCO / ImageNet weights and
#    train on the split's annotated X-rays plus the pseudo-labelled pool (Noisy Student, Xie et al. 2020; pseudo labels
#    + WBF + TTA as in Kaggle detection competitions). Validation and test X-rays are never pseudo-labelled.
#  * bigger fusions, each with a detector-only control made of the same detectors
# Every step is cached in WORK and skipped when done.
TTA_SFX = ' + TTA'
YOLO_HI = 'plain YOLO at 1280'
YOLO_TRIO = 'YOLO + YOLO seed 1 + YOLO 1280, fused (control)'
C_TRIO = 'crop only + three YOLOs, fused'
ST_YOLO, ST_CLS = 'self-trained YOLO (1280)', 'self-trained crop only'
ST_PAIR = ST_CLS + ' + self-trained YOLO, fused'
DET_ALL = 'four YOLOs incl. self-trained, fused (control)'
OURS_ALL = 'ours, everything: 3 classifiers + 4 YOLOs, fused'
OURS_ALL_PC = OURS_ALL + ', weights per disease'
# v12 (see v12_rows)
ST_FT = ST_YOLO + ', fine-tuned on annotated'
OURS_ALL_DB = OURS_ALL + ', detector boxes allowed'
R1_OURS = 'v12 round 1: crop only + self-trained crop only + 4 YOLOs, fused'
R1_DET = 'v12 round 1: the same 4 YOLOs, fused (control)'
ST2_YOLO = 'round-2 self-trained YOLO (1280)'
ST2_FT = ST2_YOLO + ', fine-tuned on annotated'
ST2_Y26 = 'round-2 self-trained YOLO26m (1280)'
ST2_Y26_FT = ST2_Y26 + ', fine-tuned on annotated'
ST2_Y26L = 'round-2 self-trained YOLO26l (1280)'
ST2_CLS = 'round-2 self-trained crop only'
R2_OURS = 'v14 round 2: classifiers + self-trained YOLOs (+ plain YOLOs), fused'
R2_DET = 'v14 round 2: the same YOLOs, fused (control)'
ST3_YOLO = 'round-3 self-trained YOLO26m (1536)'
OURS_V12 = 'ours v14: classifiers + self-trained YOLOs incl. round 3 (+ plain YOLOs), fused'
DET_V12 = 'v14: the same YOLOs, fused (control)'
OURS_V12_BEST = 'ours v14 headline (round 1, 2 or 3, TTA or not: picked on validation)'
ST_FAILED = []

def matched_pairs():
    """(model, the control it has to beat): ours vs the same detectors without our classifiers, students vs the same
    model trained without the pool, and TTA vs no TTA. Bootstrapped on the same test X-rays."""
    C = NAMES['single']
    pairs = [(C + FUSED_BOTH, YOLO_PAIR), (C_TRIO, YOLO_TRIO), (OURS_ALL, DET_ALL), (OURS_ALL_PC, DET_ALL),
             (ST_YOLO, YOLO_HI), (ST_CLS, C)]
    if CFG.get('V12'):   # v12: each new piece against the same thing without it
        pairs += [(OURS_ALL_DB, OURS_ALL), (ST_FT, ST_YOLO), (R1_OURS, R1_DET), (ST2_YOLO, ST_YOLO),
                  (ST2_FT, ST2_YOLO), (ST2_Y26, ST2_YOLO), (ST2_Y26L, ST2_Y26), (ST2_CLS, ST_CLS), (R2_OURS, R2_DET),
                  (ST3_YOLO, ST2_Y26), (OURS_V12, DET_V12), (OURS_V12, R2_OURS), (OURS_V12, OURS_ALL)]
    extra = [('plain YOLO' + TTA_SFX, 'plain YOLO'), (C + TTA_SFX, C), (OURS_ALL + TTA_SFX, OURS_ALL)]
    if CFG.get('V12'):
        extra += [(OURS_V12 + TTA_SFX, OURS_V12), (OURS_V12_BEST, OURS_ALL)]
    return pairs + [(a + TTA_SFX, b + TTA_SFX) for a, b in pairs] + extra

def matched_bootstrap(gt, picked, keys, n_boot, **extra):
    """paired bootstrap of every matched pair present in `picked`; one row per pair"""
    out = []
    for ref in dict.fromkeys(b for _, b in matched_pairs()):
        mods = [a for a, b in matched_pairs() if b == ref and a in picked]
        if ref not in picked or not mods:
            continue
        bs = U.paired_bootstrap(gt, {k: coco(picked[k]['test']) for k in [ref] + mods}, keys, ref=ref, n_boot=n_boot)
        out += [dict(extra, model=k, control=ref, **bs[k]) for k in mods]
    return out

def cls_predict(src, variant, keysets, dest, tta, manifest=None):
    """scores of an already trained classifier (src/best.pt) on {name: key file}, averaged over `tta` fixed views"""
    dest = Path(dest)
    if cls_done(dest, keysets):
        return dest
    if not (Path(src) / 'best.pt').exists():
        raise RuntimeError(f'{Path(src) / "best.pt"} is missing: attach the earlier output that has the trained '
                           'classifiers (their best.pt files)')
    pk = ' '.join(f'{n}={p}' for n, p in keysets.items())
    cmd = (f"{launcher()} {TRAIN_SCRIPT} --predict-only --weights {Path(src) / 'best.pt'} "
           f"--manifest {manifest or MANIFEST} --pred-keys {pk} --variant {variant} --backbone {CFG['CLS_BACKBONE']} "
           f"--img-bs {CFG['CLS_IMG_BS']} --crop-h {CFG['CROP_H']} --crop-w {CFG['CROP_W']} --out {dest} --workers 2 "
           f"--box-src {CFG['BOX_SRC']} --tta {tta} {CFG['CLS_EXTRA']}")
    free_gpu()
    print(cmd)
    get_ipython().system(cmd)
    if not cls_done(dest, keysets):
        raise RuntimeError(f'{dest.name}: classifier predictions failed, scroll up for the error')
    return dest

def evalc(out, variant, keysets, V=1):
    """v15: a trained classifier's scores on the CLEAN crops of the validation and test X-rays (cropped by the tooth
    detector that never saw a diagnosis X-ray), cached next to its own folder. V = number of TTA views."""
    if out is None:
        return None
    if not CFG.get('CLEAN_EVAL'):
        return out if V == 1 else cls_predict(out, variant, keysets, WORK / 'cls' / f'{Path(out).name}_tta{V}', V)
    dest = WORK / 'cls' / (f'{Path(out).name}_E' + (f'_tta{V}' if V > 1 else ''))
    return cls_predict(out, variant, keysets, dest, V, manifest=MANIFEST_E)

def det_preds(boxes, keys, teeth):
    """plain-detector boxes as predictions (FDI from the matched tooth) for X-rays outside ROWS (the pool)"""
    return U.preds_from_disdet({k: boxes.get(k, []) for k in keys}, teeth)

def cls_preds_on(rows, out, name, cfg, adapter):
    """our predictions on other X-rays made exactly like the ones picked on validation (same box source, conf_pow)"""
    pre = BOX_COLS.get(cfg['boxes'])
    return U.preds_from_teeth(rows, load_probs(out, name), conf_pow=cfg['conf_pow'],
                              adapter=adapter if cfg['boxes'] == 'median adapter' else None,
                              refine=load_boxes(out, name, pre) if pre else None)

def pool_boxes(weights, cache, tta, imgsz=None):
    """a YOLO's boxes on the pool (validator settings, score >= ST_MIN_CONF)"""
    if not (WORK / f'{cache}_validator.pkl').exists():
        ensure_pool()
    b, _ = disease_boxes(weights, {'pool': pool_keys()}, cache, augment=tta, meta=POOL_META, root=POOL_ROOT,
                         conf=CFG['ST_MIN_CONF'], imgsz=imgsz)
    return b

def pseudo_labels(tag, C_T, Y_T, cls_out, yolos, gt, tr, va, adapter, tta):
    """the teacher's labels for the pool, cached in WORK/pseudo_<tag>_<ST_TAG>.pkl:
       boxes = {pool key: [(d, x1, y1, x2, y2, score)]} above the per-disease thresholds (for the student YOLO)
       soft  = {pool crop id: 4 scores} (for the student classifier)
    C_T / Y_T: the teacher's classifier / YOLOs on validation ({val, test, cfg}); yolos: [(weights, cache, imgsz)].
    The teacher's make-up (number of models, TTA) is part of every file name made from its labels."""
    sig = f't{1 + len(Y_T)}' + ('tta' if tta else '')
    P = WORK / f'pseudo_{tag}_{ST_TAG}_{sig}.pkl'
    if P.exists():
        return pickle.loads(P.read_bytes())
    mc = CFG['ST_MIN_CONF']
    keep = lambda L: [p for p in L if p['score'] >= mc]
    # 1) the teacher on this split's validation X-rays: fusion settings, then the score threshold per disease
    parts_val = [keep(C_T['val'])] + [keep(y['val']) for y in Y_T]
    best = None
    for w, thr in kinds_grid(['c'] + ['d'] * len(Y_T)):
        fv = U.wbf(parts_val, weights=w, iou_thr=thr)
        ap = val_ap(gt, fv, va)
        if best is None or ap > best[0]:
            best = (ap, w, thr, fv)
    ap_t, w_t, iou_t, fv = best
    pooled = U.f1_threshold(gt, fv, va)
    thr = {}
    for d in range(4):
        t = U.f1_threshold(gt, fv, va, d)
        thr[d] = t if t['n_gt'] >= CFG['ST_MIN_GT'] else dict(pooled, fallback=True, n_gt_disease=t['n_gt'])
    # 2) the teacher on the pool
    V = CFG['CLS_TTA'] if tta else 1
    kp = {'pool': write_keys('pool', pool_keys())}
    cdir = WORK / 'cls' / f'{Path(cls_out).name}_pool_tta{V}'
    man = pool_manifest(make_crops=not cls_done(cdir, kp))     # crops only if the classifier must still score them
    rows = man.to_dict('records')
    rows_by_key = {}
    for r in rows:
        rows_by_key.setdefault(r['image_key'], []).append(r)
    cdir = cls_predict(cls_out, 'single', kp, cdir, V, manifest=POOL_CROPS / 'manifest.csv')
    c_pool = keep(cls_preds_on(rows, cdir, 'pool', C_T['cfg'], adapter))
    teeth = pool_teeth()
    y_pool = [det_preds(pool_boxes(wy, f'{cache}_pool{"_tta" if tta else ""}', tta, sz), pool_keys(), teeth)
              for wy, cache, sz in yolos]
    fused = U.wbf([c_pool] + y_pool, weights=w_t, iou_thr=iou_t)
    boxes = {}
    for p in fused:
        if p['score'] >= thr[p['d']]['thr']:
            boxes.setdefault(p['image_key'], []).append((p['d'], *p['box'], p['score']))
    # 3) soft labels per tooth: the teacher classifier's probability and the YOLOs' best box on that tooth
    p_cls = load_probs(cdir, 'pool')
    ys = [U.tooth_scores(yp, rows_by_key) for yp in y_pool]
    a = CFG['ST_SOFT_CLS']
    soft = {c: a * p + (1 - a) * np.mean([y.get(c, np.zeros(4)) for y in ys], 0) for c, p in p_cls.items()}
    # report
    n_pool = len(pool_keys())
    trs = set(tr)
    lab = [a_['d'] for r in gt if r['key'] in trs for a_ in r['anns'] if 'd' in a_]
    cnt_lab = np.bincount(lab, minlength=4) / max(1, len(tr))
    cnt_pool = np.bincount([b[0] for v in boxes.values() for b in v], minlength=4) / max(1, n_pool)
    S = np.array(list(soft.values())) if soft else np.zeros((0, 4))
    lab_rate = MAN[MAN.image_key.isin(trs) & (MAN.gt_only == 0)][U.LABEL_COLS].values.mean(0)
    rep = pd.DataFrame(dict(
        threshold=[round(thr[d]['thr'], 3) for d in range(4)],
        threshold_from=['all diseases pooled' if thr[d].get('fallback') else 'this disease' for d in range(4)],
        val_boxes=[thr[d].get('n_gt_disease', thr[d]['n_gt']) for d in range(4)],
        val_F1=[round(thr[d]['f1'], 3) for d in range(4)], val_precision=[round(thr[d]['p'], 3) for d in range(4)],
        val_recall=[round(thr[d]['r'], 3) for d in range(4)],
        pseudo_boxes_per_xray=np.round(cnt_pool, 2), annotated_per_train_xray=np.round(cnt_lab, 2),
        mean_soft_label=np.round(S.mean(0), 3) if len(S) else np.nan,
        annotated_tooth_rate=np.round(lab_rate, 3)), index=U.DIS)
    print(f'\nTeacher for {tag}: our classifier + {len(Y_T)} YOLOs{" with TTA" if tta else ""}, fused WBF '
          f'{":".join(f"{x:g}" for x in w_t)} iou {iou_t} (validation AP {ap_t:.2f}). '
          f'Pseudo labels on {n_pool} X-rays without diagnosis labels:')
    show(rep)
    out = dict(boxes=boxes, soft=soft, thr=thr, fusion=dict(w=w_t, thr=iou_t, val_AP=ap_t), tta=tta, report=rep,
               sig=sig)
    P.write_bytes(pickle.dumps(out))
    return out

def student_yolo(tag, tr, va, pseudo, imgsz, model=None, suffix='', hires=False):
    """YOLO trained from COCO weights on the annotated X-rays of the split (listed ST_LAB_REPEAT times) and the
    pseudo-labelled pool. model / suffix: another architecture (YOLO26m) and its file-name suffix.
    hires: the X-rays come from the higher-resolution export (labels are relative, so they are the same)."""
    name = f'dis{tag}_st{imgsz}_{pseudo["sig"]}{suffix}'
    if (WORK / 'weights' / f'{name}.pt').exists():
        return WORK / 'weights' / f'{name}.pt'
    dm = ensure_dis_images_hi() if hires else ensure_dis_images()
    ensure_pool()
    pm = ensure_pool_hi() if hires else POOL_META
    root = SCR / ('yolo_st_hi' if hires else 'yolo_st') / tag
    shutil.rmtree(root, ignore_errors=True)
    (root / 'images').mkdir(parents=True)
    (root / 'labels').mkdir()
    lines = []
    for rep_i in range(CFG['ST_LAB_REPEAT']):
        for k in tr:
            safe = k.replace('/', '__')
            img = root / 'images' / f'L{rep_i}__{safe}.jpg'
            os.symlink(dm[k][3], img)
            shutil.copy(DIS_ROOT / 'labels' / 'all' / f'{safe}.txt', root / 'labels' / f'L{rep_i}__{safe}.txt')
            lines.append(str(img))
    n_box = 0
    for k in pool_keys():
        if k not in pm:
            continue
        safe = k.replace('/', '__')
        img = root / 'images' / f'P__{safe}.jpg'
        os.symlink(pm[k][3], img)
        W, H = pm[k][:2]
        ll = U.yolo_label_lines([b[:5] for b in pseudo['boxes'].get(k, [])], W, H)
        (root / 'labels' / f'P__{safe}.txt').write_text('\n'.join(ll))
        n_box += len(ll)
        lines.append(str(img))
    (root / 'train.txt').write_text('\n'.join(lines) + '\n')
    print(f'student YOLO data: {len(tr)} annotated X-rays x{CFG["ST_LAB_REPEAT"]} + {len(pool_keys())} '
          f'pseudo-labelled ({n_box} boxes), {imgsz} px')
    # the baseline turns mosaic off for its last 10 of 80 epochs and warms up for ~100 steps; an epoch here is ~8x
    # longer, so 1 warm-up epoch and mosaic off for the last 1/8 of the epochs
    cm = max(1, round(CFG['ST_YOLO_EPOCHS'] / 8))
    b = (CFG['DET_BATCH_XHI'] if imgsz > CFG['DET_IMGSZ_HI'] else
         CFG['DET_BATCH_HI'] if imgsz > CFG['DET_IMGSZ'] else CFG['DET_BATCH'])
    return train_disease_yolo(name, None, va, CFG['ST_YOLO_EPOCHS'], seed=0, train_txt=root / 'train.txt',
                              extra=f'close_mosaic={cm} warmup_epochs=1', imgsz=imgsz, model=model, batch=b,
                              hires=hires)

def finetune_yolo(w, tr, va, imgsz):
    """a trained student YOLO, trained a little more on the annotated X-rays only, at a low learning rate (the last
    step of a self-training round in ASTOD, Vandeghen et al. 2022, and in the SIIM-COVID-19 1st-place solution).
    The checkpoint with the best validation fitness is kept, as for every YOLO here."""
    return train_disease_yolo(f'{Path(w).stem}_ft', tr, va, CFG['FT_EPOCHS'], seed=0, model=w, imgsz=imgsz,
                              batch=CFG['DET_BATCH_HI'] if imgsz > CFG['DET_IMGSZ'] else CFG['DET_BATCH'],
                              extra=f"optimizer=AdamW lr0={CFG['FT_LR']} warmup_epochs=0 "
                                    f"close_mosaic={CFG['FT_CLOSE_MOSAIC']}")

def student_cls(tag, seed, tr, vak, tek, pseudo, stag=None):
    """crop classifier trained from ImageNet weights on the annotated teeth of the split (weight 1) and the pool's
    teeth with the teacher's soft labels (weight ST_PSEUDO_W)"""
    out = WORK / 'cls' / f'{stag or ST_TAG}_{tag}_single_st_{pseudo["sig"]}'
    preds = {'val': vak, 'test': tek}
    if cls_done(out, preds):
        return out
    pool = pool_manifest().copy()
    S = np.array([pseudo['soft'].get(int(c), np.zeros(4)) for c in pool.crop_id])
    for i, c in enumerate(U.LABEL_COLS):
        pool[c] = S[:, i]
    pool['w'] = CFG['ST_PSEUDO_W']
    ev = [k for f in (vak, tek) for k in Path(f).read_text().split()]
    lab = train_frame(tr, ev)      # v15: clean crops for the validation and test X-rays
    lab['w'] = 1.0
    mp = SCR / 'st' / f'{tag}_manifest.csv'
    mp.parent.mkdir(parents=True, exist_ok=True)
    pd.concat([lab, pool], ignore_index=True).to_csv(mp, index=False)
    trk_st = write_keys(f'{tag}_train_st', list(tr) + pool_keys())
    return run_cls('single', trk_st, vak, preds, out, seed=seed, extra=f"--pool-ratio {CFG['ST_POOL_RATIO']}",
                   manifest=mp)

def split_extra_rows(tag, seed, tr, va, te, vak, tek, gt_val, gt_test, adapter, picked, cls_dirs, w_y0, w_y1):
    """the 1280 YOLO, TTA rows and the self-training round for one split. picked = this split's {name: {val, test,
    cfg}} with at least 'plain YOLO' and 'crop only'; every new model is added to it. Returns the new table rows."""
    rows = []
    tta, V, T = CFG['TTA'], CFG['CLS_TTA'], TTA_SFX
    C, TW, HI = NAMES['single'], NAMES['full'], CFG['DET_IMGSZ_HI']

    def det(boxes, variant, kind):
        s, picked[variant] = det_row(boxes, gt_val, gt_test, va, te, f'{variant}, seed {seed}')
        s.update(seed=seed, variant=variant, kind=kind)
        rows.append(s)

    def cls(out, variant):
        rs, picked[variant] = model_rows(out, gt_val, gt_test, va, te, f'{variant}, seed {seed}', adapter)
        for r in rs:
            r.update(seed=seed, variant=variant, kind='ours')
        rows.extend(rs)

    def fuse(names, variant, kind='ours', kinds=None, per_class=False, grid=None):
        """grid: None = as in v11; a function of the kinds (grid_v12) = the v12 settings"""
        if kinds is not None:      # members that are missing (no control YOLO, no TwinArch) are left out
            names, kinds = [n for n in names if n in picked], [k for n, k in zip(names, kinds) if n in picked]
        if len(names) >= 2 and all(n in picked for n in names):
            g = grid(kinds) if callable(grid) else grid
            s, picked[variant] = fusion_row([picked[n] for n in names], gt_val, gt_test, va, te,
                                            f'{variant}, seed {seed}', kinds=kinds, per_class=per_class, grid=g)
            s.update(seed=seed, variant=variant, picked=True, kind=kind)
            rows.append(s)

    w_hi = None
    if CFG['SHARP_YOLO']:
        try:
            w_hi = train_disease_yolo(f'dis{tag}_r1280', tr, va, CFG['DIS_EPOCHS'], imgsz=HI, batch=CFG['DET_BATCH_HI'])
        except Exception as e:     # the rest (TTA, self-training) goes on with the 1024 YOLOs
            ST_FAILED.append(f'{tag} 1280 YOLO')
            print(f'!! the 1280 YOLO of {tag} did not train ({str(e)[:200]}); going on without it. Send Claude the log.')
    if w_hi is not None:
        det(disease_boxes(w_hi, {'val': va, 'test': te}, f'disbox_{tag}_r1280', imgsz=HI)[0], YOLO_HI, 'control')
        fuse(['plain YOLO', YOLO_S1, YOLO_HI], YOLO_TRIO, 'control', kinds=['d', 'd', 'd'])
        fuse([C, 'plain YOLO', YOLO_S1, YOLO_HI], C_TRIO, kinds=['c', 'd', 'd', 'd'])
    if tta:
        det(disease_boxes(w_y0, {'val': va, 'test': te}, f'disbox_{tag}_tta', augment=True)[0], 'plain YOLO' + T, 'baseline')
        if w_y1 is not None:
            det(disease_boxes(w_y1, {'val': va, 'test': te}, f'disbox_{tag}_s1_tta', augment=True)[0], YOLO_S1 + T, 'control')
            fuse(['plain YOLO' + T, YOLO_S1 + T], YOLO_PAIR + T, 'control')
        if w_hi is not None:
            det(disease_boxes(w_hi, {'val': va, 'test': te}, f'disbox_{tag}_r1280_tta', augment=True, imgsz=HI)[0],
                YOLO_HI + T, 'control')
            fuse(['plain YOLO' + T, YOLO_S1 + T, YOLO_HI + T], YOLO_TRIO + T, 'control', kinds=['d', 'd', 'd'])
        cls(evalc(cls_dirs['single'], 'single', {'val': vak, 'test': tek}, V), C + T)
        if 'full' in cls_dirs:
            cls(evalc(cls_dirs['full'], 'full', {'val': vak, 'test': tek}, V), TW + T)
        fuse([C + T, 'plain YOLO' + T], C + FUSED + T)
        fuse([C + T, 'plain YOLO' + T, YOLO_S1 + T], C + FUSED_BOTH + T)
        fuse([C + T, 'plain YOLO' + T, YOLO_S1 + T, YOLO_HI + T], C_TRIO + T, kinds=['c', 'd', 'd', 'd'])
    if SELF_TRAIN:
        try:
            yolos = [(w, cache, sz, nm) for w, cache, sz, nm in (
                (w_y0, f'disbox_{tag}', None, 'plain YOLO'), (w_y1, f'disbox_{tag}_s1', None, YOLO_S1),
                (w_hi, f'disbox_{tag}_r1280', HI, YOLO_HI)) if w is not None]
            r1 = self_train_rows(tag, seed, tr, va, te, vak, tek, gt_val, adapter, picked, cls_dirs, yolos, det, cls, fuse)
        except Exception as e:      # keep the TTA rows and everything else of this split
            import traceback
            traceback.print_exc()
            ST_FAILED.append(tag)
            r1 = None
            print(f'!! self-training on {tag} failed ({str(e)[:300]}). The rest of the run goes on; send Claude the log.')
        if r1 is not None and CFG.get('V12'):
            try:
                rows.extend(v12_rows(tag, seed, tr, va, te, vak, tek, gt_val, gt_test, adapter, picked, cls_dirs,
                                     yolos, r1, det, cls, fuse))
            except Exception as e:  # keep the v11 rows of this split
                import traceback
                traceback.print_exc()
                ST_FAILED.append(f'{tag} v12')
                print(f'!! the v12 part of {tag} failed ({str(e)[:300]}). The rest of the run goes on; send Claude the log.')
    return rows

def self_train_rows(tag, seed, tr, va, te, vak, tek, gt_val, adapter, picked, cls_dirs, yolos, det, cls, fuse):
    """teacher -> pseudo labels -> student YOLO and student classifier for one split, plus the fused rows"""
    tta, V, C, TW, T = CFG['TTA'], CFG['CLS_TTA'], NAMES['single'], NAMES['full'], TTA_SFX
    S = T if tta else ''
    sz = CFG['DET_IMGSZ_HI'] if any(s == CFG['DET_IMGSZ_HI'] for _, _, s, _ in yolos) else CFG['DET_IMGSZ']
    pseudo = pseudo_labels(tag, picked[C + S], [picked[nm + S] for *_, nm in yolos], cls_dirs['single'],
                           [(w, cache, s) for w, cache, s, _ in yolos], gt_val, tr, va, adapter, tta)
    w_sy = student_yolo(tag, tr, va, pseudo, sz)
    det(disease_boxes(w_sy, {'val': va, 'test': te}, f'disbox_{Path(w_sy).stem}', imgsz=sz)[0], ST_YOLO, 'ours')
    sc = student_cls(tag, seed, tr, vak, tek, pseudo)
    if sc is not None:
        cls(evalc(sc, 'single', {'val': vak, 'test': tek}), ST_CLS)
    if tta:
        det(disease_boxes(w_sy, {'val': va, 'test': te}, f'disbox_{Path(w_sy).stem}_tta', augment=True, imgsz=sz)[0],
            ST_YOLO + T, 'ours')
        if sc is not None:
            cls(evalc(sc, 'single', {'val': vak, 'test': tek}, V), ST_CLS + T)
    for x in ([''] + ([T] if tta else [])):
        fuse([ST_CLS + x, ST_YOLO + x], ST_PAIR + x)
        fuse(['plain YOLO' + x, YOLO_S1 + x, YOLO_HI + x, ST_YOLO + x], DET_ALL + x, 'control',
             kinds=['d', 'd', 'd', 'ds'])
        members = [C + x, TW + x, ST_CLS + x, 'plain YOLO' + x, YOLO_S1 + x, YOLO_HI + x, ST_YOLO + x]
        kinds = ['c', 'c?', 'cs', 'd', 'd', 'd', 'ds']     # TwinArch may be left out (validation decides)
        fuse(members, OURS_ALL + x, kinds=kinds)
        fuse(members, OURS_ALL_PC + x, kinds=kinds, per_class=True)
    return dict(pseudo=pseudo, w_sy=w_sy, sc=sc, sz=sz)

ST2_TAG = 'v12'   # round-2 files (pseudo labels, students) carry this tag
ST3_TAG = 'v14'   # round-3 files

def pool_cls_dir(out, V):
    """where a classifier's scores on the pool are kept"""
    return WORK / 'cls' / f'{Path(out).name}_pool_tta{V}'

def pseudo_report(thr, boxes, soft, gt, tr):
    """the pseudo labels next to the annotated ones: boxes per X-ray and tooth rates per disease"""
    n_pool = len(pool_keys())
    trs = set(tr)
    lab = [a_['d'] for r in gt if r['key'] in trs for a_ in r['anns'] if 'd' in a_]
    cnt_lab = np.bincount(lab, minlength=4) / max(1, len(tr))
    cnt_pool = np.bincount([b[0] for v in boxes.values() for b in v], minlength=4) / max(1, n_pool)
    S = np.array(list(soft.values())) if soft else np.zeros((0, 4))
    lab_rate = MAN[MAN.image_key.isin(trs) & (MAN.gt_only == 0)][U.LABEL_COLS].values.mean(0)
    return pd.DataFrame(dict(
        threshold=[round(thr[d]['thr'], 3) for d in range(4)],
        threshold_from=['all diseases pooled' if thr[d].get('fallback') else 'this disease' for d in range(4)],
        val_boxes=[thr[d].get('n_gt_disease', thr[d]['n_gt']) for d in range(4)],
        val_F1=[round(thr[d]['f1'], 3) for d in range(4)], val_precision=[round(thr[d]['p'], 3) for d in range(4)],
        val_recall=[round(thr[d]['r'], 3) for d in range(4)],
        pseudo_boxes_per_xray=np.round(cnt_pool, 2), annotated_per_train_xray=np.round(cnt_lab, 2),
        mean_soft_label=np.round(S.mean(0), 3) if len(S) else np.nan,
        annotated_tooth_rate=np.round(lab_rate, 3)), index=U.DIS)

def pseudo_labels_r2(tag, names, kinds, src, picked, gt, tr, va, adapter, tta, rnd=2):
    """round 2 (or 3): the teacher is the previous round's fusion of our classifiers and the YOLOs (detector boxes
    allowed), re-picked on validation with the pool's score floor (ST_MIN_CONF), then run on the pool. Thresholds
    and soft labels are made as in round 1. Cached in WORK/pseudo_<tag>_<v12 | v14>_<sig>.pkl.
    src: {member: dict(kind='cls', out=classifier folder) or dict(kind='det', w=weights, cache=name, imgsz=size)}"""
    sig = f'r{rnd}t{len(names)}' + ('tta' if tta else '')
    P = WORK / f'pseudo_{tag}_{ST2_TAG if rnd == 2 else ST3_TAG}_{sig}.pkl'
    if P.exists():
        return pickle.loads(P.read_bytes())
    mc = CFG['ST_MIN_CONF']
    keep = lambda L: [p for p in L if p['score'] >= mc]
    members = [dict(val=keep(picked[n]['val'])) for n in names]
    grid = grid_v12(kinds)
    variants = {False: members}
    if any(g[2] for g in grid):
        variants[True] = with_detector_boxes(members, kinds, parts=('val',))
    best = None
    for w, thr_, tp in grid:
        fv = wbf_members(variants[tp], w, thr_, 'val')
        ap = val_ap(gt, fv, va)
        if best is None or ap > best[0]:
            best = (ap, w, thr_, tp, fv)
    ap_t, w_t, iou_t, tp_t, fv = best
    pooled = U.f1_threshold(gt, fv, va)
    thr = {}
    for d in range(4):
        t = U.f1_threshold(gt, fv, va, d)
        thr[d] = t if t['n_gt'] >= CFG['ST_MIN_GT'] else dict(pooled, fallback=True, n_gt_disease=t['n_gt'])
    # the teacher on the pool
    V = CFG['CLS_TTA'] if tta else 1
    kp = {'pool': write_keys('pool', pool_keys())}
    cls_names = [n for n, k in zip(names, kinds) if k[0] == 'c']
    man = pool_manifest(make_crops=not all(cls_done(pool_cls_dir(src[n]['out'], V), kp) for n in cls_names))
    rows = man.to_dict('records')
    rows_by_key = {}
    for r in rows:
        rows_by_key.setdefault(r['image_key'], []).append(r)
    teeth = pool_teeth()
    parts, probs = [], []
    for n, k in zip(names, kinds):
        sr = src[n]
        if k[0] == 'c':
            cdir = cls_predict(sr['out'], 'single', kp, pool_cls_dir(sr['out'], V), V,
                               manifest=POOL_CROPS / 'manifest.csv')
            parts.append(keep(cls_preds_on(rows, cdir, 'pool', picked[n]['cfg'], adapter)))
            probs.append(load_probs(cdir, 'pool'))
        else:
            parts.append(det_preds(pool_boxes(sr['w'], f"{sr['cache']}_pool{'_tta' if tta else ''}", tta, sr['imgsz']),
                                   pool_keys(), teeth))
    pm = [dict(pool=x) for x in parts]
    if tp_t:
        pm = with_detector_boxes(pm, kinds, parts=('pool',), rows_by_key=rows_by_key)
    fused = wbf_members(pm, w_t, iou_t, 'pool')
    boxes = {}
    for p in fused:
        if p['score'] >= thr[p['d']]['thr']:
            boxes.setdefault(p['image_key'], []).append((p['d'], *p['box'], p['score']))
    # soft labels per tooth: the classifiers' mean probability and the YOLOs' mean best box on that tooth
    ys = [U.tooth_scores(x, rows_by_key) for x, k in zip(parts, kinds) if k[0] == 'd']
    a = CFG['ST_SOFT_CLS']
    soft = {c: a * np.mean([pr[c] for pr in probs if c in pr], 0)
               + (1 - a) * np.mean([y.get(c, np.zeros(4)) for y in ys], 0) for c in probs[0]}
    rep = pseudo_report(thr, boxes, soft, gt, tr)
    print(f'\nRound-{rnd} teacher for {tag}: [{"; ".join(names)}], fused WBF {":".join(f"{x:g}" for x in w_t)} iou {iou_t}'
          f'{", detector boxes" if tp_t else ""} (validation AP {ap_t:.2f} on boxes >= {mc}). Pseudo labels on '
          f'{len(pool_keys())} X-rays without diagnosis labels:')
    show(rep)
    out = dict(boxes=boxes, soft=soft, thr=thr, fusion=dict(w=w_t, thr=iou_t, tp=tp_t, val_AP=ap_t), tta=tta,
               report=rep, sig=sig, members=names)
    P.write_bytes(pickle.dumps(out))
    return out

def v12_rows(tag, seed, tr, va, te, vak, tek, gt_val, gt_test, adapter, picked, cls_dirs, yolos, r1, det, cls, fuse):
    """v12-v14 on one split, every step cached in WORK:
     1. fusions in which our classifiers may sit on the detectors' boxes (grid_v12): with the v11 members, and with
        the round-1 members (this one, with TTA, is the round-2 teacher), each next to its detector-only control
     2. round 2: the teacher labels the pool again and YOLO students learn from it (YOLO11m and YOLO26m at 1280,
        from COCO weights); their fusion with our classifiers is the round-3 teacher
     3. round 3 (v14): that teacher labels the pool again and a YOLO26m student learns at a higher resolution
        (1536, from a second, sharper export of the X-rays): Noisy Student also grew its students round by round
     4. the final fusion of our classifiers and all the YOLOs (the plain, not self-trained ones only if validation
        wants them), next to the same YOLOs fused without our classifiers
    The v12 trial also fine-tuned every student on the annotated X-rays and trained a round-2 classifier student. On
    split 0 both were worse than without them (-0.6 to -1.8 AP for the four fine-tunes, -2.0 AP for the classifier),
    so they are off (FINETUNE_STUDENTS, ST2_CLS); where they exist (split 0) their rows are still reported, but they
    never enter a fusion.
    Returns the headline row: of the round-1, round-2 and final fusions, with and without TTA, the one with the best
    validation AP (if a later round does not help on a split, validation can say so)."""
    tta, V, T = CFG['TTA'], CFG['CLS_TTA'], TTA_SFX
    C, TW = NAMES['single'], NAMES['full']
    sz, szx = r1['sz'], CFG['DET_IMGSZ_XHI']
    modes = [''] + ([T] if tta else [])
    xt = T if tta else ''
    FT = bool(CFG.get('FINETUNE_STUDENTS'))

    def det_both(w, name, imgsz, hires=False):
        stem = Path(w).stem
        for x, aug in (('', False), (T, True)) if tta else (('', False),):
            cache = f'disbox_{stem}' + ('_tta' if aug else '')
            kw = {}
            if hires and not (WORK / f'{cache}_validator.pkl').exists():
                kw = dict(meta=ensure_dis_images_hi(), root=DIS_ROOT_HI)
            det(disease_boxes(w, {'val': va, 'test': te}, cache, augment=aug, imgsz=imgsz, **kw)[0], name + x, 'ours')

    def ft_rows(w, name):
        """the student fine-tuned on the annotated X-rays: trained only with FINETUNE_STUDENTS, otherwise reported
        where an earlier version made it (cached boxes, no GPU work)"""
        wf = WORK / 'weights' / f'{Path(w).stem}_ft.pt'
        if FT:
            wf = finetune_yolo(w, tr, va, sz)
        if Path(wf).exists():
            det_both(wf, name, sz)
        return wf

    def better(a, b):
        """of a fine-tuned student (a) and the student itself (b): the one with the better validation AP, or always
        the student itself when fine-tunes are off"""
        if not FT or a not in picked or b not in picked:
            return b if (b in picked or not FT) else a
        return a if val_ap(gt_val, picked[a]['val'], va) > val_ap(gt_val, picked[b]['val'], va) else b

    def students_on(names, kinds, x):
        """the members of a fusion that exist on this split, with their kinds"""
        keep = [(n, k) for n, k in zip(names, kinds) if n in picked]
        return [n for n, _ in keep], [k for _, k in keep]

    w_ft = ft_rows(r1['w_sy'], ST_FT)
    st1 = {x: better(ST_FT + x, ST_YOLO + x) for x in modes}
    # where each member's scores on the pool come from (for the round-2 and round-3 teachers)
    src = {}
    def add_src(name, **kw):
        for x in modes:
            src[name + x] = kw
    add_src(C, kind='cls', out=cls_dirs['single'])
    add_src(ST_CLS, kind='cls', out=r1['sc'])
    for w, cache, s_, nm in yolos:
        add_src(nm, kind='det', w=w, cache=cache, imgsz=s_)
    add_src(ST_YOLO, kind='det', w=r1['w_sy'], cache=f"disbox_{Path(r1['w_sy']).stem}", imgsz=sz)
    add_src(ST_FT, kind='det', w=w_ft, cache=f'disbox_{Path(w_ft).stem}', imgsz=sz)
    # 1. fusions with detector boxes allowed
    yn = [nm for *_, nm in yolos]          # this split's plain YOLOs (seed 0, seed 1, 1280)
    for x in modes:
        fuse([C + x, TW + x, ST_CLS + x] + [n + x for n in yn] + [ST_YOLO + x], OURS_ALL_DB + x,
             kinds=['c', 'c?', 'cs'] + ['d'] * len(yn) + ['ds'], grid=grid_v12)
        r1m, r1k = [C + x, ST_CLS + x] + [n + x for n in yn] + [st1[x]], ['c', 'cs'] + ['d'] * len(yn) + ['ds']
        fuse(r1m, R1_OURS + x, kinds=r1k, grid=grid_v12)
        fuse(r1m[2:], R1_DET + x, 'control', kinds=r1k[2:], grid=grid_v12)
    # 2. round 2, taught by the round-1 fusion (with TTA when TTA is on)
    names, kinds = students_on([C + xt, ST_CLS + xt] + [n + xt for n in yn] + [st1[xt]],
                               ['c', 'cs'] + ['d'] * len(yn) + ['ds'], xt)
    pseudo2 = pseudo_labels_r2(tag, names, kinds, src, picked, gt_val, tr, va, adapter, tta, rnd=2)
    r2 = []                                 # (name, weights) of the round-2 YOLO students
    if CFG.get('ST2_YOLO11', True):
        w2 = student_yolo(tag, tr, va, pseudo2, sz)
        det_both(w2, ST2_YOLO, sz)
        ft_rows(w2, ST2_FT)
        r2.append((ST2_YOLO, w2))           # fine-tuned students never enter a fusion (see above)
    for model, suffix, name, ftname in ((CFG.get('ST2_YOLO26'), '_y26', ST2_Y26, ST2_Y26_FT),
                                        (CFG.get('ST2_YOLO26L'), '_y26l', ST2_Y26L, None)):
        if not model:
            continue
        try:
            w_ = student_yolo(tag, tr, va, pseudo2, sz, model=model, suffix=suffix)
            det_both(w_, name, sz)
            if ftname:
                ft_rows(w_, ftname)
            r2.append((name, w_))
        except Exception as e:
            import traceback
            traceback.print_exc()
            ST_FAILED.append(f'{tag} {model}')
            print(f'!! the {model} student of {tag} failed ({str(e)[:200]}); going on without it. Send Claude the log.')
    for name, w_ in r2:
        add_src(name, kind='det', w=w_, cache=f'disbox_{Path(w_).stem}', imgsz=sz)
    # the round-2 classifier student: trained only with ST2_CLS, otherwise reported where it exists
    out2 = WORK / 'cls' / f'{ST2_TAG}_{tag}_single_st_{pseudo2["sig"]}'
    sc2 = (student_cls(tag, seed, tr, vak, tek, pseudo2, stag=ST2_TAG)
           if CFG.get('ST2_CLS') or cls_done(out2, {'val': vak, 'test': tek}) else None)
    if sc2 is not None:
        cls(evalc(sc2, 'single', {'val': vak, 'test': tek}), ST2_CLS)
        if tta:
            cls(evalc(sc2, 'single', {'val': vak, 'test': tek}, V), ST2_CLS + T)
    c2 = [ST2_CLS] if CFG.get('ST2_CLS') else []
    # the round-2 fusion (the round-3 teacher) and its control. The plain YOLOs are optional: the self-trained ones
    # are 4-9 AP stronger and may only dilute the fusion (on split 0 the six-YOLO fusion with TTA was below its best
    # member); validation decides
    def fusion_members(x, extra=()):
        studs = [st1[x]] + [n + x for n, _ in r2] + [n + x for n in extra]
        fm = [C + x, ST_CLS + x] + [c + x for c in c2] + [n + x for n in yn] + studs
        fk = ['c', 'cs'] + ['cs'] * len(c2) + ['d?'] * len(yn) + ['ds'] * len(studs)
        return fm, fk, 2 + len(c2)
    for x in modes:
        fm, fk, k = fusion_members(x)
        fuse(fm, R2_OURS + x, kinds=fk, grid=grid_v12)
        fuse(fm[k:], R2_DET + x, 'control', kinds=fk[k:], grid=grid_v12)
    # 3. round 3: a YOLO26m student at a higher resolution, taught by the round-2 fusion (with TTA)
    r3 = []
    if CFG.get('ST3') and R2_OURS + xt in picked:
        try:
            fm, fk, _ = fusion_members(xt)
            names, kinds = students_on(fm, fk, xt)
            pseudo3 = pseudo_labels_r2(tag, names, kinds, src, picked, gt_val, tr, va, adapter, tta, rnd=3)
            w3 = student_yolo(tag, tr, va, pseudo3, szx, model=CFG['ST3_MODEL'], suffix='_hi', hires=True)
            det_both(w3, ST3_YOLO, szx, hires=True)
            r3.append(ST3_YOLO)
        except Exception as e:
            import traceback
            traceback.print_exc()
            ST_FAILED.append(f'{tag} round 3')
            print(f'!! round 3 of {tag} failed ({str(e)[:200]}); going on without it. Send Claude the log.')
    # 4. the final fusion and its detector-only control
    if r3:
        for x in modes:
            fm, fk, k = fusion_members(x, r3)
            fuse(fm, OURS_V12 + x, kinds=fk, grid=grid_v12)
            fuse(fm[k:], DET_V12 + x, 'control', kinds=fk[k:], grid=grid_v12)
    cand = list(dict.fromkeys(n for b in (OURS_V12, R2_OURS, R1_OURS) for n in (b, b + xt) if n in picked))
    if not cand:
        return []
    vap = {n: val_ap(gt_val, picked[n]['val'], va) for n in cand}
    ch = max(cand, key=lambda n: vap[n])
    picked[OURS_V12_BEST] = picked[ch]
    s = score_row(gt_test, picked[ch]['test'], te, f'{OURS_V12_BEST}, seed {seed}')
    rnd = 'round 3' if ch.startswith(OURS_V12) else 'round 2' if ch.startswith(R2_OURS) else 'round 1'
    s.update(seed=seed, variant=OURS_V12_BEST, picked=True, kind='ours', boxes='fused', val_diag_AP=round(vap[ch], 2),
             chosen=rnd + (', with TTA' if ch.endswith(T) else ', no TTA'))
    return [s]

def extra_tables(MAIN, diffM, where):
    """the v11 tables: every model against its control, and AP without / with TTA side by side"""
    if diffM:
        D = pd.DataFrame(diffM)
        D.to_csv(WORK / f'bootstrap_{where}_matched_{RUN_TAG}.csv', index=False)
        print('\nEach model against the control it has to beat, on the same test X-rays (ours vs the same detectors '
              'without our classifiers; a self-trained model vs the same model trained without the pool; TTA vs no TTA):')
        show(D[[c for c in ('seed', 'model', 'control', 'AP', 'dAP', 'dAP_ci', 'AP50', 'dAP50', 'dAP50_ci', 'p_better')
                if c in D]])
        if 'seed' in D and D.seed.nunique() > 1:
            g = D.groupby(['model', 'control'], sort=False)
            print('\nOver all splits:')
            show(pd.DataFrame({'splits': g.size(), 'AP minus control, mean': g.dAP.mean().round(2),
                               'lowest': g.dAP.min(), 'highest': g.dAP.max(), 'significant better / worse':
                               [f'{(x.dAP_lo > 0).sum()} / {(x.dAP_hi < 0).sum()}' for _, x in g]}))
    C = NAMES['single']
    order = ['plain YOLO', YOLO_S1, YOLO_HI, YOLO_PAIR, YOLO_TRIO, C, NAMES['full'], C + FUSED, C + FUSED_BOTH, C_TRIO,
             ST_YOLO, ST_CLS, ST_PAIR, DET_ALL, OURS_ALL, OURS_ALL_PC]
    if CFG.get('V12'):
        order += [OURS_ALL_DB, ST_FT, R1_DET, R1_OURS, ST2_YOLO, ST2_FT, ST2_Y26, ST2_Y26_FT, ST2_Y26L, ST2_CLS, R2_DET,
                  R2_OURS, ST3_YOLO, DET_V12, OURS_V12]
    multi = where == 'Protocol B' and MAIN.seed.nunique() > 1
    if where == 'Protocol A':   # protocol A has 3 classifier seeds; the v11 rows use seed 0, so compare like with like
        MAIN = MAIN[(MAIN.kind != 'ours') | (MAIN.seed.astype(str) == '0')]
    g = MAIN.groupby('variant')[['diag_AP', 'diag_AP50', 'val_diag_AP']].mean()
    tab = []
    for v in order:
        r = dict(model=v)
        for sfx, lab in (('', 'no TTA'), (TTA_SFX, 'TTA')):
            if v + sfx in g.index:
                r[f'{lab} AP'], r[f'{lab} AP50'], r[f'{lab} val AP'] = g.loc[v + sfx].round(2).tolist()
        if len(r) > 1:
            tab.append(r)
    if tab:
        print(f'\n{where}: test AP / AP50 without and with test-time augmentation' +
              (', mean over the splits done' if multi else '') +
              f'. The teacher is "{C_TRIO}" with TTA, the self-trained models are its students' +
              (f' (round 2: "{R1_OURS}" with TTA teaches the round-2 students; round 3: "{R2_OURS}" with TTA '
               'teaches the round-3 student)' if CFG.get('V12') else '') +
              '; "(control)" rows have no classifier of ours:')
        show(pd.DataFrame(tab))
    if CFG.get('V12') and (MAIN.variant == OURS_V12_BEST).any():
        cols = [c for c in ('diag_AP', 'diag_AP50', 'diag_AP75', 'ul_old_AP', 'ul_old_AP50', 'val_diag_AP') if c in MAIN]
        sel = ['plain YOLO', OURS_ALL, DET_V12, OURS_V12, OURS_V12 + TTA_SFX, OURS_V12_BEST]
        groups = [('', MAIN)]
        if 'split_group' in MAIN and MAIN.split_group.nunique() > 1:   # v15: the fresh splits on their own
            groups = [(f' ({g} splits {sorted(M_.seed.unique().tolist())})', M_)
                      for g, M_ in MAIN.groupby('split_group', sort=False)] + [(' (all splits)', MAIN)]
        for gl, M_ in groups:
            mg = where == 'Protocol B' and M_.seed.nunique() > 1
            H = M_[M_.variant.isin(sel)].groupby('variant', sort=False)[cols]
            H = (H.agg(['mean', 'std']) if mg else H.mean()).round(2)
            H = H.reindex([v for v in sel if v in H.index])
            print(f'\n{where}, HEADLINE{gl}: the last row is the method (round 1, 2 or 3, with or without TTA, picked '
                  'on each split\'s validation X-rays); above it the plain YOLO, v11, and fixed choices for reference '
                  '(the final fusion with and without TTA, and its detector-only control)'
                  + (', mean and std over the splits' if mg else '') + '. ul_old = scored the way the YOLO papers score:')
            show(H)

def compare_versions(MAIN, old_csv, where):
    """v15 next to v14 on the same splits: the same trained models, but the validation and test X-rays are now cropped
    by the clean tooth detector (and copies found by the audit are not scored). Rows both versions picked."""
    p = WORK / old_csv
    if not p.exists() or MAIN is None or not len(MAIN):
        return
    old = pd.read_csv(p)
    old = old[old.picked.astype(str).str.lower() == 'true']
    keep = ['plain YOLO', NAMES['single'], NAMES['single'] + FUSED_BOTH, C_TRIO, ST_YOLO, OURS_ALL, R1_OURS, R2_OURS,
            ST3_YOLO, DET_V12, OURS_V12, OURS_V12_BEST]
    keep = keep + [k + TTA_SFX for k in keep]
    cols = ['diag_AP', 'diag_AP50', 'diag_AP75']
    a = MAIN.assign(seed=MAIN.seed.astype(str))
    b = old.assign(seed=old.seed.astype(str))
    m = a[a.variant.isin(keep)][['variant', 'seed'] + cols].merge(b[['variant', 'seed'] + cols], on=['variant', 'seed'],
                                                                   suffixes=(' v15', ' v14'))
    if m.empty:
        return
    for c in cols:
        m[f'{c} change'] = (m[f'{c} v15'] - m[f'{c} v14']).round(2)
    rank = {v: i for i, v in enumerate(keep)}
    m = m.assign(_r=m.variant.map(rank)).sort_values(['_r', 'seed']).drop(columns='_r')
    print(f'\n{where}: v14 as reported vs v15 (the same trained models; validation and test X-rays now cropped by the '
          'clean tooth detector, copies not scored). The YOLO rows do not use the tooth detector for their boxes:')
    show(m[['variant', 'seed', 'diag_AP v14', 'diag_AP v15', 'diag_AP change', 'diag_AP50 change', 'diag_AP75 change']])
    if m.seed.nunique() > 1:
        g = m.groupby('variant', sort=False)[[f'{c} change' for c in cols]].mean().round(2)
        print('mean change over the seeds / splits:')
        show(g)


## 8. Protocol A: official DENTEX validation set
All settings (score knob and box source) are picked on the 70 held-out training X-rays; the 50 official validation X-rays are only scored.


In [ ]:
LEADERBOARD = pd.DataFrame([   # DENTEX 2023 official test set (Hamamci et al., arXiv 2305.19112 v2)
    dict(model='HierarchicalDet (DENTEX baseline)', diag_AP=37.6, diag_AP50=60.2, diag_AP75=44.0, diag_AR=0.691, enum_AP=30.5, quad_AP=43.2),
    dict(model='He et al. (challenge winner)', diag_AP=37.06, diag_AP50=54.31, diag_AP75=44.77, diag_AR=0.5921, enum_AP=35.35, quad_AP=47.45),
    dict(model='Mei et al. YOLOrtho', diag_AP=37.44, diag_AP50=56.72, diag_AP75=45.49, diag_AR=0.5972, enum_AP=33.47, quad_AP=46.59),
    dict(model='Choi et al. DETDet', diag_AP=36.48, diag_AP50=57.30, diag_AP75=42.84, diag_AR=0.7585, enum_AP=30.77, quad_AP=43.29),
])
NAMES = {'single': 'crop only', 'twin': 'crop + twin asymmetry', 'arch': 'crop + arch context',
         'full': 'TwinArch (twin + arch)'}
FUSED = ' + YOLO, fused'
FUSED_S1, FUSED_BOTH = ' + YOLO seed 1, fused', ' + both YOLOs, fused'   # ours with the control YOLO / with both
YOLO_S1, YOLO_PAIR = 'plain YOLO, seed 1 (control)', 'YOLO + YOLO seed 1, fused (control)'
MAIN_COLS = ['model', 'seed', 'boxes', 'diag_AP', 'diag_AP50', 'diag_AP75', 'diag_AR', 'enum_AP', 'quad_AP', 'val_diag_AP', 'chosen']

def out_dir(tag, prot, v, seed):
    return WORK / 'cls' / (f'{tag}_{prot}_{v}' if seed == 0 else f'{tag}_{prot}{seed}_{v}')

def adapter_table(adapter, counts):
    rows = [dict(disease=U.DIS[d], jaw='upper' if up else 'lower', pairs=counts.get((d, up), 0),
                 **dict(zip(['dx1', 'dy1', 'dx2', 'dy2'], np.round(m, 3)))) for (d, up), m in sorted(adapter.items())]
    return pd.DataFrame(rows)

def per_class(T, col, digits=1):
    return pd.DataFrame({v: pd.DataFrame(list(s)).mean() for v, s in T.groupby('model', sort=False)[col]}).T.round(digits)

if RUN_PROTOCOL_A:
    gt_split = {'val': SETS['dis_val']['images'] if HAS_TEST else SETS['dis_train']['images'],
                'test': SETS['dis_test']['images'] if HAS_TEST else SETS['dis_val']['images']}
    trk, vak, tek = write_keys('A_train', A_TRAIN), write_keys('A_val', A_VAL), write_keys('A_test', A_TEST)
    ADAPTER_A, counts_A = ({}, {})
    if CFG['BOX_ADAPTER']:
        ADAPTER_A, counts_A = U.fit_box_adapter(SETS['dis_train']['images'], TEETH, A_TRAIN)
        print('Box adapter, learned on the training X-rays only (edge offsets in tooth widths/heights):')
        show(adapter_table(ADAPTER_A, counts_A))
    with stage(f'classifiers_A_{CLS_TAG}'):
        man_A = train_manifest('A', A_TRAIN, A_VAL + A_TEST)
        outs = {(v, sd): run_cls(v, trk, vak, {'val': vak, 'test': tek}, out_dir(CLS_TAG, 'A', v, sd), seed=sd,
                                 manifest=man_A)
                for sd in CFG['SEEDS_A'] for v in CFG['VARIANTS_A']}
        outs = {k: o for k, o in outs.items() if o is not None}
    with stage('clean_preds_A'):   # v15: every classifier scores the clean crops of the validation and test X-rays
        outs_e = {k: evalc(o, k[0], {'val': vak, 'test': tek}) for k, o in outs.items()}
    with stage('eval_A'):
        resA, PICKED_A = [], {}
        teeth_tag = f"[{Path(CFG['TOOTH_MODEL']).stem} teeth]"
        BASE_NAME = f"plain {CFG['DIS_MODEL'].replace('.pt', '')} detector (baseline)"
        if DISBOX_A is not None:
            s, PICKED_A[BASE_NAME] = det_row(DISBOX_A, gt_split['val'], gt_split['test'], A_VAL, A_TEST, BASE_NAME)
            s.update(seed='-', kind='baseline')
            resA.append(s)
        if DISBOX_A2 is not None and BASE_NAME in PICKED_A:   # control: two plain detectors fused the same way
            s, PICKED_A[YOLO_S1] = det_row(DISBOX_A2, gt_split['val'], gt_split['test'], A_VAL, A_TEST, YOLO_S1)
            s.update(seed=1, variant=YOLO_S1, kind='control'); resA.append(s)
            s, PICKED_A[YOLO_PAIR] = fusion_row([PICKED_A[BASE_NAME], PICKED_A[YOLO_S1]], gt_split['val'],
                                                gt_split['test'], A_VAL, A_TEST, YOLO_PAIR)
            s.update(seed='0+1', variant=YOLO_PAIR, picked=True, kind='control'); resA.append(s)
        for (v, sd), out in outs_e.items():
            nm = NAMES.get(v, v)
            rows, PICKED_A[(v, sd)] = model_rows(out, gt_split['val'], gt_split['test'], A_VAL, A_TEST, f'{nm} {teeth_tag}', ADAPTER_A)
            for r in rows:
                r.update(seed=sd, variant=nm, kind='ours')
            resA += rows
            if CFG['FUSION'] and BASE_NAME in PICKED_A:
                s, PICKED_A[(v + FUSED, sd)] = fusion_row([PICKED_A[(v, sd)], PICKED_A[BASE_NAME]], gt_split['val'],
                                                          gt_split['test'], A_VAL, A_TEST, nm + FUSED)
                s.update(seed=sd, variant=nm + FUSED, picked=True, kind='ours')
                resA.append(s)
                if v == 'single' and YOLO_S1 in PICKED_A:   # is our classifier more than "a second detector"?
                    for suffix, others in ((FUSED_S1, [PICKED_A[YOLO_S1]]),
                                           (FUSED_BOTH, [PICKED_A[BASE_NAME], PICKED_A[YOLO_S1]])):
                        s, PICKED_A[(v + suffix, sd)] = fusion_row([PICKED_A[(v, sd)]] + others, gt_split['val'],
                                                                   gt_split['test'], A_VAL, A_TEST, nm + suffix)
                        s.update(seed=sd, variant=nm + suffix, picked=True, kind='ours')
                        resA.append(s)
    # v11 (cell 13b): the 1280 YOLO, TTA and self-training, with the seed-0 classifiers and both YOLOs
    PK_A, diffM_A = {}, []
    if (CFG['TTA'] or SELF_TRAIN or CFG['SHARP_YOLO']) and BASE_NAME in PICKED_A and ('single', 0) in outs:
        PK_A = {'plain YOLO': PICKED_A[BASE_NAME]}
        PK_A.update({n: PICKED_A[n] for n in (YOLO_S1, YOLO_PAIR) if n in PICKED_A})
        for v in ('single', 'full'):
            for suffix in ('', FUSED, FUSED_S1, FUSED_BOTH):
                if (v + suffix, 0) in PICKED_A:
                    PK_A[NAMES[v] + suffix] = PICKED_A[(v + suffix, 0)]
        first = set(PK_A)
        try:
            with stage(f'extra_A_{RUN_TAG}'):
                resA += split_extra_rows('A', 0, A_TRAIN, A_VAL, A_TEST, vak, tek, gt_split['val'], gt_split['test'],
                                         ADAPTER_A, PK_A, {v: outs[(v, 0)] for v in ('single', 'full') if (v, 0) in outs},
                                         DIS_W_A, DIS_W_A2)
        except Exception as e:
            import traceback
            traceback.print_exc()
            ST_FAILED.append('A')
            print(f'!! the v11 part of protocol A failed ({str(e)[:300]}). The rest of the run goes on; send Claude the log.')
        NEW_A = [k for k in PK_A if k not in first]     # the models v11 added, for the bootstraps
    with stage('eval_A_tables'):
        TAB_A = pd.DataFrame(resA)
        TAB_A.to_csv(WORK / f'results_protocol_A_{RUN_TAG}.csv', index=False)
        (WORK / f'results_protocol_A_{RUN_TAG}.json').write_text(json.dumps(resA, indent=1, default=str))
        MAIN_A = TAB_A[TAB_A.picked].copy().reset_index(drop=True)
        MAIN_A['model'] = [m if v == 'plain YOLO' else v for m, v in zip(MAIN_A.model, MAIN_A.variant)]

    if not FINAL:
        print(f'DEVELOPMENT run {RUN_TAG}: judge changes on val_diag_AP. Test numbers are for reference only.')
    print('\nProtocol A: ' + ('official DENTEX test set' if HAS_TEST else 'official 50-image DENTEX validation set'))
    print('DENTEX / COCO scores (pycocotools). Every setting, incl. the box source and the fusion, '
          'is picked on VALIDATION; the test set never picks anything:')
    show(MAIN_A[MAIN_COLS])
    print('\nThe same predictions scored the way ultralytics\' validator does (what the YOLO papers report). '
          'ul_old = the AP formula of ultralytics <= 8.4.36, which the 2024-2026 papers used:')
    show(MAIN_A[UL_COLS])
    if len(CFG['SEEDS_A']) > 1:
        m = MAIN_A[MAIN_A.kind == 'ours'].groupby('model', sort=False)[
            ['diag_AP', 'diag_AP50', 'diag_AP75', 'diag_AR', 'enum_AP', 'quad_AP', 'ul_old_AP', 'ul_old_AP50']].agg(['mean', 'std']).round(2)
        print(f"\nMean and std over seeds {CFG['SEEDS_A']}:"); show(m)
    ours = TAB_A[(TAB_A.kind == 'ours') & (TAB_A.boxes != 'fused')]
    if len(ours):
        print('\nBox ablation, every box source. Validation AP (this decides) | test AP:')
        pv = ours.pivot_table(index=['variant', 'seed'], columns='boxes', values='val_diag_AP', sort=False)
        pt = ours.pivot_table(index=['variant', 'seed'], columns='boxes', values='diag_AP', sort=False)
        show(pd.concat({'val AP': pv, 'test AP': pt}, axis=1).round(2))
    print(f"\nPer-class AP (mean over seeds {CFG['SEEDS_A']}):"); show(per_class(MAIN_A, 'per_class_AP'))
    print('Per-class best F1 at IoU 0.5 (mean over seeds):'); show(per_class(MAIN_A, 'best_F1_iou50', 3))
    print('Per-class AP50 the ultralytics way, old formula (as in the papers\' per-class tables):')
    show(per_class(MAIN_A, 'ul_old_per_class_AP50'))
    print('\nContext only, NOT comparable (hidden 2023 challenge test set whose 4-class labels were never released):')
    show(LEADERBOARD)
    compare_versions(MAIN_A, 'results_protocol_A_v14.csv', 'Protocol A')

    # paired bootstrap over the test X-rays: is a model really better than the plain detector?
    if DISBOX_A is not None and outs:
        with stage('bootstrap_A'):
            sd0 = CFG['SEEDS_A'][0]
            cmp = {BASE_NAME: PICKED_A[BASE_NAME]['test']}
            cmp.update({n: PICKED_A[n]['test'] for n in (YOLO_S1, YOLO_PAIR) if n in PICKED_A})
            for v in CFG['VARIANTS_A']:
                for suffix in ('', FUSED, FUSED_S1, FUSED_BOTH):
                    if (v + suffix if suffix else v, sd0) in PICKED_A:
                        cmp[NAMES.get(v, v) + suffix] = PICKED_A[(v + suffix if suffix else v, sd0)]['test']
            nb = 1000 if FINAL else 300
            if PK_A:
                cmp.update({k: PK_A[k]['test'] for k in NEW_A})
                diffM_A = matched_bootstrap(gt_split['test'], PK_A, A_TEST, nb)
            BOOT_A = bootstrap_table(gt_split['test'], cmp, A_TEST, ref=BASE_NAME, n_boot=nb)
            BOOT_A.to_csv(WORK / f'bootstrap_A_{RUN_TAG}.csv', index=False)
            vs_pair = {k: v for k, v in cmp.items() if k == YOLO_PAIR or k.endswith((FUSED, FUSED_S1, FUSED_BOTH))}
            BOOT_A2 = (bootstrap_table(gt_split['test'], vs_pair, A_TEST, ref=YOLO_PAIR, n_boot=nb)
                       if YOLO_PAIR in vs_pair and len(vs_pair) > 1 else None)
            if BOOT_A2 is not None:
                BOOT_A2.to_csv(WORK / f'bootstrap_A_vs_pair_{RUN_TAG}.csv', index=False)
        print(f'\nDifference to the plain detector, seed {sd0}, 95% paired bootstrap interval over the test X-rays '
              '(p_better = share of resamples where the model wins):')
        show_boot(BOOT_A)
        if BOOT_A2 is not None:
            print('\nDifference to the two-YOLO control (two detectors fused the same way). Positive = our classifier '
                  'adds more than a second YOLO does:')
            show_boot(BOOT_A2)
    if PK_A:
        extra_tables(MAIN_A, diffM_A, 'Protocol A')

    # the same detector scored four ways: shows how much of the gap to the papers is the metric
    if BASE_NAME in PICKED_A:
        MCHECK_A = metric_check(PRINTED_A, [
            ('held-out 70 (picked its checkpoint)', 'val', gt_split['val'], A_VAL, PICKED_A[BASE_NAME]['val']),
            ('official 50 (test)', 'test', gt_split['test'], A_TEST, PICKED_A[BASE_NAME]['test'])])
        print('\nThe plain detector, one set of boxes, four ways of scoring it (diagnosis mAP50-95 and mAP50):')
        show(MCHECK_A)

    # where the AP goes, measured on the VALIDATION X-rays (never the test set)
    if outs:
        val_recs = gt_split['val']
        first = outs_e.get(('single', CFG['SEEDS_A'][0]), list(outs_e.values())[0])
        pv0 = load_probs(first, 'val')
        learned = {lab: load_boxes(first, 'val', pre) for lab, pre in BOX_COLS.items()}
        learned = {k: v for k, v in learned.items() if v is not None}
        st = [dict(boxes='tooth boxes', **U.box_iou_stats(val_recs, TEETH, A_VAL))]
        if ADAPTER_A:
            st.append(dict(boxes='median adapter', **U.box_iou_stats(val_recs, TEETH, A_VAL, ADAPTER_A)))
        for lab, bv in learned.items():
            st.append(dict(boxes=lab, **U.refined_iou_stats(val_recs, ROWS_BY_KEY, A_VAL, bv)))
        print(f'\nIoU between our boxes and the disease boxes they match (validation X-rays, {first.name}):'); show(pd.DataFrame(st))
        orc = U.oracle_boxes(val_recs, TEETH, A_VAL)
        choices = [('tooth boxes', {}), ('median adapter', dict(adapter=ADAPTER_A))]
        choices += [(lab, dict(refine=bv)) for lab, bv in learned.items()]
        choices.append(('annotated boxes (oracle)', dict(oracle=orc)))
        an = []
        for lab, kw in choices:
            r = U.dentex_eval(val_recs, make_preds('tooth', A_VAL, pv0, conf_pow=0.5, **kw), A_VAL)['diagnosis']
            an.append(dict(boxes=lab, AP=round(r['AP'], 2), AP50=round(r['AP50'], 2), AP75=round(r['AP75'], 2)))
        print('Same scores, different boxes (validation; the oracle row is analysis only):'); show(pd.DataFrame(an))

    # efficiency per X-ray: our pipeline = tooth detector + classifier on ~28 teeth, vs the plain detector
    with stage('efficiency_A'):
        tooth_cost = yolo_cost(TOOTH_W, CFG['TOOTH_MODEL'])
        dis_cost = yolo_cost(DIS_W_A)
        sd0 = CFG['SEEDS_A'][0]
        effs = {v: efficiency(outs[(v, sd0)]) for v in CFG['VARIANTS_A'] if (v, sd0) in outs}
        add = lambda *xs: sum(xs) if all(x is not None for x in xs) else None
        K = ('params_M', 'GFLOPs', 'ms_per_xray')
        rows = [dict(component=f"tooth detector {CFG['TOOTH_MODEL']}", **{k: tooth_cost.get(k) for k in K})]
        for v, e in effs.items():
            c = dict(params_M=e.get('params_M'), GFLOPs=e.get('GFLOPs_per_xray'), ms_per_xray=e.get('ms_per_xray'))
            rows.append(dict(component=f'{NAMES.get(v, v)} classifier ({e.get("teeth", 28)} teeth)', **c))
            rows.append(dict(component=f'  = whole pipeline with {NAMES.get(v, v)}', **{k: add(tooth_cost.get(k), c[k]) for k in K}))
            if CFG['FUSION'] and dis_cost:
                rows.append(dict(component=f'  = {NAMES.get(v, v)}{FUSED}',
                                 **{k: add(tooth_cost.get(k), c[k], dis_cost.get(k)) for k in K}))
                if v == 'single' and DISBOX_A2 is not None:
                    rows.append(dict(component=f'  = {NAMES.get(v, v)}{FUSED_BOTH}',
                                     **{k: add(tooth_cost.get(k), c[k], dis_cost.get(k), dis_cost.get(k)) for k in K}))
        rows.append(dict(component=f"baseline {CFG['DIS_MODEL']} disease detector", **{k: dis_cost.get(k) for k in K}))
        if DISBOX_A2 is not None and dis_cost:
            rows.append(dict(component='  = two of them fused (control)', **{k: add(dis_cost.get(k), dis_cost.get(k)) for k in K}))
        w_hi = WORK / 'weights' / 'disA_r1280.pt'
        hi_cost = yolo_cost(w_hi, shape=(640, 1280)) if w_hi.exists() else {}
        if hi_cost:
            rows.append(dict(component=f"{CFG['DIS_MODEL']} at 1280x640 (the sharper YOLO, also the student's size)",
                             **{k: hi_cost.get(k) for k in K}))
            cs = effs.get('single', {})
            cs = dict(params_M=cs.get('params_M'), GFLOPs=cs.get('GFLOPs_per_xray'), ms_per_xray=cs.get('ms_per_xray'))
            if dis_cost and cs['params_M'] is not None:
                rows.append(dict(component=f'  = {OURS_ALL} (no TTA)', **{k: add(
                    tooth_cost.get(k), *[cs[k]] * 3, *[dis_cost.get(k)] * 2, *[hi_cost.get(k)] * 2) for k in K}))
                extra = []     # the round-2 students of other architectures (YOLO26m, YOLO26l)
                for suffix, key in (('_y26', 'ST2_YOLO26'), ('_y26l', 'ST2_YOLO26L')):
                    ws_ = sorted((WORK / 'weights').glob(f'disA_st*_r2t*{suffix}.pt'))
                    c_ = yolo_cost(ws_[0], shape=(640, 1280)) if ws_ else {}
                    if c_:
                        rows.append(dict(component=f"{CFG.get(key)} at 1280x640 (a round-2 student)",
                                         **{k: c_.get(k) for k in K}))
                        extra.append(c_)
                ws_ = sorted((WORK / 'weights').glob('disA_st*_r3t*_hi.pt'))
                c_ = yolo_cost(ws_[0], shape=(768, CFG['DET_IMGSZ_XHI'])) if ws_ else {}
                if c_:
                    rows.append(dict(component=f"{CFG['ST3_MODEL']} at {CFG['DET_IMGSZ_XHI']}x768 (the round-3 student)",
                                     **{k: c_.get(k) for k in K}))
                    extra.append(c_)
                if extra:     # upper bound: every member kept (2 classifiers, 3 plain YOLOs, all the students)
                    rows.append(dict(component=f'  = {OURS_V12} (no TTA, all members)', **{k: add(
                        tooth_cost.get(k), *[cs[k]] * 2, *[dis_cost.get(k)] * 2, *[hi_cost.get(k)] * 3,
                        *[c_.get(k) for c_ in extra]) for k in K}))
        EFF = pd.DataFrame(rows).round(2)
        EFF.to_csv(WORK / 'efficiency_A.csv', index=False)
    print('\nEfficiency per X-ray on one T4 (fp16, batch 1, detectors at 1024x512 unless marked, model time only; '
          'cropping the teeth adds a few ms on CPU; TTA costs about 2.1x for a YOLO and 5x for a classifier):')
    show(EFF)
    errs = {n: c['error'] for n, c in (('tooth detector', tooth_cost), ('disease detector', dis_cost)) if c.get('error')}
    if errs:
        print('cost measurement problems:', errs)


## 9. Protocol B: peer protocol (random 80:20 splits)
Plain YOLO is retrained on every split, so every difference is paired. Splits 0-2 are the development splits; v15 adds splits 3 and 4, made once with the method frozen, and reports them on their own.


In [ ]:
PEERS_B = pd.DataFrame([   # as reported in each paper (their own random splits of the 705 labelled X-rays)
    dict(model='Hoang et al. 2026, Odovtos (hierarchical YOLOv8, 80:20)', diag_AP=48.0, diag_AP50=68.7,
         scored_with='ultralytics validator', F1='Impacted .904, Caries .885, Periapical .568, Deep caries .585'),
    dict(model='Sadr et al. 2026, Oral Radiology (YOLOv11-TAM, 80:20, val split also used for early stopping)', diag_AP=44.7, diag_AP50=65.4,
         scored_with='ultralytics validator', F1='AP50 per class: Caries .584, Deep caries .678, Impacted .661, Periapical .687'),
    dict(model='Nassiri & Akhloufi 2025, Neural Comput Appl (YOLOv9e, 80:10:10)', diag_AP=44.7, diag_AP50=61.0,
         scored_with='ultralytics validator', F1=''),
    dict(model='DinoDental 2026, arXiv (DINOv3-L LoRA + Mask R-CNN, 80:20)', diag_AP=35.4, diag_AP50=55.7,
         scored_with='COCO-style (to check)', F1=''),
    dict(model='SSAD 2024, arXiv (Faster R-CNN ViTDet-B + SSAD, 80:20)', diag_AP=31.04, diag_AP50=48.01,
         scored_with='COCO-style (to check)', F1=''),
])

B_STOPPED = None   # the split protocol B stopped at to stay inside Kaggle's 12 h (None = all done)

def split_need_h(tag, seed=0, variants=()):
    """hours one split still needs on 2x T4: what is cached in WORK costs nothing (times measured in the v11 runs)"""
    W = WORK / 'weights'
    w = lambda n: (W / f'{n}.pt').exists()
    h = 0.15                                                     # rows and bootstraps
    if RUN_YOLO_BASELINE_B or tag == 'A':
        h += 0.5 * (not w(f'dis{tag}')) + 0.5 * (CFG['ENSEMBLE_CONTROL'] and not w(f'dis{tag}_s1'))
    h += 0.25 * sum(not cls_done(WORK / 'cls' / f'{CLS_TAG}_{tag}_{v}', ['val', 'test']) for v in variants)
    h += 0.8 * (CFG['SHARP_YOLO'] and not w(f'dis{tag}_r1280'))   # 1280 YOLO, 80 epochs
    h += 0.1 * CFG['TTA']
    if SELF_TRAIN:
        st = [q.stem for q in W.glob(f'dis{tag}_st*.pt')]
        has = lambda r2, y26, ft: any(('_r2t' in n) == r2 and ('_y26' in n) == y26 and n.endswith('_ft') == ft for n in st)
        h += 2.0 * (not has(False, False, False))    # round 1: pool + teacher (~0.6 h), YOLO student (~1 h), classifier (~0.4 h)
        if CFG.get('V12'):        # times of the v12 trial on split 0
            ft = CFG.get('FINETUNE_STUDENTS')
            h += 0.2 * (ft and not has(False, False, True))                       # fine-tune of the round-1 student
            h += 0.3 * (not any(WORK.glob(f'pseudo_{tag}_{ST2_TAG}_*.pkl')))      # round-2 teacher on the pool
            h += 0.7 * (CFG.get('ST2_YOLO11', True) and not has(True, False, False)) \
                 + 0.2 * (ft and not has(True, False, True))                       # round-2 YOLO11m
            if CFG.get('ST2_YOLO26'):
                y26 = [n for n in st if n.endswith('_y26') or n.endswith('_y26_ft')]
                h += 0.8 * (not any(n.endswith('_y26') for n in y26)) + 0.2 * (ft and not any(n.endswith('_y26_ft') for n in y26))
            if CFG.get('ST2_YOLO26L'):
                h += 1.0 * (not any(n.endswith('_y26l') for n in st))            # round-2 YOLO26l
            if CFG.get('ST3'):                                                    # round 3: teacher on the pool + student
                h += 1.4 * (not any(n.endswith('_hi') for n in st))
            h += 0.4 * (CFG.get('ST2_CLS', False) and not any((WORK / 'cls').glob(f'{ST2_TAG}_{tag}_single_st_r2t*/log.json')))
            h += 0.2                                                              # the bigger fusions
    if CFG.get('CLEAN_EVAL'):         # v15: every classifier scores the clean crops (~1 min each, with and without TTA)
        h += 0.2 * (not (WORK / 'cls' / f'{CLS_TAG}_{tag}_single_E').exists())
    fresh = not w(f'dis{tag}')        # nothing of this split exists yet: allow for a slower session
    return h * (CFG.get('SPLIT_SAFETY', 1.0) if fresh else 1.0) * (1.0 if not QUICK else 0.5)

if RUN_PROTOCOL_B:
    resB, diffB, diffB2, diffM, MCHECK_B = [], [], [], [], None
    all_train = sorted(KEYS['dis_train'])
    gt = SETS['dis_train']['images']
    split_h = 0.0
    for seed in CFG['SEEDS_B']:
        # hours this split needs (what is cached costs nothing); the measured time of the previous split if longer
        need = max(split_h, split_need_h(f'B{seed}', seed, CFG['VARIANTS_B']))
        if hours_used() + need > CFG['TIME_BUDGET_H']:
            B_STOPPED = seed
            print(f'\n!! {hours_used():.1f} h of this session used and a split needs about {need:.1f} h: protocol B '
                  f'stops before split {seed} so the 12 h limit cannot cut the run and Kaggle saves everything. '
                  'Run again with the same settings and this version\'s output attached: finished parts are reused.')
            break
        t_split = time.time()
        kk = all_train[:]; random.Random(1000 + seed).shuffle(kk)
        n_te = round(0.2 * len(kk)); te = kk[:n_te]; rest = kk[n_te:]
        n_va = round(0.1 * len(rest)); va = rest[:n_va]; tr = rest[n_va:]
        va, te, dropped = drop_copies(tr, va, te)    # v15 audit: X-rays whose copy this split trains on are not scored
        if dropped:
            print(f'split {seed}: {len(dropped)} validation / test X-rays have a copy among its training X-rays and are '
                  f'not scored: {dropped}')
        trk, vak, tek = write_keys(f'B{seed}_train', tr), write_keys(f'B{seed}_val', va), write_keys(f'B{seed}_test', te)
        man_tr = train_manifest(f'B{seed}', tr, va + te)
        AD_B = U.fit_box_adapter(gt, TEETH, tr)[0] if CFG['BOX_ADAPTER'] else {}   # training X-rays of this split only
        with stage(f'protocol_B_seed{seed}_{RUN_TAG}'):
            picked, cls_dirs, w, w2 = {}, {}, None, None
            if RUN_YOLO_BASELINE_B:
                w = train_disease_yolo(f'disB{seed}', tr, va, CFG['DIS_EPOCHS'])
                boxes, printed = disease_boxes(w, {'val': va, 'test': te}, f'disbox_B{seed}')
                s, picked['plain YOLO'] = det_row(boxes, gt, gt, va, te, f"plain {CFG['DIS_MODEL'].replace('.pt', '')}, seed {seed}")
                s.update(seed=seed, kind='baseline'); resB.append(s)
                if seed == CFG['SEEDS_B'][0]:
                    MCHECK_B = metric_check(printed, [
                        ('val split (picked its checkpoint)', 'val', gt, va, picked['plain YOLO']['val']),
                        ('test split', 'test', gt, te, picked['plain YOLO']['test'])])
                if CFG['ENSEMBLE_CONTROL']:   # the same detector with another seed, fused with the first
                    w2 = train_disease_yolo(f'disB{seed}_s1', tr, va, CFG['DIS_EPOCHS'], seed=1)
                    boxes2, _ = disease_boxes(w2, {'val': va, 'test': te}, f'disbox_B{seed}_s1')
                    s, picked[YOLO_S1] = det_row(boxes2, gt, gt, va, te, YOLO_S1)
                    s.update(seed=seed, variant=YOLO_S1, kind='control'); resB.append(s)
                    s, picked[YOLO_PAIR] = fusion_row([picked['plain YOLO'], picked[YOLO_S1]], gt, gt, va, te, YOLO_PAIR)
                    s.update(seed=seed, variant=YOLO_PAIR, picked=True, kind='control'); resB.append(s)
            for v in CFG['VARIANTS_B']:
                out = run_cls(v, trk, vak, {'val': vak, 'test': tek}, WORK / 'cls' / f'{CLS_TAG}_B{seed}_{v}', seed=seed,
                              manifest=man_tr)
                if out is None:
                    continue
                cls_dirs[v] = out
                nm = NAMES.get(v, v)
                rows, picked[nm] = model_rows(evalc(out, v, {'val': vak, 'test': tek}), gt, gt, va, te,
                                              f'{nm}, seed {seed}', AD_B)
                for r in rows:
                    r.update(seed=seed, variant=nm, kind='ours')
                resB += rows
                if CFG['FUSION'] and 'plain YOLO' in picked:
                    s, picked[nm + FUSED] = fusion_row([picked[nm], picked['plain YOLO']], gt, gt, va, te, f'{nm}{FUSED}, seed {seed}')
                    s.update(seed=seed, variant=nm + FUSED, picked=True, kind='ours')
                    resB.append(s)
                    if v == 'single' and YOLO_S1 in picked:   # is our classifier more than "a second detector"?
                        for suffix, others in ((FUSED_S1, [picked[YOLO_S1]]),
                                               (FUSED_BOTH, [picked['plain YOLO'], picked[YOLO_S1]])):
                            s, picked[nm + suffix] = fusion_row([picked[nm]] + others, gt, gt, va, te, f'{nm}{suffix}, seed {seed}')
                            s.update(seed=seed, variant=nm + suffix, picked=True, kind='ours')
                            resB.append(s)
            if (CFG['TTA'] or SELF_TRAIN or CFG['SHARP_YOLO']) and w is not None and 'single' in cls_dirs:
                try:      # the 1280 YOLO, test-time augmentation and the self-training round (cell 13b)
                    resB += split_extra_rows(f'B{seed}', seed, tr, va, te, vak, tek, gt, gt, AD_B, picked, cls_dirs,
                                             w, w2 if CFG['ENSEMBLE_CONTROL'] else None)
                except Exception as e:
                    import traceback
                    traceback.print_exc()
                    ST_FAILED.append(f'B{seed}')
                    print(f'!! TTA / self-training on split {seed} failed ({str(e)[:300]}). The rest of the run goes '
                          'on; send Claude the log.')
            nb = 1000 if FINAL and not (SELF_TRAIN and ST_TRIAL) else 300
            if 'plain YOLO' in picked and len(picked) > 1:   # paired differences on this split's test X-rays
                bs = U.paired_bootstrap(gt, {k: coco(p['test']) for k, p in picked.items()}, te, ref='plain YOLO', n_boot=nb)
                diffB += [dict(seed=seed, model=k, **v) for k, v in bs.items()]
            ours_fused = {r['variant'] for r in resB if r.get('seed') == seed and r.get('kind') == 'ours'
                          and r.get('boxes') == 'fused'}
            vs_pair = {k: p for k, p in picked.items() if k == YOLO_PAIR or (k in ours_fused and not k.endswith(TTA_SFX))}
            if YOLO_PAIR in vs_pair and len(vs_pair) > 1:
                bs = U.paired_bootstrap(gt, {k: coco(p['test']) for k, p in vs_pair.items()}, te, ref=YOLO_PAIR, n_boot=nb)
                diffB2 += [dict(seed=seed, model=k, **v) for k, v in bs.items()]
            diffM += matched_bootstrap(gt, picked, te, nb, seed=seed)   # each model vs its own control
        split_h = max(split_h, (time.time() - t_split) / 3600)

if RUN_PROTOCOL_B and not resB:
    print('Protocol B: no split finished in this session.')
elif RUN_PROTOCOL_B:
    TAB_B = pd.DataFrame(resB)
    TAB_B['split_group'] = np.where(TAB_B.seed.isin(CFG.get('CONFIRM_SEEDS', [])), 'confirmation', 'development')
    TAB_B.to_csv(WORK / f'results_protocol_B_{RUN_TAG}.csv', index=False)
    MAIN_B = TAB_B[TAB_B.picked].copy().reset_index(drop=True)
    MAIN_B['model'] = MAIN_B['variant']
    print('Protocol B (random 80:20 splits of the 705). DENTEX / COCO scores; box source and fusion picked on '
          'each split\'s validation X-rays:')
    show(MAIN_B[['model', 'seed', 'boxes', 'diag_AP', 'diag_AP50', 'diag_AP75', 'diag_AR', 'val_diag_AP', 'chosen']])
    print('\nThe same predictions scored the way ultralytics\' validator does. Compare the papers with ul_old '
          '(their software\'s AP formula):')
    show(MAIN_B[UL_COLS])
    K_AGG = ['diag_AP', 'diag_AP50', 'diag_AP75', 'diag_AR', 'ul_old_AP', 'ul_old_AP50']
    agg = MAIN_B.groupby('variant', sort=False)[K_AGG].agg(['mean', 'std']).round(2)
    print('\nProtocol B: mean and std over seeds', sorted(MAIN_B.seed.unique().tolist())); show(agg)
    if MAIN_B.split_group.nunique() > 1:     # v15: the splits no version had looked at, on their own
        cf = MAIN_B[MAIN_B.split_group == 'confirmation']
        print('\nProtocol B, only the fresh confirmation splits', sorted(cf.seed.unique().tolist()),
              '(method frozen before they were made):')
        show(cf.groupby('variant', sort=False)[K_AGG].agg(['mean', 'std']).round(2))
    compare_versions(MAIN_B[MAIN_B.split_group == 'development'], 'results_protocol_B_v14.csv', 'Protocol B')
    print('\nper-class AP, mean over seeds:'); show(per_class(MAIN_B, 'per_class_AP'))
    print('\nbest F1 per class (IoU 0.5), mean over seeds:'); show(per_class(MAIN_B, 'best_F1_iou50', 3))
    print('\nper-class AP50 the ultralytics way, old formula (compare with Sadr et al.\'s per-class AP50):')
    show(per_class(MAIN_B, 'ul_old_per_class_AP50'))
    def over_splits(D, ref):
        """one line per model: the difference to `ref` on every split and on how many splits it is significant"""
        g = D.groupby('model', sort=False)
        return pd.DataFrame({'splits': g.size(), f'AP minus {ref}, mean': g.dAP.mean().round(2),
                             'lowest': g.dAP.min(), 'highest': g.dAP.max(),
                             'significant better / worse': [f'{(x.dAP_lo > 0).sum()} / {(x.dAP_hi < 0).sum()}' for _, x in g]})
    if diffB:
        DIFF_B = pd.DataFrame(diffB)
        DIFF_B.to_csv(WORK / f'bootstrap_B_{RUN_TAG}.csv', index=False)
        print('\nDifference to plain YOLO trained on the SAME split (DENTEX / COCO AP, 95% paired bootstrap interval per split):')
        show_boot(DIFF_B)
        print('\nOver all splits:'); show(over_splits(DIFF_B, 'YOLO'))
    if diffB2:
        DIFF_B2 = pd.DataFrame(diffB2)
        DIFF_B2.to_csv(WORK / f'bootstrap_B_vs_pair_{RUN_TAG}.csv', index=False)
        print('\nDifference to the two-YOLO control on the same split (two detectors fused the same way). Positive = our '
              'classifier adds more than a second YOLO does:')
        show_boot(DIFF_B2)
        print('\nOver all splits:'); show(over_splits(DIFF_B2, 'YOLO pair'))
    extra_tables(MAIN_B, diffM, 'Protocol B')
    if ST_FAILED:
        print('!! TTA / self-training failed on', ST_FAILED, '(scroll up for the error)')
    if MCHECK_B is not None:
        print(f"\nPlain YOLO of seed {CFG['SEEDS_B'][0]}, one set of boxes, four ways of scoring it (diagnosis mAP50-95 and mAP50):")
        show(MCHECK_B)
    ours = TAB_B[(TAB_B.kind == 'ours') & (TAB_B.boxes != 'fused')]
    if len(ours):
        print('\nBox ablation, every box source. Validation AP (this decides) | test AP:')
        pv = ours.pivot_table(index=['variant', 'seed'], columns='boxes', values='val_diag_AP', sort=False)
        pt = ours.pivot_table(index=['variant', 'seed'], columns='boxes', values='diag_AP', sort=False)
        show(pd.concat({'val AP': pv, 'test AP': pt}, axis=1).round(2))
    print('\nPeers (as reported on their own splits; compare the ultralytics ones with our ul_old columns). Hoang et al. '
          'and DinoDental report on the split their checkpoint was picked on (no separate test split):'); show(PEERS_B)


## 10. GPU time used

In [ ]:
gpu_h = sum(TIMES.values()) / 3600
print('Wall-clock per stage (hours):')
for k, v in TIMES.items():
    print(f'  {k:28s} {v/3600:5.2f}')
print(f'TOTAL {gpu_h:.2f} h  (on GPU T4 x2 one wall-clock hour uses one hour of Kaggle GPU quota)')
# v15: Kaggle keeps only ~500 output files, so everything a later session needs goes into ONE file, state.zip
# (stored, not compressed); the results meant for reading stay next to it as they are
def pack_state():
    """every file of WORK except the readable results (top-level csv / jpg, timing.json) and crops.zip -> state.zip"""
    import zipfile
    unpack_state()           # (only if this cell runs a second time in an interactive session)
    loose = lambda q: (q.parent == WORK and (q.suffix in ('.csv', '.jpg') or q.name in ('timing.json', 'crops.zip'))
                       and not q.name.endswith('_manifest.csv'))
    files = [q for q in WORK.rglob('*') if q.is_file() and q.name != 'state.zip' and not loose(q)]
    if not files:
        return 0
    with zipfile.ZipFile(WORK / 'state.zip', 'w', zipfile.ZIP_STORED, allowZip64=True) as zf:
        for q in files:
            zf.write(q, str(q.relative_to(WORK)))
    for q in files:
        q.unlink()
    for d in sorted([q for q in WORK.rglob('*') if q.is_dir()], key=lambda q: -len(q.parts)):
        if not any(d.iterdir()):
            d.rmdir()
    return len(files)

n_packed = pack_state()
n_files = sum(1 for p in Path('/kaggle/working' if ON_KAGGLE else WORK).rglob('*') if p.is_file())
print(f'{n_packed} files packed into state.zip (the next session unpacks it); files in the saved output: {n_files}'
      + ('  !! close to Kaggle\'s ~500 limit, tell Claude' if n_files > 450 else ''))
if FAILED:
    print('\n!! classifier runs that failed twice and were skipped:', FAILED, '-> send Claude the log')
if ST_FAILED:
    print('\n!! TTA / self-training failed on', ST_FAILED, '-> send Claude the log')
if B_STOPPED is not None:
    print(f'\n!! protocol B stopped before split {B_STOPPED} to stay inside the 12 h limit. Run this notebook again with '
          'the same settings and this version\'s output attached (Add Input -> Your Work): it reuses everything '
          'and only does the missing splits.')
print(f'\nThis session used {hours_used():.1f} h. Send Claude the log of this version.')
